# PWNSAT OMR Scanner v18 — direct download buttons + manual-edit image exports
Mobile-friendly NSAT + NEET OMR. Excel, results ZIP and corrected-image ZIP are direct download buttons shown in the same tab as export. Pages with explicit manual bubble edits can export a separate corrected copy before whole-row VERIFIED; for such pages only the saved manual bubbles are changed. Original scans remain unchanged.


In [ ]:
# PWNSAT OMR HYBRID SCANNER v18 — MOBILE DOWNLOAD FIX — Fixed-slot typing, source-pixel correction, source format/DPI preservation
# Upload a blank/reference template once per OMR design, then scan many student sheets.
# It reads fixed template bubble positions and exports a clean student-details sheet.
# Bubble-derived P01 name is primary. OCR is an optional comparison candidate and never redraws bubbles or silently overwrites the bubble result.
# Optional AI handwriting OCR (TrOCR when available, bounded Tesseract fallback), manual review, and validated stable Excel/ZIP downloads.
# Automatic processing includes progress, bounded OCR timeouts, and manual result editing; original scans are never overwritten.

import os, sys, subprocess, importlib.util, json, re, zipfile, shutil, math, time, traceback, base64, datetime
from pathlib import Path

# Install only missing dependencies. Avoid reinstalling everything on every run.
_required_modules = {
    'gradio': 'gradio', 'cv2': 'opencv-python-headless', 'pandas': 'pandas',
    'openpyxl': 'openpyxl', 'PIL': 'pillow', 'pytesseract': 'pytesseract',
    'fitz': 'pymupdf', 'scipy': 'scipy'
}
_missing_packages = [pkg for module, pkg in _required_modules.items() if importlib.util.find_spec(module) is None]
if _missing_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing_packages], check=True, timeout=240)
if subprocess.run(['bash', '-lc', 'command -v tesseract >/dev/null 2>&1']).returncode != 0:
    subprocess.run(['bash', '-lc', 'apt-get -qq update && apt-get -qq install -y tesseract-ocr'], check=True, timeout=300)
import cv2
import numpy as np
import pandas as pd
import pytesseract
import gradio as gr
from PIL import Image, ImageOps

def draw_filled_bubble(image, x, y, r, color=None, fill_ratio=0.92):
    """Fill most of a bubble's inside area (not a centre dot), leaving the printed outline visible.

    The detected radius represents the printed circle's outer radius. Filling approximately 92% of that
    radius produces a near-full solid mark while leaving a thin margin inside the printed ring. Works with grayscale and BGR images.
    """
    if image is None or not hasattr(image, 'shape'):
        raise ValueError('Bubble fill target image is empty.')
    rr = max(2, int(round(max(1.0, float(r)) * min(0.94, max(0.72, float(fill_ratio))))) )
    cx, cy = int(round(float(x))), int(round(float(y)))
    if color is None:
        color = 0 if image.ndim == 2 else (0, 0, 0)
    # Anti-aliased outer edge, solid-black interior.
    cv2.circle(image, (cx, cy), rr, color, thickness=-1, lineType=cv2.LINE_AA)
    return image

try:
    from scipy.spatial import cKDTree
except Exception:
    cKDTree = None

ROOT = Path('/content/universal_omr_scanner')
PROFILE_DIR = ROOT / 'profiles'
RUN_DIR = ROOT / 'runs'
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Built-in blank reference for the PW Physics Wallah 180-question bubble-only sheet.
# This dedicated mode does not run handwriting OCR and never exports handwritten name fields.
_BUBBLE_ONLY_PDF_B64 = """...
"""
BUBBLE_ONLY_TEMPLATE_PDF = ROOT / 'U_NEET_N_180Q_V2_NG_26.pdf'
if not BUBBLE_ONLY_TEMPLATE_PDF.exists():
    BUBBLE_ONLY_TEMPLATE_PDF.write_bytes(base64.b64decode(_BUBBLE_ONLY_PDF_B64))
BUBBLE_ONLY_TEMPLATE_PNG = ROOT / 'U_NEET_N_180Q_V2_NG_26_template.png'

# Main export contains only student detail fields requested for PWNSAT, plus Grade/Class.
STUDENT_DETAIL_COLUMNS = [
    'Student Name',
    'Mobile No.',
    'Parent/Guardian Mobile No.',
    'Exam Code',
    'DOB DDMMYY',
    'Class / Grade',
]
STUDENT_GROUP_FIELDS = {
    'P01': 'Student Name',
    'P02': 'Mobile No.',
    'P03': 'Exam Code',
    'P04': 'Parent/Guardian Mobile No.',
    'P05': 'DOB DDMMYY',
    'P06': 'Class / Grade',
}


# Optional AI handwriting recognizer (loaded once only when name OCR is enabled).
_TROCR_PROCESSOR = None
_TROCR_MODEL = None
_TROCR_DEVICE = 'cpu'
_TROCR_LOAD_ATTEMPTED = False
_TROCR_LOAD_ERROR = ''
_TROCR_MODEL_ID = 'microsoft/trocr-small-handwritten'


def initialize_trocr_model(progress=None):
    """Load a handwriting-trained TrOCR model once; gracefully fall back to Tesseract on failure."""
    global _TROCR_PROCESSOR, _TROCR_MODEL, _TROCR_DEVICE, _TROCR_LOAD_ATTEMPTED, _TROCR_LOAD_ERROR
    if _TROCR_MODEL is not None:
        return True, f'TrOCR ready ({_TROCR_MODEL_ID})'
    if _TROCR_LOAD_ATTEMPTED:
        return False, (_TROCR_LOAD_ERROR or 'TrOCR is unavailable; Tesseract fallback will be used.')
    _TROCR_LOAD_ATTEMPTED = True
    try:
        # Keep Hub connection waits bounded; an offline runtime must fall back, not hang forever.
        os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '10')
        os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '60')
        if importlib.util.find_spec('transformers') is None:
            if progress:
                progress(0.01, desc='Installing optional handwriting OCR support (one time)...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.40,<5', 'sentencepiece'],
                           check=True, timeout=240)
        import torch
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
        if progress:
            progress(0.02, desc='Loading handwriting OCR model (first use may download weights)...')
        _TROCR_PROCESSOR = TrOCRProcessor.from_pretrained(_TROCR_MODEL_ID)
        _TROCR_MODEL = VisionEncoderDecoderModel.from_pretrained(_TROCR_MODEL_ID)
        _TROCR_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
        _TROCR_MODEL.to(_TROCR_DEVICE)
        _TROCR_MODEL.eval()
        if _TROCR_DEVICE == 'cpu':
            try: torch.set_num_threads(max(1, min(4, os.cpu_count() or 2)))
            except Exception: pass
        _TROCR_LOAD_ERROR = ''
        return True, f'TrOCR ready on {_TROCR_DEVICE}; OCR is a review proposal, not auto-verified.'
    except Exception as exc:
        _TROCR_LOAD_ERROR = f'TrOCR unavailable ({type(exc).__name__}: {str(exc)[:220]}); gridline-cleaned Tesseract fallback used.'
        _TROCR_PROCESSOR = None; _TROCR_MODEL = None
        return False, _TROCR_LOAD_ERROR


def stage_download_file(path, validate_kind=None):
    """Validate then stage an artifact in a unique allowed path for reliable Gradio downloads."""
    source = Path(str(path)).expanduser().resolve()
    if not source.exists() or not source.is_file() or source.stat().st_size == 0:
        raise FileNotFoundError(f'Download file was not created or is empty: {source.name}')
    kind = validate_kind or ('xlsx' if source.suffix.lower() == '.xlsx' else ('zip' if source.suffix.lower() == '.zip' else 'file'))
    def validate(p):
        if not p.exists() or not p.is_file() or p.stat().st_size == 0:
            raise FileNotFoundError(f'Validated download is missing/empty: {p.name}')
        if kind == 'xlsx':
            import openpyxl
            wb = openpyxl.load_workbook(p, read_only=True, data_only=True)
            try:
                if not wb.sheetnames:
                    raise ValueError(f'Excel workbook has no worksheet: {p.name}')
                # Touch one sheet/cell so the workbook stream is actually readable.
                ws = wb[wb.sheetnames[0]]
                _ = ws.max_row, ws.max_column
            finally:
                wb.close()
        elif kind == 'zip':
            with zipfile.ZipFile(p, 'r') as zf:
                bad = zf.testzip()
                if bad:
                    raise ValueError(f'ZIP validation failed at {bad}')
        return p.stat().st_size
    validate(source)
    run_tag = source.parent.parent.name if source.parent.name in ('outputs', 'manual_final_outputs') else source.parent.name
    safe_tag = slugify(run_tag or str(int(time.time())))
    # Unique per output avoids a second run overwriting an older file the browser is downloading.
    target_dir = ROOT / 'gradio_downloads' / f'{safe_tag}_{time.time_ns()}'
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / source.name
    shutil.copy2(source, target)
    validate(target)
    root = ROOT.resolve()
    try:
        target.resolve().relative_to(root)
    except ValueError:
        raise ValueError('Download path escaped the explicitly allowed application folder.')
    return str(target.resolve())

def student_detail_column(group):
    """Map a mapped profile group to one of the requested student fields."""
    gid = str(group.get('group_id', ''))
    if gid in STUDENT_GROUP_FIELDS:
        return STUDENT_GROUP_FIELDS[gid]
    label = str(group.get('label') or group.get('label_hint') or '').strip().lower()
    if 'parent' in label or 'guardian' in label:
        if 'mobile' in label or 'phone' in label:
            return 'Parent/Guardian Mobile No.'
    if 'student' in label and 'name' in label:
        return 'Student Name'
    if 'mobile' in label or 'phone' in label:
        return 'Mobile No.'
    if 'exam code' in label:
        return 'Exam Code'
    if 'dob' in label or 'date of birth' in label:
        return 'DOB DDMMYY'
    if 'grade' in label or 'class' in label or 'studying' in label:
        return 'Class / Grade'
    return None

# ---------- File and image utilities ----------
def slugify(text):
    s = re.sub(r'[^A-Za-z0-9_-]+', '_', str(text or 'omr_profile')).strip('_')
    return (s[:60] or 'omr_profile')

def safe_read_image(path):
    path = str(path)
    arr = np.fromfile(path, dtype=np.uint8)
    im = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if im is None:
        raise ValueError(f'Image open nahi hui: {Path(path).name}')
    return im

def save_image(path, image):
    path = str(path)
    ext = Path(path).suffix.lower() or '.png'
    ok, encoded = cv2.imencode(ext, image)
    if not ok:
        raise ValueError(f'Image save nahi hui: {path}')
    encoded.tofile(path)
    return path

def paths_to_pages(file_path, out_dir, prefix):
    """Return (page_path, source_name, page_number) items for image or PDF."""
    if not file_path:
        return []
    p = Path(file_path)
    ext = p.suffix.lower()
    if ext == '.pdf':
        import fitz
        doc = fitz.open(str(p))
        items = []
        for idx, page in enumerate(doc, start=1):
            pix = page.get_pixmap(matrix=fitz.Matrix(2.5, 2.5), alpha=False)
            out = Path(out_dir) / f'{prefix}_{slugify(p.stem)}_page{idx}.png'
            pix.save(str(out))
            items.append((str(out), p.name, idx))
        if not items:
            raise ValueError(f'PDF khali hai: {p.name}')
        return items
    allowed = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
    if ext not in allowed:
        raise ValueError(f'Unsupported file format: {p.name}')
    # Copy input to the run folder; never modify the source.
    out = Path(out_dir) / f'{prefix}_{slugify(p.stem)}{ext}'
    shutil.copy2(p, out)
    return [(str(out), p.name, 1)]

def normalize_gray(gray, max_dim=2200):
    h, w = gray.shape[:2]
    scale = min(1.0, max_dim / float(max(h, w)))
    if scale < 1.0:
        gray = cv2.resize(gray, (max(1, int(round(w*scale))), max(1, int(round(h*scale)))), interpolation=cv2.INTER_AREA)
    return gray

def ocr_text(gray):
    """Read raw page text. Handwriting results can be imperfect."""
    try:
        g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(gray)
        return pytesseract.image_to_string(g, config='--psm 6', timeout=8).strip()
    except Exception as e:
        return f'[OCR ERROR: {e}]'

def ocr_words(gray):
    rows = []
    try:
        data = pytesseract.image_to_data(gray, config='--psm 6', output_type=pytesseract.Output.DATAFRAME, timeout=8)
        if data is None or data.empty:
            return rows
        data = data.dropna(subset=['text'])
        for _, r in data.iterrows():
            txt = str(r.get('text','')).strip()
            if txt and txt.lower() != 'nan':
                rows.append({'text':txt, 'conf':float(r.get('conf',-1) or -1), 'x':int(r['left']), 'y':int(r['top']), 'w':int(r['width']), 'h':int(r['height'])})
    except Exception:
        pass
    return rows

# ---------- Bubble detection ----------
def detect_circles(gray):
    """Find likely circular bubble outlines; parameters scale with image dimensions."""
    src = normalize_gray(gray, max_dim=2200)
    sh, sw = src.shape[:2]
    short = min(sh, sw)
    # Radius range scales with page size. Typical circles are small relative to sheet width.
    min_r = max(3, int(round(short * 0.0032)))
    max_r = max(min_r + 3, int(round(short * 0.0135)))
    min_dist = max(6, int(round(short * 0.0072)))
    blur = cv2.medianBlur(src, 3 if min(src.shape) < 1800 else 5)
    circles = cv2.HoughCircles(blur, cv2.HOUGH_GRADIENT, dp=1.1, minDist=min_dist,
                               param1=90, param2=20, minRadius=min_r, maxRadius=max_r)
    if circles is None:
        # A slightly more permissive second pass may recover faint printed circles.
        circles = cv2.HoughCircles(blur, cv2.HOUGH_GRADIENT, dp=1.15, minDist=min_dist,
                                   param1=80, param2=17, minRadius=min_r, maxRadius=max_r)
    if circles is None:
        return []
    scale_x = gray.shape[1] / float(sw)
    scale_y = gray.shape[0] / float(sh)
    candidates = []
    for x, y, r in np.round(circles[0]).astype(float):
        candidates.append((float(x*scale_x), float(y*scale_y), float(r*(scale_x+scale_y)/2.0)))
    # Prefer the larger/stronger circle when Hough detects the same printed circle twice.
    candidates.sort(key=lambda c: c[2], reverse=True)
    kept = []
    for c in candidates:
        dup = False
        for k in kept:
            dist = math.hypot(c[0]-k[0], c[1]-k[1])
            if dist < max(3.0, 0.62 * min(c[2], k[2])):
                dup = True
                break
        if not dup:
            kept.append(c)
    kept.sort(key=lambda c: (c[1], c[0]))
    return kept

def group_circle_components(circles):
    """Cluster neighboring detected circles into spatial grid candidates."""
    n = len(circles)
    if n == 0:
        return [], []
    pts = np.array([[c[0], c[1]] for c in circles], dtype=float)
    medr = float(np.median([c[2] for c in circles]))
    eps = max(13.0, medr * 3.35)
    parent = list(range(n))
    rank = [0]*n
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    def union(a,b):
        ra,rb=find(a),find(b)
        if ra == rb: return
        if rank[ra] < rank[rb]: ra,rb=rb,ra
        parent[rb]=ra
        if rank[ra] == rank[rb]: rank[ra]+=1
    if cKDTree is not None:
        tree = cKDTree(pts)
        pairs = tree.query_pairs(eps)
        for a,b in pairs:
            union(a,b)
    else:
        for i in range(n):
            ds = np.sqrt(((pts[i+1:] - pts[i])**2).sum(axis=1))
            for off in np.where(ds <= eps)[0]: union(i, i+1+int(off))
    components = {}
    for i in range(n): components.setdefault(find(i), []).append(i)
    groups = []
    circle_group = ['UNGROUPED']*n
    comps = [ids for ids in components.values() if len(ids) >= 6]
    comps.sort(key=lambda ids: (float(np.mean(pts[ids,1])), float(np.mean(pts[ids,0]))))
    for k, ids in enumerate(comps, start=1):
        gid = f'G{k:03d}'
        arr = pts[ids]
        rs = [circles[i][2] for i in ids]
        tol = max(3.0, float(np.median(rs))*0.78)
        ys = cluster_axis([circles[i][1] for i in ids], tol)
        xs = cluster_axis([circles[i][0] for i in ids], tol)
        bbox = [float(arr[:,0].min()), float(arr[:,1].min()), float(arr[:,0].max()), float(arr[:,1].max())]
        cell_fill = len(ids)/max(1, len(xs)*len(ys))
        g = {'group_id':gid, 'circle_indices':ids, 'rows':len(ys), 'cols':len(xs), 'count':len(ids),
             'grid_fill_ratio':round(cell_fill,3), 'bbox':bbox, 'x_centers':xs, 'y_centers':ys,
             'suggested_type':guess_group_type(len(ys),len(xs),cell_fill), 'label':'', 'question_start':''}
        groups.append(g)
        for i in ids: circle_group[i]=gid
    return groups, circle_group

def cluster_axis(values, tol):
    vals = sorted(float(v) for v in values)
    if not vals: return []
    buckets = [[vals[0]]]
    for v in vals[1:]:
        if abs(v - float(np.median(buckets[-1]))) <= tol:
            buckets[-1].append(v)
        else:
            buckets.append([v])
    return [float(np.median(b)) for b in buckets]

def guess_group_type(rows, cols, fill_ratio):
    # Name-letter grids usually have about 26 option rows; numeric grids about 10.
    # A typical MCQ block has question rows (often 5-20) and 2-6 option columns.
    if 22 <= rows <= 30 and cols >= 5 and fill_ratio > 0.35:
        return 'letter_grid (options down; slots across)'
    if 8 <= rows <= 11 and 7 <= cols <= 12 and fill_ratio > 0.35:
        return 'digit_grid (digits down; slots across)'
    if 9 <= rows <= 11 and 3 <= cols <= 6 and fill_ratio > 0.35:
        return 'digit_grid (digits down; slots across)'
    if rows >= 4 and 2 <= cols <= 6 and fill_ratio > 0.4:
        return 'mcq_grid (questions down; options across)'
    if rows <= 2 and cols >= 4:
        return 'single_row_choices'
    return 'generic_grid'

def label_hint_for_group(group, words):
    x1,y1,x2,y2 = group['bbox']
    width=max(1,x2-x1)
    # Tokens immediately above or slightly to the left of the component are likely field labels.
    candidates=[]
    for w in words:
        wx,wy,ww,wh=w['x'],w['y'],w['w'],w['h']
        if wy+wh <= y1 + max(3, group['y_centers'][1]-group['y_centers'][0] if len(group['y_centers'])>1 else 10):
            gap = y1-(wy+wh)
            xover = max(0, min(x2, wx+ww)-max(x1,wx)) / max(1,min(width,ww))
            if gap <= max(80, (y2-y1)*0.22) and (xover > 0.25 or abs((wx+ww/2)-((x1+x2)/2)) < width*0.75):
                candidates.append((gap - 10*xover, w['text']))
    if not candidates:
        return ''
    candidates.sort(key=lambda q:q[0])
    return ' '.join(t for _,t in candidates[:4])[:100]

def bubble_metrics(gray, x, y, r):
    """Measure ink from black AND lighter grey pixels inside the bubble.

    The template-relative delta is calculated later, so printed digits/circle residue
    common to both reference and student sheets is mostly cancelled out. A soft score
    helps pick up pencil/weak pen marks that do not cross the old <105 grayscale cutoff.
    """
    h,w=gray.shape[:2]
    x,y,r=int(round(x)),int(round(y)),max(3,int(round(r)))
    x0,x1=max(0,x-r),min(w,x+r+1)
    y0,y1=max(0,y-r),min(h,y+r+1)
    roi=gray[y0:y1,x0:x1]
    if roi.size == 0: return 0.0, 255.0
    yy,xx=np.ogrid[y0:y1,x0:x1]
    d2=(xx-x)**2+(yy-y)**2
    mask=d2 <= (0.57*r)**2
    vals=roi[mask]
    if vals.size == 0: return 0.0,255.0
    # Separate bins avoid letting a few anti-aliased pixels count as full black ink.
    black=float(np.mean(vals < 105))
    mid=float(np.mean((vals >= 105) & (vals < 155)))
    light=float(np.mean((vals >= 155) & (vals < 205)))
    soft_ink=min(1.0, black + 0.65*mid + 0.25*light)
    return soft_ink, float(np.mean(vals))

def configure_pwnsat_profile(H, W, circle_rows, detected_groups, force=False):
    """Apply geometry-based mapping for the known PWNSAT Grade V-XII form.

    This only activates when the detected shape matches its distinctive layout.
    All coordinates are scaled relative to the reference page; marks are still read
    from each aligned student sheet at the fixed reference positions.
    """
    try:
        has_name = any(26 <= int(g.get('rows', 0)) <= 28 and 16 <= int(g.get('cols', 0)) <= 18 for g in detected_groups)
        digit10x10 = sum(1 for g in detected_groups if int(g.get('rows',0)) == 10 and int(g.get('cols',0)) == 10)
        digit10x5 = any(int(g.get('rows',0)) == 10 and int(g.get('cols',0)) == 5 for g in detected_groups)
        mcq_like = sum(1 for g in detected_groups if int(g.get('rows',0)) >= 11 and 3 <= int(g.get('cols',0)) <= 5 and float(g.get('bbox',[0,0,0,0])[1]) > H*0.70)
        if not force and not (has_name and digit10x10 >= 2 and digit10x5 and mcq_like >= 4):
            return None
        # This notebook targets the PWNSAT Grade V-XII template. On an explicitly loaded
        # PWNSAT template, force=True rebuilds the known field geometry from all detected
        # circle coordinates even if an older generic profile grouped circles incorrectly.

        sx, sy = W / 1489.0, H / 2106.0
        # Exact expected geometry of this printed template. The DOB grid is sparse by design.
        regions = [
            ('P01', 'Student Name', 'letter_grid (options down; slots across)',
             lambda x,y: 120*sx <= x <= 700*sx and 530*sy <= y <= 1380*sy, 26,17,None,None),
            ('P02', 'Mobile No.', 'digit_grid (digits down; slots across)',
             lambda x,y: 720*sx <= x <= 1060*sx and 530*sy <= y <= 860*sy, 10,10,None,None),
            ('P03', 'Exam Code', 'digit_grid (digits down; slots across)',
             lambda x,y: 1100*sx <= x <= 1290*sx and 530*sy <= y <= 860*sy, 10,5,None,None),
            ('P04', 'Parent/Guardian Mobile No.', 'digit_grid (digits down; slots across)',
             lambda x,y: 720*sx <= x <= 1060*sx and 950*sy <= y <= 1290*sy, 10,10,None,None),
            ('P05', 'DOB DDMMYY (sparse grid)', 'digit_grid (digits down; slots across)',
             lambda x,y: 1080*sx <= x <= 1290*sx and 960*sy <= y <= 1290*sy, 10,6,None,None),
            ('P06', 'Grade (Studying)', 'single_row_choices',
             lambda x,y: 1454*sy <= y <= 1465*sy and min(abs(x-z*sx) for z in [183,281,380,478,576,674,805,936,1099,1262]) < 18*sx, 1,10,None,
             ['Class V','Class VI','Class VII','Class VIII','Class IX','Class X','11th JEE','12th JEE','11th NEET','12th NEET']),
        ]
        mcq_xbands = [(200,335),(430,560),(660,785),(890,1015),(1115,1245)]
        for k,(x1,x2) in enumerate(mcq_xbands):
            qstart = 1 + 12*k
            regions.append((f'P{7+k:02d}', f'MCQ Q{qstart}-{qstart+11}', 'mcq_grid (questions down; options across)',
                lambda x,y,x1=x1,x2=x2: x1*sx <= x <= x2*sx and 1545*sy <= y <= 1930*sy,
                12,4,str(qstart),None))

        kept=[]
        groups=[]
        for gid,label,typ,inside,expect_rows,expect_cols,qstart,choice_labels in regions:
            ids=[i for i,c in enumerate(circle_rows) if inside(float(c['x_px']),float(c['y_px']))]
            if not ids:
                return None
            items=[circle_rows[i] for i in ids]
            if not items:
                return None
            medr=float(np.median([c['r_px'] for c in items]))
            tol=max(3.0,medr*0.72)
            if gid == 'P06':
                # Grade bubbles are deliberately far apart; use the known 10-position row.
                xs=[z*sx for z in [183,281,380,478,576,674,805,936,1099,1262]]
                ys=[1460*sy]
            else:
                xs=cluster_axis([c['x_px'] for c in items],tol)
                ys=cluster_axis([c['y_px'] for c in items],tol)
            # The printed grids are lattices. Reconstruct a missed circle at its expected
            # template cell instead of dropping/shifting the row or column mapping.
            if len(xs) != expect_cols or len(ys) != expect_rows:
                return None
            sparse_valid=None
            if gid == 'P05':
                # DOB DDMMYY has intentionally absent cells: rows 0-2 have all columns,
                # row 3 is missing column 3, and rows 4-9 contain columns 2,4,5,6 only.
                sparse_valid=set()
                for rr in range(1,4):
                    for cc in range(1,7): sparse_valid.add((rr,cc))
                for cc in [1,2,4,5,6]: sparse_valid.add((4,cc))
                for rr in range(5,11):
                    for cc in [2,4,5,6]: sparse_valid.add((rr,cc))
            new_ids=[]
            grid_items=[]
            used_detection_ids=set()
            for rr in range(1,expect_rows+1):
                for cc in range(1,expect_cols+1):
                    if sparse_valid is not None and (rr,cc) not in sparse_valid:
                        continue
                    x=float(xs[cc-1]); y=float(ys[rr-1])
                    nearest=None; nearest_d=float('inf')
                    for c in items:
                        d=math.hypot(float(c['x_px'])-x,float(c['y_px'])-y)
                        if d < nearest_d:
                            nearest,nearest_d=c,d
                    # Reuse radius/ID only for a detection close to this expected cell.
                    if nearest is not None and nearest_d <= max(6.0, medr*0.55) and nearest['bubble_id'] not in used_detection_ids:
                        n=dict(nearest); used_detection_ids.add(nearest['bubble_id'])
                    else:
                        n={'bubble_id':f'V_{gid}_{rr:02d}_{cc:02d}'}
                    n.update({'x_px':x,'y_px':y,'r_px':medr,'x_norm':x/W,'y_norm':y/H,
                              'r_norm':medr/min(H,W),'group_id':gid,'grid_row':rr,'grid_col':cc})
                    new_ids.append(len(kept)); kept.append(n); grid_items.append(n)
            fill_ratio=len(grid_items)/max(1,expect_rows*expect_cols)
            x1=min(c['x_px'] for c in grid_items); x2=max(c['x_px'] for c in grid_items)
            y1=min(c['y_px'] for c in grid_items); y2=max(c['y_px'] for c in grid_items)
            groups.append({
                'group_id':gid,'circle_indices':new_ids,'rows':expect_rows,'cols':expect_cols,
                'count':len(grid_items),'grid_fill_ratio':round(fill_ratio,3),
                'bbox':[float(x1),float(y1),float(x2),float(y2)],
                'x_centers':[float(v) for v in xs],'y_centers':[float(v) for v in ys],
                'suggested_type':typ,'label_hint':label,'label':label,
                'question_start':qstart or '', 'choice_labels':choice_labels or []
            })
        return kept, groups
    except Exception:
        # Do not fail template analysis: unsupported/ambiguous layouts use the generic mode.
        return None

_PWNSAT_CANONICAL_GEOMETRY = {'P01': {'label': 'Student Name Bubbles', 'label_hint': 'Student Name Bubbles', 'suggested_type': 'letter_grid (options down; slots across)', 'rows': 26, 'cols': 17, 'xs': [0.1007388, 0.1229013, 0.1450638, 0.1672263, 0.1887173, 0.2102082, 0.2323707, 0.2545332, 0.2766958, 0.2985225, 0.320685, 0.3431833, 0.3653459, 0.3868368, 0.4089993, 0.4311619, 0.4526528], 'ys': [0.2583096, 0.2739791, 0.2891738, 0.3048433, 0.3205128, 0.3361823, 0.351377, 0.3670465, 0.382716, 0.3979107, 0.4135802, 0.4292498, 0.4449193, 0.4605888, 0.4762583, 0.4919278, 0.5071225, 0.522792, 0.5384615, 0.5541311, 0.5688509, 0.5845204, 0.6001899, 0.6153846, 0.6310541, 0.6467236], 'r_norm': 0.00738751, 'question_start': '', 'choice_labels': []}, 'P02': {'label': 'Mobile No.', 'label_hint': 'Mobile No.', 'suggested_type': 'digit_grid (digits down; slots across)', 'rows': 10, 'cols': 10, 'xs': [0.4969778, 0.5191404, 0.5402955, 0.5627938, 0.5849563, 0.6064473, 0.6286098, 0.6507723, 0.6729349, 0.6937542], 'ys': [0.2583096, 0.2739791, 0.2891738, 0.3048433, 0.3205128, 0.3361823, 0.3516144, 0.3670465, 0.382716, 0.3981481], 'r_norm': 0.0080591, 'question_start': '', 'choice_labels': []}, 'P03': {'label': 'Exam Code', 'label_hint': 'Exam Code', 'suggested_type': 'digit_grid (digits down; slots across)', 'rows': 10, 'cols': 5, 'xs': [0.7602418, 0.7817327, 0.8038952, 0.8260578, 0.8475487], 'ys': [0.2583096, 0.2739791, 0.2891738, 0.3048433, 0.3205128, 0.3361823, 0.3518519, 0.3675214, 0.3831909, 0.3983856], 'r_norm': 0.00738751, 'question_start': '', 'choice_labels': []}, 'P04': {'label': 'Parent/Guardian Mobile No.', 'label_hint': 'Parent/Guardian Mobile No.', 'suggested_type': 'digit_grid (digits down; slots across)', 'rows': 10, 'cols': 10, 'xs': [0.4963062, 0.5184688, 0.5399597, 0.5621222, 0.5849563, 0.6064473, 0.6286098, 0.6507723, 0.6729349, 0.6937542], 'ys': [0.4603514, 0.4757835, 0.491453, 0.5071225, 0.522792, 0.5384615, 0.5541311, 0.5688509, 0.5845204, 0.6001899], 'r_norm': 0.00738751, 'question_start': '', 'choice_labels': []}, 'P05': {'label': 'DOB DDMMYY (sparse grid)', 'label_hint': 'DOB DDMMYY (sparse grid)', 'suggested_type': 'digit_grid (digits down; slots across)', 'rows': 10, 'cols': 6, 'xs': [0.7380792, 0.7602418, 0.7817327, 0.8038952, 0.8260578, 0.8482203], 'ys': [0.460114, 0.4762583, 0.4919278, 0.5071225, 0.522792, 0.5384615, 0.5541311, 0.5688509, 0.5845204, 0.6001899], 'r_norm': 0.0080591, 'question_start': '', 'choice_labels': []}, 'P06': {'label': 'Grade (Studying)', 'label_hint': 'Grade (Studying)', 'suggested_type': 'single_row_choices', 'rows': 1, 'cols': 10, 'xs': [0.1229013, 0.1887173, 0.2552048, 0.3210208, 0.3868368, 0.4526528, 0.5406313, 0.6286098, 0.7380792, 0.8475487], 'ys': [0.6932574], 'r_norm': 0.0080591, 'question_start': '', 'choice_labels': ['5th', '6th', '7th', '8th', '9th', '10th', '11th JEE', '12th JEE', '11th NEET', '12th NEET']}, 'P07': {'label': 'MCQ Q1-12', 'label_hint': 'MCQ Q1-12', 'suggested_type': 'mcq_grid (questions down; options across)', 'rows': 12, 'cols': 4, 'xs': [0.1450638, 0.1672263, 0.1880457, 0.2102082], 'ys': [0.7395537, 0.7552232, 0.7711301, 0.7863248, 0.8019943, 0.8176638, 0.8333333, 0.848528, 0.8641975, 0.879867, 0.8950617, 0.9107312], 'r_norm': 0.00738751, 'question_start': '1', 'choice_labels': []}, 'P08': {'label': 'MCQ Q13-24', 'label_hint': 'MCQ Q13-24', 'suggested_type': 'mcq_grid (questions down; options across)', 'rows': 12, 'cols': 4, 'xs': [0.2988583, 0.3203492, 0.3431833, 0.3653459], 'ys': [0.7397911, 0.7554606, 0.7711301, 0.7863248, 0.8019943, 0.8176638, 0.8333333, 0.848528, 0.8641975, 0.879867, 0.8950617, 0.9107312], 'r_norm': 0.00738751, 'question_start': '13', 'choice_labels': []}, 'P09': {'label': 'MCQ Q25-36', 'label_hint': 'MCQ Q25-36', 'suggested_type': 'mcq_grid (questions down; options across)', 'rows': 12, 'cols': 4, 'xs': [0.4526528, 0.4748153, 0.4969778, 0.5191404], 'ys': [0.7397911, 0.7554606, 0.7711301, 0.7863248, 0.8019943, 0.8176638, 0.8333333, 0.848528, 0.8641975, 0.879867, 0.8950617, 0.9107312], 'r_norm': 0.00738751, 'question_start': '25', 'choice_labels': []}, 'P10': {'label': 'MCQ Q37-48', 'label_hint': 'MCQ Q37-48', 'suggested_type': 'mcq_grid (questions down; options across)', 'rows': 12, 'cols': 4, 'xs': [0.6064473, 0.6286098, 0.6507723, 0.6729349], 'ys': [0.7397911, 0.7554606, 0.7711301, 0.7863248, 0.8019943, 0.8176638, 0.8333333, 0.848528, 0.8641975, 0.879867, 0.8950617, 0.9107312], 'r_norm': 0.00738751, 'question_start': '37', 'choice_labels': []}, 'P11': {'label': 'MCQ Q49-60', 'label_hint': 'MCQ Q49-60', 'suggested_type': 'mcq_grid (questions down; options across)', 'rows': 12, 'cols': 4, 'xs': [0.7602418, 0.7817327, 0.8038952, 0.8260578], 'ys': [0.7397911, 0.7554606, 0.7711301, 0.7863248, 0.8019943, 0.8176638, 0.8333333, 0.848528, 0.8641975, 0.879867, 0.8950617, 0.9109687], 'r_norm': 0.00738751, 'question_start': '49', 'choice_labels': []}}

def _looks_like_pwnsat_template(template_gray, profile=None, groups=None, circles=None):
    """Identify the known PWNSAT form using header OCR first, then a strict layout signature."""
    profile = profile if isinstance(profile, dict) else {}
    groups = groups if isinstance(groups, list) else []
    circles = circles if isinstance(circles, list) else []
    text_parts = [str(profile.get(k, '') or '') for k in ('profile_name','template_filename','ocr_template_text','mapping_mode')]
    combined = ' '.join(text_parts).upper()
    if re.search(r'PW\s*N\s*SAT|PHYSICS\s+WALLAH|TALENT\s+HUNT\s+OF\s+INDIA', combined):
        return True, 'profile metadata/text signature'
    # Only OCR the top header band; one bounded call, not the full page.
    try:
        h, w = template_gray.shape[:2]
        header = template_gray[:max(1, int(h * 0.17)), :]
        header = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(header)
        text = pytesseract.image_to_string(header, config='--psm 6', timeout=3.0).upper()
        if re.search(r'PW\s*N\s*SAT|PHYSICS\s+WALLAH|TALENT\s+HUNT\s+OF\s+INDIA', text):
            return True, 'template header OCR signature'
    except Exception:
        pass
    # Conservative geometry signature fallback for older profiles whose OCR text was not saved.
    h, w = template_gray.shape[:2]
    ratio = w / max(1, h)
    shapes = [(int(g.get('rows', 0)), int(g.get('cols', 0))) for g in groups]
    name_like = any(24 <= r <= 29 and 14 <= c <= 20 for r, c in shapes)
    digit10x10 = sum(1 for r, c in shapes if 9 <= r <= 11 and 9 <= c <= 11)
    digit10x5 = any(9 <= r <= 11 and 4 <= c <= 6 for r, c in shapes)
    mcq_like = sum(1 for r, c in shapes if 11 <= r <= 13 and 3 <= c <= 5)
    has_dense_profile = len(circles) >= 600
    if 0.62 <= ratio <= 0.78 and name_like and digit10x10 >= 1 and digit10x5 and mcq_like >= 3 and has_dense_profile:
        return True, 'PWNSAT grid geometry signature'
    return False, 'no PWNSAT signature detected'


def _build_pwnsat_canonical_mapping(profile, template_gray, reason=''):
    """Build known PWNSAT groups from normalized template geometry, independent of noisy Hough group counts."""
    h, w = template_gray.shape[:2]
    profile = dict(profile or {})
    circles = []
    groups = []
    for gid, gd in _PWNSAT_CANONICAL_GEOMETRY.items():
        xs = [float(v) * w for v in gd['xs']]
        ys = [float(v) * h for v in gd['ys']]
        radius = max(3.0, float(gd['r_norm']) * min(h, w))
        valid_cells = None
        if gid == 'P05':
            valid_cells = set()
            for rr in range(1, 4):
                for cc in range(1, 7): valid_cells.add((rr, cc))
            for cc in (1, 2, 4, 5, 6): valid_cells.add((4, cc))
            for rr in range(5, 11):
                for cc in (2, 4, 5, 6): valid_cells.add((rr, cc))
        indices = []
        group_circles = []
        for rr, y in enumerate(ys, start=1):
            for cc, x in enumerate(xs, start=1):
                if valid_cells is not None and (rr, cc) not in valid_cells:
                    continue
                idx = len(circles)
                item = {
                    'bubble_id': f'CANON_{gid}_{rr:02d}_{cc:02d}',
                    'x_px': float(x), 'y_px': float(y), 'r_px': float(radius),
                    'x_norm': float(x / w), 'y_norm': float(y / h),
                    'r_norm': float(radius / min(h, w)),
                    'group_id': gid, 'grid_row': int(rr), 'grid_col': int(cc)
                }
                circles.append(item); indices.append(idx); group_circles.append(item)
        if not group_circles:
            raise ValueError(f'Canonical PWNSAT geometry produced no cells for {gid}')
        groups.append({
            'group_id': gid, 'circle_indices': indices,
            'rows': int(gd['rows']), 'cols': int(gd['cols']), 'count': len(indices),
            'grid_fill_ratio': round(len(indices) / max(1, int(gd['rows']) * int(gd['cols'])), 3),
            'bbox': [min(c['x_px'] for c in group_circles), min(c['y_px'] for c in group_circles),
                     max(c['x_px'] for c in group_circles), max(c['y_px'] for c in group_circles)],
            'x_centers': xs, 'y_centers': ys,
            'suggested_type': gd['suggested_type'], 'label_hint': gd['label_hint'],
            'label': gd['label'], 'question_start': gd['question_start'],
            'choice_labels': gd['choice_labels']
        })
    profile['circles'] = circles
    profile['groups'] = groups
    profile['template_width'] = int(w)
    profile['template_height'] = int(h)
    profile['detected_bubbles'] = len(circles)
    profile['group_count'] = len(groups)
    profile['mapping_mode'] = 'PWNSAT_FIXED_TEMPLATE_MAPPING (canonical geometry fallback)'
    profile['_mapping_warning'] = ''
    profile['_mapping_source'] = reason or 'canonical PWNSAT geometry'
    return profile


def analyze_template(template_file, profile_name):
    try:
        if not template_file:
            raise ValueError('Pehle blank/reference OMR template upload karein.')
        job_id = f"profile_{slugify(profile_name)}_{int(time.time())}"
        folder = PROFILE_DIR / job_id
        folder.mkdir(parents=True, exist_ok=True)
        source = str(template_file)
        # First PDF page used as layout reference; image templates are copied exactly.
        pages = paths_to_pages(source, folder, 'template')
        if not pages:
            raise ValueError('Template file se page nahi mila.')
        template_path = Path(pages[0][0])
        template_bgr = safe_read_image(template_path)
        template_gray_raw = cv2.cvtColor(template_bgr, cv2.COLOR_BGR2GRAY)
        # Use CLAHE only to help locate printed circles/text, not to estimate prefilled marks.
        template_gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(template_gray_raw)
        circles = detect_circles(template_gray)
        if len(circles) < 10:
            raise ValueError(f'Sirf {len(circles)} circle candidates mile. Clear, seedha template use karein.')
        groups, circle_group = group_circle_components(circles)
        words = []
        # Store normalized geometry to keep profile reusable across image scaling.
        H,W=template_gray.shape
        circle_rows=[]
        group_by_id={g['group_id']:g for g in groups}
        for i,(x,y,r) in enumerate(circles):
            gid=circle_group[i]
            g=group_by_id.get(gid)
            rr=cc=None
            if g:
                rr = int(np.argmin(np.abs(np.array(g['y_centers'])-y)))+1
                cc = int(np.argmin(np.abs(np.array(g['x_centers'])-x)))+1
            circle_rows.append({'bubble_id':f'B{i+1:04d}','x_norm':x/W,'y_norm':y/H,'r_norm':r/min(H,W),
                                'x_px':x,'y_px':y,'r_px':r,'group_id':gid,'grid_row':rr,'grid_col':cc})
        # Detect supported PWNSAT geometry first. That fixed mapping needs no full-page OCR.
        detected_profile_hint = {'profile_name': str(profile_name or ''), 'template_filename': Path(source).name}
        looks_pwnsat, pwnsat_reason = _looks_like_pwnsat_template(template_gray_raw, detected_profile_hint, groups, circle_rows)
        nsat_mapping = configure_pwnsat_profile(H, W, circle_rows, groups) if looks_pwnsat else None
        mapping_mode = 'GENERIC_TEMPLATE_MAPPING'
        if nsat_mapping:
            circle_rows, groups = nsat_mapping
            mapping_mode = 'PWNSAT_FIXED_TEMPLATE_MAPPING'
        elif looks_pwnsat:
            canonical = _build_pwnsat_canonical_mapping({'profile_name': profile_name, 'template_filename': Path(source).name, 'template_path': str(template_path)}, template_gray_raw, pwnsat_reason)
            circle_rows, groups = canonical['circles'], canonical['groups']
            mapping_mode = canonical['mapping_mode']
        else:
            # OCR is run only for unknown layouts, and it has a timeout to avoid hanging.
            words = ocr_words(template_gray)
            for g in groups:
                g['label_hint']=label_hint_for_group(g,words)
                g['label']=g['label_hint'] or g['group_id']
        base_scores=[bubble_metrics(template_gray_raw,c['x_px'],c['y_px'],c['r_px'])[0] for c in circle_rows]
        likely_filled=sum(s>=0.35 for s in base_scores)
        prof={
            'schema_version':1,'profile_name':str(profile_name or 'OMR Template'),'created_at':time.strftime('%Y-%m-%d %H:%M:%S'),
            'template_filename':Path(source).name,'template_path':str(template_path),
            'template_width':int(W),'template_height':int(H),'detected_bubbles':len(circle_rows),
            'group_count':len(groups),'likely_marked_bubbles_in_template':int(likely_filled),
            'groups':groups,'circles':circle_rows,'mapping_mode':mapping_mode,
            'ocr_template_text':('PWNSAT fixed geometry mapping; header signature confirmed.' if mapping_mode.startswith('PWNSAT_FIXED_TEMPLATE_MAPPING') else ocr_text(template_gray))
        }
        # Save a compact profile bundle containing reference image and profile JSON.
        profile_json=folder/'profile.json'
        profile_json.write_text(json.dumps(prof,ensure_ascii=False,indent=2),encoding='utf-8')
        bundle=PROFILE_DIR/f'{slugify(profile_name)}_template_profile.zip'
        with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(profile_json,'profile.json')
            z.write(template_path,'template_reference'+template_path.suffix.lower())
        # Overlay circle detections and group bounding boxes.
        preview=template_bgr.copy()
        for c in circle_rows:
            x,y,r=int(c['x_px']),int(c['y_px']),max(2,int(c['r_px']))
            cv2.circle(preview,(x,y),r,(0,160,255),1)
        for g in groups:
            x1,y1,x2,y2=map(int,g['bbox'])
            cv2.rectangle(preview,(x1,y1),(x2,y2),(0,180,0),2)
            cv2.putText(preview,g['group_id'],(max(0,x1),max(20,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,120,0),2,cv2.LINE_AA)
        preview_path=folder/'template_detected_preview.jpg'
        save_image(preview_path,preview)
        map_rows=[]
        for g in groups:
            map_rows.append({'Group ID':g['group_id'],'Detected label hint':g.get('label_hint',''),
                             'Your field/group label':g.get('label',g['group_id']),
                             'Suggested type':g['suggested_type'],'Rows':g['rows'],'Columns':g['cols'],
                             'Detected bubbles':g['count'],'Grid fill ratio':g['grid_fill_ratio'],'Question start (optional)':g.get('question_start','')})
        map_df=pd.DataFrame(map_rows,columns=['Group ID','Detected label hint','Your field/group label','Suggested type','Rows','Columns','Detected bubbles','Grid fill ratio','Question start (optional)'])
        warning = ''
        if likely_filled > max(5, int(len(circles)*0.03)):
            warning = f'WARNING: Template mein approx {likely_filled} bubbles pehle se dark/filled lag rahe hain. Behtar hai blank template use karein; filled sample se mark detection kam reliable ho sakti hai.'
        status=f'Template analyzed: {W}×{H}px, {len(circle_rows)} mapped bubbles, {len(groups)} grid groups ({mapping_mode}). {warning} Verify preview aur group types; mapping ek hi baar save karke is profile ko reuse kar sakte hain.'
        return prof, str(preview_path), map_df, status, str(bundle)
    except Exception as e:
        err=f'Template analysis failed: {e}'
        return {}, None, pd.DataFrame(), err, None

def save_group_mapping(profile, table):
    try:
        if not profile or not isinstance(profile,dict) or not profile.get('groups'):
            raise ValueError('Pehle template analyze karein.')
        df = table if isinstance(table,pd.DataFrame) else pd.DataFrame(table)
        mapping = {str(row.get('Group ID')):row for _,row in df.iterrows()}
        for g in profile['groups']:
            row=mapping.get(g['group_id'])
            if row is None: continue
            label=str(row.get('Your field/group label',g.get('label',g['group_id']))).strip()
            typ=str(row.get('Suggested type',g.get('suggested_type','generic_grid'))).strip()
            qstart=str(row.get('Question start (optional)',g.get('question_start',''))).strip()
            g['label']=label or g['group_id']
            g['suggested_type']=typ or 'generic_grid'
            g['question_start']=qstart
        # Rebuild profile bundle
        prof_dir=Path(profile['template_path']).parent
        pj=prof_dir/'profile.json'; pj.write_text(json.dumps(profile,ensure_ascii=False,indent=2),encoding='utf-8')
        bundle=PROFILE_DIR/f"{slugify(profile.get('profile_name','omr'))}_template_profile.zip"
        with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(pj,'profile.json')
            z.write(profile['template_path'],'template_reference'+Path(profile['template_path']).suffix.lower())
        # Return updated table to make state explicit.
        out=[]
        for g in profile['groups']:
            out.append({'Group ID':g['group_id'],'Detected label hint':g.get('label_hint',''),
                        'Your field/group label':g.get('label',g['group_id']),'Suggested type':g['suggested_type'],
                        'Rows':g['rows'],'Columns':g['cols'],'Detected bubbles':g['count'],
                        'Grid fill ratio':g['grid_fill_ratio'],'Question start (optional)':g.get('question_start','')})
        return profile, pd.DataFrame(out), f'Mapping saved. Profile name: {profile.get("profile_name")}. Next time use the saved profile ZIP.', str(bundle)
    except Exception as e:
        return profile, table, f'Mapping save failed: {e}', None

def load_profile_bundle(bundle_path, workdir):
    if not bundle_path:
        return None
    Path(workdir).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_path,'r') as z:
        names=z.namelist()
        if 'profile.json' not in names:
            raise ValueError('Profile ZIP mein profile.json nahi hai. Template Profile ZIP upload karein.')
        z.extractall(workdir)
    p=json.loads((Path(workdir)/'profile.json').read_text(encoding='utf-8'))
    refs=[x for x in Path(workdir).iterdir() if x.is_file() and x.name.startswith('template_reference')]
    if refs:
        p['template_path']=str(refs[0])
    if not Path(p.get('template_path','')).exists():
        raise ValueError('Reference template image profile ZIP mein nahi mila.')
    return p

def upgrade_profile_mapping(profile, template_gray):
    """Upgrade older or noisy profile files to canonical PWNSAT geometry if the template matches."""
    if not isinstance(profile, dict):
        profile = {}
    h, w = template_gray.shape[:2]
    circles = profile.get('circles') or []
    groups = profile.get('groups') or []
    is_pwnsat, reason = _looks_like_pwnsat_template(template_gray, profile, groups, circles)
    if is_pwnsat:
        # Important: do not derive row/column centers from noisy Hough counts. This is what
        # caused old profiles (e.g. a 27-row NAME detection) to lose all P01-P06 field mapping.
        return _build_pwnsat_canonical_mapping(profile, template_gray, reason)
    # Backward-compatible generic-template path.
    for c in circles:
        try:
            if c.get('x_norm') is not None: c['x_px'] = float(c['x_norm']) * w
            if c.get('y_norm') is not None: c['y_px'] = float(c['y_norm']) * h
            if c.get('r_norm') is not None: c['r_px'] = float(c['r_norm']) * min(h, w)
        except Exception:
            pass
    if not circles or not groups:
        profile['_mapping_warning'] = 'Profile has no circles/groups; re-analyze the blank template in Tab 1.'
        return profile
    try:
        mapped = configure_pwnsat_profile(h, w, circles, groups, force=True)
        if mapped:
            profile['circles'], profile['groups'] = mapped
            profile['mapping_mode'] = 'PWNSAT_FIXED_TEMPLATE_MAPPING (upgraded on load)'
            profile['_mapping_warning'] = ''
        else:
            profile['_mapping_warning'] = 'PWNSAT fixed mapping could not be safely inferred from this saved profile.'
    except Exception as exc:
        profile['_mapping_warning'] = f'Profile mapping upgrade failed: {exc}'
    return profile

def align_to_template(template_gray, page_gray):
    """Align a page to template and return original-page -> template transform."""
    th,tw=template_gray.shape[:2]
    ph,pw=page_gray.shape[:2]
    aspect_delta=abs((pw/max(1,ph))-(tw/max(1,th))) / max(0.001, tw/max(1,th))
    resized=cv2.resize(page_gray,(tw,th),interpolation=cv2.INTER_AREA if (pw>tw or ph>th) else cv2.INTER_CUBIC)
    scale_source=np.array([[tw/max(1,pw),0.0,0.0],[0.0,th/max(1,ph),0.0],[0.0,0.0,1.0]],dtype=np.float64)
    src=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(resized)
    ref=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(template_gray)
    status='Resize fallback — alignment needs review'; confidence=0.25
    try:
        orb=cv2.ORB_create(nfeatures=4500, scaleFactor=1.2, nlevels=8, fastThreshold=10)
        k1,d1=orb.detectAndCompute(ref,None)
        k2,d2=orb.detectAndCompute(src,None)
        if d1 is not None and d2 is not None and len(k1)>=10 and len(k2)>=10:
            matcher=cv2.BFMatcher(cv2.NORM_HAMMING)
            pairs=matcher.knnMatch(d2,d1,k=2)
            good=[]
            for pair in pairs:
                if len(pair)==2 and pair[0].distance < 0.76*pair[1].distance:
                    good.append(pair[0])
            if len(good)>=12:
                src_pts=np.float32([k2[m.queryIdx].pt for m in good]).reshape(-1,1,2)
                dst_pts=np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1,1,2)
                H,mask=cv2.findHomography(src_pts,dst_pts,cv2.RANSAC,4.0)
                if H is not None and mask is not None:
                    inliers=int(mask.sum()); ratio=inliers/max(1,len(good))
                    if inliers>=10 and ratio>=0.20:
                        warped=cv2.warpPerspective(resized,H,(tw,th),flags=cv2.INTER_CUBIC,borderValue=255)
                        confidence=min(0.99,0.35+0.65*ratio)
                        status=f'ORB aligned ({inliers} inliers; match ratio {ratio:.2f})'
                        return warped,status,confidence,aspect_delta,(H @ scale_source).tolist()
    except Exception:
        pass
    if aspect_delta > 0.025:
        status += f'; page aspect ratio differs by {aspect_delta*100:.1f}%'
    return resized,status,confidence,aspect_delta,scale_source.tolist()


def classify_group(group, bubble_rows, threshold):
    """Decode an inferred grid using common OMR conventions; retain positional detail."""
    typ=str(group.get('suggested_type','generic_grid')).lower()
    rows=int(group.get('rows',0)); cols=int(group.get('cols',0))
    selected=[b for b in bubble_rows if b['group_id']==group['group_id'] and b['state']=='FILLED']
    ambiguous=[b for b in bubble_rows if b['group_id']==group['group_id'] and b['state']=='AMBIGUOUS']
    result=[]
    if 'digit_grid' in typ or 'letter_grid' in typ:
        option_labels=[str(i) for i in range(10)] if 'digit_grid' in typ else list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
        nopts=min(rows,len(option_labels))
        for col in range(1,cols+1):
            marks=[b for b in selected if b.get('grid_col')==col]
            poss=[b for b in ambiguous if b.get('grid_col')==col]
            val=''; status='BLANK'
            if len(marks)==1 and not poss:
                rr=marks[0].get('grid_row')
                if rr and 1<=rr<=nopts: val=option_labels[rr-1]; status='ONE_MARK'
                else: status='REVIEW_ROW_MAPPING'
            elif len(marks)>1: val='|'.join(str(b.get('grid_row')) for b in marks); status='MULTIPLE'
            elif poss: status='REVIEW_AMBIGUOUS'
            result.append({'field_or_group':group.get('label',group['group_id']),'group_id':group['group_id'],
                           'response_type':'character_position','question_or_position':col,'value':val,'status':status,
                           'filled_count':len(marks),'ambiguous_count':len(poss)})
    elif 'single_row_choices' in typ:
        choices=list(group.get('choice_labels', [])) or [f'Choice {i}' for i in range(1,cols+1)]
        marks=[b for b in selected if b.get('grid_row')==1]
        poss=[b for b in ambiguous if b.get('grid_row')==1]
        vals=[]
        for b in marks:
            cc=b.get('grid_col')
            if cc and 1 <= cc <= len(choices): vals.append(choices[cc-1])
        status='BLANK' if not vals and not poss else ('ONE_MARK' if len(vals)==1 and not poss else ('MULTIPLE' if len(vals)>1 else 'REVIEW_AMBIGUOUS'))
        result.append({'field_or_group':group.get('label',group['group_id']),'group_id':group['group_id'],
                       'response_type':'single_choice','question_or_position':'selected_option',
                       'value':'|'.join(vals),'status':status,'filled_count':len(vals),'ambiguous_count':len(poss)})
    elif 'mcq_grid' in typ:
        opt_labels=list('ABCD') if cols==4 else (list('ABCDE')[:cols] if cols<=5 else [str(i+1) for i in range(cols)])
        try: start=int(float(str(group.get('question_start','')).strip()))
        except Exception: start=None
        for row in range(1,rows+1):
            marks=[b for b in selected if b.get('grid_row')==row]
            poss=[b for b in ambiguous if b.get('grid_row')==row]
            vals=[]
            for b in marks:
                cc=b.get('grid_col')
                if cc and 1<=cc<=len(opt_labels): vals.append(opt_labels[cc-1])
            status='BLANK' if not vals and not poss else ('ONE_MARK' if len(vals)==1 and not poss else ('MULTIPLE' if len(vals)>1 else 'REVIEW_AMBIGUOUS'))
            result.append({'field_or_group':group.get('label',group['group_id']),'group_id':group['group_id'],
                           'response_type':'mcq_candidate','question_or_position':(start+row-1 if start is not None else f'row_{row}'),
                           'value':'|'.join(vals),'status':status,'filled_count':len(vals),'ambiguous_count':len(poss)})
    else:
        # Generic grid: preserve every row/column location; never make a semantic guess.
        for row in range(1,rows+1):
            marks=[b for b in selected if b.get('grid_row')==row]
            poss=[b for b in ambiguous if b.get('grid_row')==row]
            result.append({'field_or_group':group.get('label',group['group_id']),'group_id':group['group_id'],
                           'response_type':'generic_grid_row','question_or_position':f'row_{row}',
                           'value':','.join(str(b.get('grid_col')) for b in marks),'status':('BLANK' if not marks and not poss else ('REVIEW_AMBIGUOUS' if poss else ('ONE_MARK' if len(marks)==1 else 'MULTIPLE'))),
                           'filled_count':len(marks),'ambiguous_count':len(poss)})
    return result

def crop_handwriting_band(gray, group):
    """Crop the handwritten text boxes immediately above the bubbles, excluding the bubble row."""
    ys = sorted(float(v) for v in group.get('y_centers', []))
    xs = sorted(float(v) for v in group.get('x_centers', []))
    x1, y1, x2, y2 = group.get('bbox', [0, 0, gray.shape[1], gray.shape[0]])
    pitch_y = float(np.median(np.diff(ys))) if len(ys) > 1 else max(12.0, (y2-y1)/max(1, int(group.get('rows', 1))))
    pitch_x = float(np.median(np.diff(xs))) if len(xs) > 1 else max(12.0, (x2-x1)/max(1, int(group.get('cols', 1))))
    top = min(ys) if ys else float(y1)
    xa = max(0, int(round((min(xs) if xs else x1) - 0.42*pitch_x)))
    xb = min(gray.shape[1], int(round((max(xs) if xs else x2) + 0.42*pitch_x)))
    # In this PWNSAT template the pen-written row ends just above the first bubble outline.
    # Previous yb=top-0.35*pitch included bubble-circle tops and polluted handwriting OCR.
    ya = max(0, int(round(top - 1.90*pitch_y)))
    yb = min(gray.shape[0], int(round(top - 0.55*pitch_y)))
    if xb <= xa or yb <= ya:
        return np.full((8, 8), 255, dtype=np.uint8)
    return gray[ya:yb, xa:xb].copy()


def name_handwriting_slot_positions(gray, reference_gray, group):
    """Estimate which of the 17 handwritten name boxes contain ink, independent of bubble marks."""
    xs = [float(v) for v in group.get('x_centers', [])]
    ys = sorted(float(v) for v in group.get('y_centers', []))
    if not xs or len(ys) < 2 or reference_gray is None or reference_gray.shape != gray.shape:
        return list(range(len(xs))), []
    px = float(np.median(np.diff(sorted(xs)))) if len(xs) > 1 else 20.0
    py = float(np.median(np.diff(ys)))
    top = min(ys)
    ya = max(0, int(round(top - 2.05*py)))
    yb = min(gray.shape[0], int(round(top - 0.55*py)))
    ref = reference_gray[ya:yb, :].astype(np.int16)
    page = gray[ya:yb, :].astype(np.int16)
    delta = np.clip(ref - page, 0, 255).astype(np.uint8)
    counts = []
    for xc in xs:
        x0 = max(0, int(round(xc - 0.34*px)))
        x1 = min(gray.shape[1], int(round(xc + 0.34*px)))
        cell = delta[:, x0:x1]
        # Remove low-level scan noise; the template difference removes printed box rules.
        ink = cv2.threshold(cell, 38, 255, cv2.THRESH_BINARY)[1]
        counts.append(int(np.count_nonzero(ink)))
    peak = max(counts, default=0)
    cutoff = max(65, int(peak * 0.22))
    active = [i for i, count in enumerate(counts) if count >= cutoff]
    # A pen stroke from a neighboring character may spill into an intentionally blank name box.
    # Prune only an isolated, low-ink interior slot when both adjacent slots have strong ink;
    # this recovers spaces such as SUHANA [blank box] SHAIKH while keeping the original counts
    # available in diagnostics for human review.
    active_set = set(active)
    for i in range(1, len(counts)-1):
        if i in active_set and (i-1) in active_set and (i+1) in active_set:
            local_ref = min(counts[i-1], counts[i+1])
            if counts[i] < 0.50 * local_ref and counts[i] < 0.55 * peak:
                active_set.discard(i)
    active = sorted(active_set)
    return active, counts


def _clean_name_ocr_text(value):
    value = re.sub(r'[^A-Z ]', '', str(value or '').upper())
    return re.sub(r'\s+', ' ', value).strip()


def _tesseract_text(image, psm, whitelist, timeout_seconds):
    if image is None or image.size == 0:
        return ''
    try:
        raw = pytesseract.image_to_string(
            image,
            config=f'--psm {int(psm)} -c preserve_interword_spaces=1 -c tessedit_char_whitelist={whitelist}',
            timeout=float(timeout_seconds),
        ).strip()
        return raw
    except Exception:
        return ''


def _name_glyph_montage(gray, reference_gray, group, use_difference=False):
    """Create a line of fixed-width name-box crops so OCR does not see bubble circles."""
    xs = [float(v) for v in group.get('x_centers', [])]
    ys = sorted(float(v) for v in group.get('y_centers', []))
    if not xs or len(ys) < 2:
        return None
    px = float(np.median(np.diff(sorted(xs)))) if len(xs) > 1 else 20.0
    py = float(np.median(np.diff(ys)))
    top = min(ys)
    ya = max(0, int(round(top - 1.55*py)))
    yb = min(gray.shape[0], int(round(top - 0.65*py)))
    tiles = []
    for xc in xs:
        x0 = max(0, int(round(xc - 0.38*px)))
        x1 = min(gray.shape[1], int(round(xc + 0.38*px)))
        crop = gray[ya:yb, x0:x1]
        if crop.size == 0:
            tiles.append(np.full((96, 62), 255, dtype=np.uint8)); continue
        if use_difference and reference_gray is not None and reference_gray.shape == gray.shape:
            ref_crop = reference_gray[ya:yb, x0:x1]
            ink = np.clip(ref_crop.astype(np.int16) - crop.astype(np.int16), 0, 255).astype(np.uint8)
            bin_ink = cv2.threshold(ink, 35, 255, cv2.THRESH_BINARY)[1]
            tile = 255 - bin_ink
        else:
            enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(crop)
            tile = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
        tile = cv2.resize(tile, (54, 84), interpolation=cv2.INTER_CUBIC)
        # Recenter the glyph in a fixed tile; this drops uneven margins while keeping slot order.
        ink_coords = cv2.findNonZero(255 - tile)
        canvas = np.full((96, 62), 255, dtype=np.uint8)
        if ink_coords is not None:
            bx, by, bw, bh = cv2.boundingRect(ink_coords)
            glyph = tile[by:by+bh, bx:bx+bw]
            if bw > 0 and bh > 0:
                scale = min(50.0/max(1, bw), 76.0/max(1, bh))
                glyph = cv2.resize(glyph, (max(1, int(bw*scale)), max(1, int(bh*scale))), interpolation=cv2.INTER_CUBIC)
                yy = (96-glyph.shape[0])//2; xx = (62-glyph.shape[1])//2
                canvas[yy:yy+glyph.shape[0], xx:xx+glyph.shape[1]] = glyph
        tiles.append(canvas)
    montage = np.full((116, len(tiles)*66+12), 255, dtype=np.uint8)
    for j, tile in enumerate(tiles):
        montage[10:106, 8+j*66:8+j*66+62] = tile
    return montage


def format_name_candidate_by_slots(candidate, active_positions, cols=17):
    """Restore spaces where handwritten name boxes are blank, but only when lengths agree exactly."""
    letters = re.sub(r'[^A-Z]', '', str(candidate or '').upper())
    if not active_positions or len(active_positions) > int(cols):
        return _clean_name_ocr_text(candidate)
    # OCR sometimes inserts a spurious character where the handwritten name has a blank box
    # between first and last names. If there is exactly one extra OCR character and exactly one
    # internal blank slot, drop only the character at that boundary. The result remains a proposal.
    if len(letters) == len(active_positions) + 1:
        gap_boundaries = [j for j in range(1, len(active_positions))
                          if int(active_positions[j]) - int(active_positions[j-1]) > 1]
        if len(gap_boundaries) == 1:
            j = gap_boundaries[0]
            letters = letters[:j] + letters[j+1:]
    if len(letters) != len(active_positions):
        return _clean_name_ocr_text(candidate)
    slots = [' '] * int(cols)
    for pos, char in zip(active_positions, letters):
        if 0 <= int(pos) < len(slots):
            slots[int(pos)] = char
    return re.sub(r'\s+', ' ', ''.join(slots)).strip()



def build_name_bubble_proposal(aligned_gray, template_gray, profile, group, confirmed_or_candidate_name):
    """Return a separate aligned preview with the P01 bubble grid redrawn from a name proposal.

    This is a proposal image only. The input/original scan is not modified or overwritten.
    Human verification is still required before the corrected PNG export is created.
    """
    name=str(confirmed_or_candidate_name or '').upper().strip()
    if (not name or '?' in name or len(name)>int(group.get('cols',17)) or
        not re.fullmatch(r'[A-Z ]+',name)):
        raise ValueError('Name candidate is incomplete or contains unsupported characters.')
    cols=int(group.get('cols',17))
    slots=list(name)
    if len(slots)>cols:
        raise ValueError(f'Name candidate has {len(slots)} characters including spaces; grid has {cols} slots.')
    slots += [' ']*(cols-len(slots))
    if aligned_gray.shape != template_gray.shape:
        raise ValueError('Aligned image and template dimensions differ.')
    gid=str(group.get('group_id','P01'))
    cells=[c for c in profile.get('circles',[]) if str(c.get('group_id'))==gid]
    if not cells:
        raise ValueError(f'No bubble cells mapped for name group {gid}.')
    result=aligned_gray.copy()
    th,tw=template_gray.shape[:2]
    # Restore the entire P01 grid rectangle from the blank template. This removes oversized
    # or off-centre incorrect fills instead of only cleaning the centre of each circle.
    x0,y0,x1,y1=mapped_circle_bounds(cells,tw,th,pad_factor=1.35)
    result[y0:y1,x0:x1]=template_gray[y0:y1,x0:x1]
    # Refill the bubble corresponding to each confirmed letter at its box/column.
    for col, symbol in enumerate(slots,1):
        if symbol == ' ':
            continue
        row=ord(symbol)-ord('A')+1
        hit=[c for c in cells if int(c.get('grid_col') or -1)==col and int(c.get('grid_row') or -1)==row]
        if len(hit)!=1:
            raise ValueError(f'Name bubble mapping is not unique for slot {col}, letter {symbol}.')
        c=hit[0]
        x=int(round(float(c['x_norm'])*tw)); y=int(round(float(c['y_norm'])*th))
        r=max(3,int(round(float(c['r_norm'])*min(th,tw))))
        draw_filled_bubble(result,x,y,r,fill_ratio=0.80)
    return result

def _remove_name_grid_rules(crop):
    """Remove long printed box rules while retaining handwritten strokes."""
    if crop is None or crop.size == 0:
        return crop
    gray = np.asarray(crop, dtype=np.uint8)
    inv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    h, w = inv.shape[:2]
    vlen = max(8, int(h * 0.68))
    hlen = max(70, int(w * 0.16))
    vlines = cv2.morphologyEx(inv, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, vlen)))
    hlines = cv2.morphologyEx(inv, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (hlen, 1)))
    cleaned = cv2.subtract(cv2.subtract(inv, vlines), hlines)
    # A scan shifted by a few pixels can make template-difference text too sparse; use original threshold in that case.
    if np.count_nonzero(cleaned) < max(12, int(gray.size * 0.002)):
        cleaned = inv
    return 255 - cleaned


def _trocr_name_line(gray, group, reference_gray=None, timeout_note=None):
    """Recognize the full handwritten name line with the optional handwriting-trained model."""
    if _TROCR_MODEL is None or _TROCR_PROCESSOR is None:
        return '', 'TrOCR not loaded'
    try:
        import torch
        from PIL import Image
        crop = crop_handwriting_band(gray, group)
        if crop is None or crop.size == 0:
            return '', 'Name crop is empty'
        clean = _remove_name_grid_rules(crop)
        # Use the clean rule-suppressed line; a generous white canvas prevents extreme aspect compression.
        coords = cv2.findNonZero(255 - clean)
        if coords is not None:
            x, y, w, h = cv2.boundingRect(coords)
            tight = clean[max(0,y-2):min(clean.shape[0],y+h+2), max(0,x-4):min(clean.shape[1],x+w+4)]
            if tight.size and min(tight.shape) >= 5:
                clean = tight
        h, w = clean.shape[:2]
        pad_y = max(16, int(w * 0.34) - h // 2)
        canvas = cv2.copyMakeBorder(clean, pad_y, pad_y, 12, 12, cv2.BORDER_CONSTANT, value=255)
        # Avoid tiny source dimensions without changing the original record.
        if canvas.shape[1] < 160:
            canvas = cv2.resize(canvas, (160, max(48, int(canvas.shape[0]*160/canvas.shape[1]))), interpolation=cv2.INTER_CUBIC)
        pil = Image.fromarray(canvas).convert('RGB')
        inputs = _TROCR_PROCESSOR(images=pil, return_tensors='pt')
        pixel_values = inputs.pixel_values.to(_TROCR_DEVICE)
        with torch.inference_mode():
            generated = _TROCR_MODEL.generate(pixel_values, max_new_tokens=32, num_beams=3, early_stopping=True)
        raw = _TROCR_PROCESSOR.batch_decode(generated, skip_special_tokens=True)[0]
        text = _clean_name_ocr_text(raw)
        return text, f'TrOCR handwriting candidate; verify against original; model={_TROCR_MODEL_ID}'
    except Exception as exc:
        return '', f'TrOCR prediction failed ({type(exc).__name__}: {str(exc)[:120]})'


def _tesseract_name_candidate_pool(gray, group, reference_gray=None, timeout_seconds=1.25):
    """Bounded ensemble of text-line OCR variants; no unbounded per-character OCR loops."""
    crop = crop_handwriting_band(gray, group)
    if crop is None or crop.size == 0:
        return []
    candidates=[]
    def _run(img, label, psm, scale=5.0):
        if img is None or img.size == 0:
            return
        up=cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        up=cv2.copyMakeBorder(up, 24, 24, 32, 32, cv2.BORDER_CONSTANT, value=255)
        try:
            raw=pytesseract.image_to_string(up, config=f'--psm {int(psm)} -c preserve_interword_spaces=1 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ ', timeout=float(timeout_seconds))
            text=_clean_name_ocr_text(raw)
            letters=re.sub(r'[^A-Z]','',text)
            if len(letters)>=3:
                candidates.append({'text':text,'source':label,'letters':letters,'count':len(letters)})
        except Exception:
            return
    # One original grayscale line and its binarized form.
    enhanced=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(4,4)).apply(crop)
    _run(enhanced,'Tesseract grayscale line',7,5.0)
    # Removing box rules is especially helpful on ruled letter-name boxes.
    clean=_remove_name_grid_rules(crop)
    _run(clean,'Tesseract gridline-removed line',6,5.0)
    # Template difference suppresses printed magenta/grey box rules when registration is accurate.
    if reference_gray is not None and reference_gray.shape == gray.shape:
        ref_crop=crop_handwriting_band(reference_gray,group)
        if ref_crop is not None and ref_crop.shape == crop.shape:
            delta=np.clip(ref_crop.astype(np.int16)-crop.astype(np.int16),0,255).astype(np.uint8)
            ink=cv2.threshold(delta,30,255,cv2.THRESH_BINARY)[1]
            inv=255-ink
            # Remove long residual rules in the delta map without deleting short letter strokes.
            cleaned=_remove_name_grid_rules(inv)
            _run(cleaned,'Tesseract template-difference line',7,5.0)
    # A strict binarized line is useful on low-contrast scans.
    th=cv2.threshold(enhanced,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)[1]
    _run(th,'Tesseract Otsu line',6,5.0)
    return candidates


def _legacy_name_ocr_candidate(gray, group, reference_gray=None, timeout_seconds=1.2):
    """Keep the older reference-difference + montage pipeline as a candidate source, not a hard decision."""
    crop=crop_handwriting_band(gray,group)
    if crop is None or crop.size==0:
        return '', 'legacy OCR crop empty'
    active_positions,_=name_handwriting_slot_positions(gray,reference_gray,group)
    target_len=len(active_positions)
    candidates=[]
    if reference_gray is not None and reference_gray.shape==gray.shape:
        ref_crop=crop_handwriting_band(reference_gray,group)
        if ref_crop.shape==crop.shape:
            ink=np.clip(ref_crop.astype(np.int16)-crop.astype(np.int16),0,255).astype(np.uint8)
            up=cv2.resize(ink,None,fx=4.0,fy=4.0,interpolation=cv2.INTER_CUBIC)
            binary=255-cv2.threshold(up,45,255,cv2.THRESH_BINARY)[1]
            text=_clean_name_ocr_text(_tesseract_text(binary,7,'ABCDEFGHIJKLMNOPQRSTUVWXYZ ',timeout_seconds))
            if len(re.sub(r'[^A-Z]','',text))>=3:
                candidates.append(('reference-difference line OCR',text))
            first_n=len(re.sub(r'[^A-Z]','',candidates[0][1])) if candidates else 0
            if not candidates or abs(first_n-target_len)>0:
                montage_gray=_name_glyph_montage(gray,reference_gray,group,use_difference=False)
                text=_clean_name_ocr_text(_tesseract_text(montage_gray,7,'ABCDEFGHIJKLMNOPQRSTUVWXYZ ',timeout_seconds))
                if len(re.sub(r'[^A-Z]','',text))>=3:
                    candidates.append(('box montage OCR',text))
            if not candidates or min(abs(len(re.sub(r'[^A-Z]','',t))-target_len) for _,t in candidates)>0:
                montage_diff=_name_glyph_montage(gray,reference_gray,group,use_difference=True)
                text=_clean_name_ocr_text(_tesseract_text(montage_diff,7,'ABCDEFGHIJKLMNOPQRSTUVWXYZ ',timeout_seconds))
                if len(re.sub(r'[^A-Z]','',text))>=3:
                    candidates.append(('difference montage OCR',text))
    if not candidates:
        up=cv2.resize(cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(crop),None,fx=4.0,fy=4.0,interpolation=cv2.INTER_CUBIC)
        text=_clean_name_ocr_text(_tesseract_text(up,13,'ABCDEFGHIJKLMNOPQRSTUVWXYZ ',timeout_seconds))
        if len(re.sub(r'[^A-Z]','',text))>=3:
            candidates.append(('fallback line OCR',text))
    if not candidates:
        return '', 'legacy OCR found no candidate'
    def score(item):
        method,text=item
        n=len(re.sub(r'[^A-Z]','',text))
        gaps=sum(1 for j in range(1,len(active_positions)) if int(active_positions[j])-int(active_positions[j-1])>1)
        adjusted=n-1 if n==target_len+1 and gaps==1 else n
        source=0 if 'reference-difference' in method else (1 if 'box montage' in method else 2)
        return (abs(adjusted-target_len),source,-n)
    candidates.sort(key=score)
    method,text=candidates[0]
    formatted=format_name_candidate_by_slots(text,active_positions,cols=int(group.get('cols',17)))
    return formatted or text, f'legacy {method}; count={len(re.sub(r"[^A-Z]", "", text))}; uncalibrated'


def bounded_handwriting_ocr(gray, group, field_kind='text', timeout_seconds=2.5,
                            reference_gray=None, use_trocr=False, bubble_grid_results=None):
    """Read a handwritten field using TrOCR when available and a bounded Tesseract ensemble as fallback.

    Confidence is deliberately reported as uncalibrated. OCR candidates always need review.
    """
    crop = crop_handwriting_band(gray, group)
    if crop.size == 0 or min(crop.shape[:2]) < 8:
        return '', -1.0, 'OCR SKIPPED: handwriting crop too small'
    if field_kind == 'letters':
        active_positions, _ = name_handwriting_slot_positions(gray, reference_gray, group)
        target_len = len(active_positions)
        model_candidate = ''
        model_note = ''
        if use_trocr and _TROCR_MODEL is not None:
            model_candidate, model_note = _trocr_name_line(gray, group, reference_gray)
            model_letters = re.sub(r'[^A-Z]', '', str(model_candidate or '').upper())
            if 3 <= len(model_letters) <= 17:
                formatted = format_name_candidate_by_slots(model_candidate, active_positions, cols=int(group.get('cols',17))) if len(model_letters)==target_len else _clean_name_ocr_text(model_candidate)
                return formatted or model_candidate, -1.0, model_note + f'; letter-count={len(model_letters)}, visible handwriting boxes={target_len}'
        candidates = _tesseract_name_candidate_pool(gray, group, reference_gray, timeout_seconds=min(float(timeout_seconds),1.25))
        legacy_text, legacy_note = _legacy_name_ocr_candidate(gray, group, reference_gray, timeout_seconds=min(float(timeout_seconds),1.2))
        if legacy_text:
            legacy_letters=re.sub(r'[^A-Z]','',legacy_text.upper())
            candidates.append({'text':legacy_text,'source':'Legacy reference/montage ensemble','letters':legacy_letters,'count':len(legacy_letters)})
        # Keep a lower-threshold difference crop as another candidate; never trust one OCR path alone.
        if reference_gray is not None and reference_gray.shape == gray.shape:
            ref_crop=crop_handwriting_band(reference_gray,group)
            if ref_crop.shape == crop.shape:
                delta=np.clip(ref_crop.astype(np.int16)-crop.astype(np.int16),0,255).astype(np.uint8)
                up=cv2.resize(delta,None,fx=4.0,fy=4.0,interpolation=cv2.INTER_CUBIC)
                inv=cv2.threshold(up,35,255,cv2.THRESH_BINARY)[1]
                inv=255-inv
                inv=cv2.copyMakeBorder(inv,24,24,32,32,cv2.BORDER_CONSTANT,value=255)
                try:
                    raw=pytesseract.image_to_string(inv,config='--psm 7 -c preserve_interword_spaces=1 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ ',timeout=min(float(timeout_seconds),1.25))
                    text=_clean_name_ocr_text(raw); letters=re.sub(r'[^A-Z]','',text)
                    if len(letters)>=3: candidates.append({'text':text,'source':'Tesseract reference-difference line','letters':letters,'count':len(letters)})
                except Exception: pass
        if not candidates:
            if model_note:
                return '', -1.0, model_note + '; Tesseract found no usable candidate; bubble result retained'
            return '', -1.0, 'No usable handwritten-name OCR candidate; bubble result retained and ? marks require review.'
        # Prefer candidates fitting the visibly inked character-box count. Frequent identical readings get a small vote bonus.
        frequencies={}
        for item in candidates:
            key=item['letters']
            frequencies[key]=frequencies.get(key,0)+1
        def _score(item):
            n=item['count']
            score=-2.0*abs(n-target_len)
            if n==target_len: score+=2.0
            if target_len and len(item['text'].split())>=2: score+=0.25
            score+=0.35*(frequencies.get(item['letters'],1)-1)
            src=item['source']
            if 'Legacy reference/montage' in src: score+=0.95
            elif 'reference-difference' in src: score+=0.70
            elif 'gridline-removed' in src: score+=0.45
            elif 'grayscale' in src: score+=0.25
            return score
        candidates.sort(key=_score,reverse=True)
        best=candidates[0]
        formatted=(format_name_candidate_by_slots(best['text'],active_positions,cols=int(group.get('cols',17)))
                   if best['count']==target_len else _clean_name_ocr_text(best['text']))
        vote_support=int(frequencies.get(best['letters'],1))
        note=f"{best['source']}; count={best['count']}, visible handwriting boxes={target_len}; candidate-support={vote_support}/{len(candidates)}; confidence uncalibrated; compare bubble reading and review manually"
        if model_note:
            note += '; ' + model_note
        return formatted or best['text'], -1.0, note

    # Numeric OCR remains an aid only; the bubble value stays separately auditable.
    whitelist = '0123456789/' if field_kind == 'digits' else 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789/ -'
    up = cv2.resize(cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(crop), None, fx=3.0, fy=3.0, interpolation=cv2.INTER_CUBIC)
    raw = _tesseract_text(up, 7, whitelist, timeout_seconds)
    cleaned = re.sub(r'[^0-9/]', '', raw.upper()) if field_kind == 'digits' else re.sub(r'[^A-Z0-9/ -]', '', raw.upper()).strip()
    if not cleaned:
        return '', -1.0, 'OCR returned no usable text; bubble result retained'
    return cleaned, -1.0, 'OCR confidence not calibrated; candidate only; bubble result retained'

def crop_text_above_grid(gray, group):
    """Crop the likely handwritten text band immediately above a bubble grid."""
    x1,y1,x2,y2=group.get('bbox',[0,0,gray.shape[1],gray.shape[0]])
    yc=group.get('y_centers',[])
    if len(yc)>1:
        pitch=float(np.median(np.diff(sorted(yc))))
    else:
        pitch=max(8.0, (y2-y1)/max(1,int(group.get('rows',1))))
    band=max(24.0, min(100.0, pitch*3.2))
    margin=max(6.0, pitch*0.5)
    xa=max(0,int(x1-margin)); xb=min(gray.shape[1],int(x2+margin)+1)
    ya=max(0,int(y1-band)); yb=min(gray.shape[0],int(y1+max(2,pitch*0.15)))
    if xb<=xa or yb<=ya:
        return np.full((2,2),255,dtype=np.uint8)
    crop=gray[ya:yb,xa:xb]
    if crop.size:
        crop=cv2.resize(crop,None,fx=2.0,fy=2.0,interpolation=cv2.INTER_CUBIC)
    return crop

def handwritten_name_slot_ocr(gray, group, reference_gray=None, grid_results=None):
    """Read handwritten name characters from the boxes above the letter grid.

    Fast OCR runs first. If a letter is blank/ambiguous, the handwriting conflicts with
    the bubble, or OCR confidence is weak, several small horizontal crops and Tesseract
    segmentation modes are voted. This function is used for Student Name only.
    """
    xs = [float(v) for v in group.get('x_centers', [])]
    ys = [float(v) for v in group.get('y_centers', [])]
    cols = int(group.get('cols', len(xs) or 0))
    if len(xs) != cols or len(ys) < 2 or cols < 1:
        return [{'char': '', 'confidence': -1.0, 'ink_fraction': 0.0, 'support': 0} for _ in range(max(0, cols))]
    px = float(np.median(np.diff(sorted(xs)))) if len(xs) > 1 else max(12.0, (group['bbox'][2]-group['bbox'][0])/max(cols,1))
    py = float(np.median(np.diff(sorted(ys))))
    top_bubble_y = min(ys)
    ya = max(0, int(round(top_bubble_y - 1.48 * py + 2)))
    yb = min(gray.shape[0], int(round(top_bubble_y - 0.34 * py - 2)))
    ordered = sorted(grid_results or [], key=lambda r: int(r.get('question_or_position', 0)))
    results = []
    for slot_index, xc in enumerate(xs):
        xa = max(0, int(round(xc - 0.51 * px)))
        xb = min(gray.shape[1], int(round(xc + 0.51 * px)))
        crop = gray[ya:yb, xa:xb]
        bubble_row = ordered[slot_index] if slot_index < len(ordered) else {}
        slot_status = str(bubble_row.get('status', 'REVIEW_AMBIGUOUS'))
        bubble_char = str(bubble_row.get('value') or '').strip().upper() if slot_status == 'ONE_MARK' else ''
        if crop.size == 0:
            results.append({'char':'','confidence':-1.0,'ink_fraction':0.0,'support':0})
            continue
        inner = crop[3:-3,3:-3] if crop.shape[0] > 7 and crop.shape[1] > 7 else crop
        ink_fraction = 0.0
        if reference_gray is not None and reference_gray.shape == gray.shape:
            ref_crop = reference_gray[ya:yb,xa:xb]
            ref_inner = ref_crop[3:-3,3:-3] if ref_crop.shape[0] > 7 and ref_crop.shape[1] > 7 else ref_crop
            if ref_inner.shape == inner.shape and inner.size:
                added_ink = ref_inner.astype(np.int16) - inner.astype(np.int16)
                ink_fraction = float(np.mean(added_ink > 40))
            else:
                ink_fraction = float(np.mean(inner < 110)) if inner.size else 0.0
        else:
            ink_fraction = float(np.mean(inner < 110)) if inner.size else 0.0
        if ink_fraction < 0.065:
            results.append({'char':'','confidence':-1.0,'ink_fraction':ink_fraction,'support':0})
            continue

        enlarged = cv2.resize(crop, None, fx=5.0, fy=5.0, interpolation=cv2.INTER_CUBIC)
        enlarged = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(enlarged)
        _, binary = cv2.threshold(enlarged, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        trials = []
        # Quick pass, similar to the original scanner but with a little more padding.
        for image in (enlarged, binary):
            padded = cv2.copyMakeBorder(image, 24, 24, 24, 24, cv2.BORDER_CONSTANT, value=255)
            try:
                data = pytesseract.image_to_data(
                    padded,
                    config='--psm 10 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ',
                    output_type=pytesseract.Output.DICT,
                    timeout=2,
                )
                for token, conf in zip(data.get('text', []), data.get('conf', [])):
                    token = re.sub(r'[^A-Z]', '', str(token).upper())
                    try: score = float(conf)
                    except Exception: score = -1.0
                    if len(token) == 1 and score >= 0:
                        trials.append((token, score))
            except Exception:
                pass
        if trials:
            counts = {}
            for ch, score in trials: counts[ch] = counts.get(ch, 0) + 1
            trials.sort(key=lambda item:(item[1] + 7*(counts.get(item[0],0)>1), counts.get(item[0],0)), reverse=True)
            initial_char, initial_conf = trials[0]
            initial_support = counts.get(initial_char,0)
        else:
            initial_char, initial_conf, initial_support = '', -1.0, 0

        # Do enhanced voting for problematic slots, including a single bubble that conflicts
        # with the written character. The hand-written line is authoritative for this field.
        needs_rescue = (
            slot_status != 'ONE_MARK' or not initial_char or
            (bubble_char and initial_char and initial_char != bubble_char and initial_conf < 98)
        )
        votes = {}
        if needs_rescue:
            # The handwriting is not always centred over the bubble centres. Try two offsets
            # first; expand the search only if they do not agree, to keep batch speed reasonable.
            def collect_name_votes(dx_values, psm_values, include_binary=True, scales=(3.0,)):
                local_votes = {}
                for dx in dx_values:
                    sx0 = max(0, int(round(xc - 0.51*px + dx)))
                    sx1 = min(gray.shape[1], int(round(xc + 0.51*px + dx)))
                    shifted = gray[ya:yb, sx0:sx1]
                    if shifted.size == 0:
                        continue
                    for scale in scales:
                        enlarged_vote = cv2.resize(shifted, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
                        _, shifted_binary = cv2.threshold(enlarged_vote, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                        variants = (enlarged_vote, shifted_binary) if include_binary else (enlarged_vote,)
                        for image in variants:
                            padded = cv2.copyMakeBorder(image, 42, 42, 42, 42, cv2.BORDER_CONSTANT, value=255)
                            for psm in psm_values:
                                try:
                                    raw = pytesseract.image_to_string(
                                        padded,
                                        config=f'--psm {psm} -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ',
                                        timeout=1.5,
                                    )
                                    token = re.sub(r'[^A-Z]', '', str(raw).upper())
                                    if len(token) == 1:
                                        local_votes[token] = local_votes.get(token, 0) + 1
                                except Exception:
                                    pass
                return local_votes
            # Small glyph crops work better at 3x with single-character/sparse-text segmentation
            # than at 6x. Start with the unshifted slot and neighboring alignments.
            votes = collect_name_votes((-4, 0, 4), (8, 13), True, scales=(3.0,))
            ranked_fast = sorted(votes.items(), key=lambda item:(item[1], item[0]), reverse=True)
            fast_has_consensus = bool(ranked_fast and ranked_fast[0][1] >= 2 and
                                      (len(ranked_fast) == 1 or ranked_fast[0][1] > ranked_fast[1][1]))
            if not fast_has_consensus:
                extra_votes = collect_name_votes((-12, -8, 0, 8), (8, 13, 10), True, scales=(3.0, 5.0))
                for ch, count in extra_votes.items():
                    votes[ch] = votes.get(ch, 0) + count
        if votes:
            ranked = sorted(votes.items(), key=lambda item:(item[1], item[0]), reverse=True)
            vote_char, vote_count = ranked[0]
            runner_count = ranked[1][1] if len(ranked) > 1 else 0
            # A clear majority of independent crop/mode trials is stronger than a lone OCR hit.
            if vote_count >= 2 and vote_count > runner_count:
                vote_conf = min(98.0, 76.0 + 6.0*vote_count)
                if vote_char == initial_char and initial_conf >= 80:
                    vote_conf = max(vote_conf, min(98.0, initial_conf + 12.0))
                if (not initial_char or vote_conf >= initial_conf or vote_char == initial_char):
                    results.append({'char':vote_char,'confidence':vote_conf,'ink_fraction':ink_fraction,
                                    'support':max(vote_count, initial_support if vote_char == initial_char else 0)})
                    continue
        results.append({'char':initial_char,'confidence':float(initial_conf),'ink_fraction':ink_fraction,
                        'support':int(initial_support)})
    return results

def fuse_name_bubbles_with_handwriting(grid_results, handwritten_slots):
    """Return bubble-derived name; OCR is only a cross-check and never edits the value.

    A real OMR name is encoded by the letter bubbles. Handwriting OCR may be noisy or
    shifted by a slot, so it must not insert, delete, or replace characters in the final
    bubble-derived name. Conflicts and visible handwriting over a blank bubble are flagged.
    """
    ordered = sorted(grid_results, key=lambda r: int(r.get('question_or_position', 0)))
    pieces = []
    review_positions = []
    ocr_conflicts = []
    for idx, rr in enumerate(ordered):
        status = str(rr.get('status', 'BLANK'))
        bubble_char = str(rr.get('value') or '').strip().upper()
        hand = handwritten_slots[idx] if idx < len(handwritten_slots) else {'char': '', 'confidence': -1, 'ink_fraction': 0.0, 'support': 0}
        hand_char = str(hand.get('char') or '').strip().upper()
        try: ink = float(hand.get('ink_fraction', 0))
        except Exception: ink = 0.0

        if status == 'ONE_MARK' and len(bubble_char) == 1 and bubble_char in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
            # The bubbled letter is authoritative. Never overwrite it with OCR.
            pieces.append(bubble_char)
            if hand_char in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ' and hand_char != bubble_char and ink >= 0.08:
                review_positions.append(idx + 1)
                ocr_conflicts.append(f'{idx+1}:{bubble_char}/{hand_char}')
        elif status == 'BLANK':
            # OCR-looking ink in a blank bubble slot may indicate an omitted/missed mark.
            # Keep the slot blank in the machine reading and ask for review instead.
            pieces.append(' ')
            if ink >= 0.08 or hand_char in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
                review_positions.append(idx + 1)
        else:
            # Multiple marks and uncertain bubbles are unresolved; never guess from OCR.
            pieces.append('?')
            review_positions.append(idx + 1)

    value = re.sub(r'\s+', ' ', ''.join(pieces)).strip()
    value = re.sub(r'\s*\?\s*', '?', value)
    status_text = 'BUBBLE-ONLY FINAL; handwriting OCR is comparison-only and cannot overwrite bubbles'
    if ocr_conflicts:
        status_text += '; OCR conflicts (slot:bubble/OCR): ' + ','.join(ocr_conflicts)
    if review_positions:
        status_text += '; REVIEW SLOTS: ' + ','.join(map(str, sorted(set(review_positions))))
    return value, status_text


def extract_numeric_candidate(text, expected_length):
    """Return a numeric OCR candidate only when one token has the expected number of digits."""
    if not text or expected_length < 1:
        return ''
    candidates=[]
    # Avoid turning arbitrary words into digits: require at least two numeric characters in the token.
    tokens=re.findall(r'[0-9OQIL|SZB]{3,}',str(text).upper())
    trans=str.maketrans({'O':'0','Q':'0','I':'1','L':'1','|':'1','S':'5','Z':'2','B':'8'})
    for tok in tokens:
        normalized=tok.translate(trans)
        digits=''.join(ch for ch in normalized if ch.isdigit())
        actual_digits=sum(ch.isdigit() for ch in tok)
        if len(digits)==int(expected_length) and actual_digits>=max(2,int(expected_length*0.35)):
            candidates.append((actual_digits,digits))
    if not candidates:
        return ''
    candidates.sort(key=lambda t:(t[0],len(t[1])),reverse=True)
    # Repeated identical candidates are fine; conflicting candidates are not.
    values={d for _,d in candidates}
    return candidates[0][1] if len(values)==1 else ''

def propose_numeric_bubble_correction(aligned_gray, template_bgr, profile, group, digit_string):
    """Create a separate proposal image using the blank template to restore only this grid."""
    th,tw=aligned_gray.shape[:2]
    proposed=cv2.cvtColor(aligned_gray,cv2.COLOR_GRAY2BGR)
    circles=[c for c in profile.get('circles',[]) if c.get('group_id')==group.get('group_id') and c.get('grid_row') and c.get('grid_col')]
    # Restore the complete bubble cell area from the blank template before applying proposed marks.
    for c in circles:
        x=int(round(float(c['x_norm'])*tw)); y=int(round(float(c['y_norm'])*th))
        r=max(3,int(round(float(c['r_norm'])*min(th,tw))))
        y0=max(0,y-r); y1=min(th,y+r+1); x0=max(0,x-r); x1=min(tw,x+r+1)
        yy,xx=np.ogrid[y0:y1,x0:x1]
        mask=(xx-x)**2+(yy-y)**2 <= (0.92*r)**2
        roi=proposed[y0:y1,x0:x1]
        src=template_bgr[y0:y1,x0:x1]
        roi[mask]=src[mask]
    # Digits 0-9 are assumed to be listed vertically from row 1 to row 10.
    for pos,ch in enumerate(digit_string, start=1):
        if not ch.isdigit():
            continue
        wanted_row=int(ch)+1
        match=[c for c in circles if int(c.get('grid_col') or -1)==pos and int(c.get('grid_row') or -1)==wanted_row]
        if len(match)==1:
            c=match[0]
            x=int(round(float(c['x_norm'])*tw)); y=int(round(float(c['y_norm'])*th))
            r=max(3,int(round(float(c['r_norm'])*min(th,tw))))
            draw_filled_bubble(proposed,x,y,r,color=(0,0,0),fill_ratio=0.80)
    return proposed

def active_slot_bubble_name(grid_results, active_positions, cols=17):
    """Return only the name bubbles corresponding to visibly written boxes, plus a quality score."""
    ordered=sorted(grid_results or [],key=lambda r:int(r.get('question_or_position',0)))
    slots=[' ']*int(cols)
    chars=[]
    known=0
    for pos in active_positions:
        pos=int(pos)
        rr=ordered[pos] if 0<=pos<len(ordered) else {}
        status=str(rr.get('status','REVIEW_AMBIGUOUS'))
        value=str(rr.get('value') or '').strip().upper()
        if status=='ONE_MARK' and len(value)==1 and value in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
            ch=value; known+=1
        else:
            ch='?'
        if 0<=pos<len(slots): slots[pos]=ch
        chars.append(ch)
    text=re.sub(r'\s+',' ',''.join(slots)).strip()
    quality=(known/len(active_positions)) if active_positions else 0.0
    return text, quality, ''.join(chars), known


def choose_name_value(bubble_name, bubble_quality, bubble_active, ocr_name, ocr_note, active_count, use_trocr=False):
    """Bubble-first name selection. OCR is only a comparison candidate, never an auto-overwrite."""
    bubble = str(bubble_name or '').strip()
    clean_ocr = _clean_name_ocr_text(ocr_name)
    letters = re.sub(r'[^A-Z]', '', clean_ocr)
    note = str(ocr_note or '')
    if bubble and bubble not in ('?', '??'):
        if clean_ocr and 2 <= len(letters) <= 17 and re.fullmatch(r'[A-Z ]+', clean_ocr):
            if re.sub(r'[^A-Z]', '', bubble) != letters:
                return bubble, False, 'BUBBLE-FIRST: OCR text disagrees with bubbles; bubble result retained and mismatch flagged for manual review.'
            return bubble, False, 'BUBBLE-FIRST: OCR broadly agrees with bubbles; bubble result retained.'
        return bubble, False, 'BUBBLE-FIRST: handwriting OCR unavailable/uncertain; bubble result retained. Check ? slots manually.'
    if clean_ocr and 2 <= len(letters) <= 17 and re.fullmatch(r'[A-Z ]+', clean_ocr):
        return bubble or '?', False, 'BUBBLE-FIRST: OCR candidate is shown for comparison only; missing/ambiguous bubble slots need manual review.'
    return bubble or '?', False, 'BUBBLE-FIRST: no confident bubble name; kept ? markers for manual review rather than guessing.'

def process_batch(profile_state, bundle_file, omr_files, fill_sensitivity=0.70, run_text_crosscheck=False, enable_name_ocr=False, use_ai_name_ocr=True, progress=gr.Progress(track_tqdm=True)):
    run_id=f'run_{int(time.time()*1000)}_{os.getpid()}'
    work=RUN_DIR/run_id
    work.mkdir(parents=True,exist_ok=True)
    try:
        if bundle_file:
            profile=load_profile_bundle(str(bundle_file),work/'loaded_profile')
        elif profile_state and isinstance(profile_state,dict) and profile_state.get('circles'):
            profile=profile_state
        else:
            raise ValueError('Pehle template analyze karein ya saved Template Profile ZIP upload karein.')
        if not omr_files:
            raise ValueError('Student OMR files upload nahi ki gayi hain.')
        template_bgr=safe_read_image(profile['template_path'])
        # Keep a run-local reference so manual image correction remains available for this batch.
        template_copy=work/('template_reference'+Path(profile['template_path']).suffix.lower())
        if Path(profile['template_path']).resolve() != template_copy.resolve():
            shutil.copy2(profile['template_path'],template_copy)
        template_gray=cv2.cvtColor(template_bgr,cv2.COLOR_BGR2GRAY)
        # Upgrade the profile BEFORE persisting it; older code saved generic groups for manual
        # image redraw, which could make Step 3 use stale geometry even when Step 2 was fixed.
        profile=upgrade_profile_mapping(profile, template_gray)
        profile['template_path']=str(template_copy)
        (work/'profile_used.json').write_text(json.dumps(profile,ensure_ascii=False,indent=2),encoding='utf-8')
        th,tw=template_gray.shape[:2]
        circles=profile.get('circles',[])
        groups=profile.get('groups',[])
        if len(circles)<10:
            raise ValueError('Profile mein sufficient bubble coordinates nahi mile. Tab 1 mein blank template dobara analyze karke naya Template Profile ZIP save karein.')
        mapped_fields=[student_detail_column(g) for g in groups]
        mapped_set={x for x in mapped_fields if x}
        missing_core=[x for x in STUDENT_DETAIL_COLUMNS if x not in mapped_set]
        if missing_core:
            warning = profile.get('_mapping_warning') or 'PWNSAT form signature was not confirmed; automatic canonical mapping was not applied.'
            raise ValueError('Student-detail mapping incomplete; missing: ' + ', '.join(missing_core) +
                '. No misleading Excel was generated. Details: ' + str(warning) +
                '. Verify that the ZIP contains the blank PWNSAT template. In Tab 1, upload that blank template and save the fresh profile if it is a different form/design.')
        allpages=[]
        orig_files=[]
        for ix,f in enumerate(omr_files,1):
            p=str(f)
            try:
                if Path(p).exists(): orig_files.append((p,Path(p).name))
                allpages.extend(paths_to_pages(p,work,f'input{ix}'))
            except Exception as e:
                allpages.append((None,Path(p).name,0))
        trocr_available=False; trocr_status='TrOCR not requested; Tesseract name OCR fallback is used.'
        if enable_name_ocr and use_ai_name_ocr:
            progress(0.01, desc='Loading optional handwriting OCR model (first run may download it)...')
            trocr_available, trocr_status=initialize_trocr_model(progress=progress)
        summaries=[]; bubble_details=[]; responses=[]; ocr_records=[]; field_comparisons=[]; student_rows=[]; review_rows=[]; review_paths=[]; errors=[]; page_records=[]
        for pi,item in enumerate(allpages,1):
            page_path,source_name,page_no=item
            progress((pi, max(1, len(allpages))), desc=f'Processing {pi}/{len(allpages)}: {source_name}')
            if not page_path:
                errors.append({'file':source_name,'page':page_no,'error':'File conversion failed'})
                review_rows.append({'Source File':source_name,'Page':page_no,'Review Status':'PROCESSING_FAILED: file conversion failed', **{col:'' for col in STUDENT_DETAIL_COLUMNS}})
                continue
            page_student_details = {col: '' for col in STUDENT_DETAIL_COLUMNS}
            try:
                bgr=safe_read_image(page_path)
                original_h,original_w=bgr.shape[:2]
                gray=cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY)
                aligned,align_status,align_conf,aspect_delta,source_to_template=align_to_template(template_gray,gray)
                page_records.append({'Source File':source_name,'Page':page_no,'page_path':str(page_path),
                                     'original_width':original_w,'original_height':original_h,
                                     'source_to_template':source_to_template,'alignment_status':align_status,
                                     'alignment_confidence':round(float(align_conf),3)})
                # Build per-bubble scores using reference coordinates and template-relative baselines.
                bgr_aligned=cv2.cvtColor(aligned,cv2.COLOR_GRAY2BGR)
                page_bubbles=[]
                target_scores=[]
                # The control is now SENSITIVITY, not a required darkness threshold:
                # higher values lower the decision threshold so faint marks are not missed.
                sensitivity=float(np.clip(float(fill_sensitivity),0.0,1.0))
                decision_threshold=0.55 - 0.36*sensitivity
                decision_margin=max(0.045, 0.080 - 0.025*sensitivity)
                filled_cutoff=decision_threshold + decision_margin
                empty_cutoff=max(0.04, decision_threshold - decision_margin)
                for c in circles:
                    x=float(c['x_norm'])*tw; y=float(c['y_norm'])*th; r=float(c['r_norm'])*min(th,tw)
                    target_dark,target_mean=bubble_metrics(aligned,x,y,r)
                    ref_dark,ref_mean=bubble_metrics(template_gray,x,y,r)
                    # Compare the student bubble with the same printed bubble on the blank template.
                    # This suppresses the printed digit/letter and outline as a false fill signal.
                    delta=target_dark-ref_dark
                    score=max(0.0, delta + 0.08)
                    if score >= filled_cutoff:
                        state='FILLED'
                    elif score <= empty_cutoff:
                        state='EMPTY'
                    else:
                        state='AMBIGUOUS'
                    row=dict(c)
                    row.update({'source_file':source_name,'page':page_no,'dark_fraction':round(target_dark,4),
                                'template_dark_fraction':round(ref_dark,4),'relative_darkness_delta':round(delta,4),
                                'fill_score':round(score,4),'state':state})
                    page_bubbles.append(row); target_scores.append(score)

                # Contextual faint-mark rescue: compare alternatives within one digit/name
                # slot, one MCQ question, or the Grade row. A clear winner can be promoted;
                # a weaker winner is marked AMBIGUOUS (amber), never silently treated as blank.
                group_meta={str(g.get('group_id')):g for g in groups}
                by_slot={}
                for b in page_bubbles:
                    g=group_meta.get(str(b.get('group_id')), {})
                    typ=str(g.get('suggested_type','')).lower()
                    if 'digit_grid' in typ or 'letter_grid' in typ:
                        slot=b.get('grid_col')
                    elif 'mcq_grid' in typ:
                        slot=b.get('grid_row')
                    elif 'single_row_choices' in typ:
                        slot='choice_row'
                    else:
                        continue
                    by_slot.setdefault((str(b.get('group_id')),slot),[]).append(b)
                for _, alternatives in by_slot.items():
                    if any(b['state']=='FILLED' for b in alternatives):
                        continue
                    ranked=sorted(alternatives,key=lambda b:float(b.get('fill_score',0.0)),reverse=True)
                    if len(ranked)<2: continue
                    best=float(ranked[0].get('fill_score',0.0))
                    second=float(ranked[1].get('fill_score',0.0))
                    contrast=best-second
                    contrast_needed=max(0.045, second*0.25)
                    if best >= 0.15 and contrast >= contrast_needed:
                        if best >= decision_threshold-0.075:
                            ranked[0]['state']='FILLED'
                        else:
                            ranked[0]['state']='AMBIGUOUS'
                # Annotation; red=filled, amber=ambiguous, green=empty.
                state_by_id={b['bubble_id']:b['state'] for b in page_bubbles}
                for b in page_bubbles:
                    x=int(round(b['x_norm']*tw)); y=int(round(b['y_norm']*th)); r=max(2,int(round(b['r_norm']*min(th,tw))))
                    st=b['state']; color=(0,0,255) if st=='FILLED' else ((0,190,255) if st=='AMBIGUOUS' else (0,150,0))
                    cv2.circle(bgr_aligned,(x,y),r,color,1)
                    if st=='FILLED': cv2.circle(bgr_aligned,(x,y),max(2,int(r*0.25)),(0,0,255),-1)
                # Put legend on a top margin overlay.
                cv2.rectangle(bgr_aligned,(8,8),(min(tw-8,450),42),(255,255,255),-1)
                cv2.putText(bgr_aligned,'RED=FILLED  AMBER=UNCERTAIN  GREEN=EMPTY',(14,30),cv2.FONT_HERSHEY_SIMPLEX,0.52,(0,0,0),1,cv2.LINE_AA)
                annpath=work/f'review_{pi:04d}_{slugify(Path(source_name).stem)}_p{page_no}.jpg'
                save_image(annpath,bgr_aligned)
                # Keep the preview light enough for mobile even when the batch has 500 pages.
                thumb=bgr_aligned.copy()
                scale_thumb=min(1.0, 480.0/max(thumb.shape[:2]))
                if scale_thumb < 1.0:
                    thumb=cv2.resize(thumb,(max(1,int(thumb.shape[1]*scale_thumb)),max(1,int(thumb.shape[0]*scale_thumb))),interpolation=cv2.INTER_AREA)
                thumbpath=work/f'preview_{pi:04d}_{slugify(Path(source_name).stem)}_p{page_no}.jpg'
                save_image(thumbpath,thumb); review_paths.append(str(thumbpath))
                raw_ocr='Full-page OCR skipped for speed; see Personal Detail Review for optional field-level OCR.'
                ocr_records.append({'file':source_name,'page':page_no,'raw_ocr_text':raw_ocr,'alignment':align_status,'alignment_confidence':round(align_conf,3)})
                # Store full bubble diagnostics (including ungrouped detections)
                for b in page_bubbles:
                    bubble_details.append({k:b.get(k) for k in ['source_file','page','bubble_id','group_id','grid_row','grid_col','x_px','y_px','r_px','dark_fraction','template_dark_fraction','relative_darkness_delta','fill_score','state']})
                # Decode each grid group and compare personal numeric text to digit bubbles where possible.
                proposed_page = cv2.cvtColor(aligned, cv2.COLOR_GRAY2BGR)
                page_has_proposal = False
                page_proposal_notes = []
                page_student_details = {col: '' for col in STUDENT_DETAIL_COLUMNS}
                page_name_ocr_text = ''
                name_bubble_comparison_value = ''
                name_ocr_conf = -1.0
                name_ocr_note = 'OCR disabled for speed'
                page_name_comparison = 'NAME TEXT/BUBBLES NOT COMPARED'
                name_ocr_proposal_applied = False
                page_name_value_set = False
                for g in groups:
                    field_ocr_conf = -1.0
                    ocr_note = 'OCR disabled for speed'
                    grid_results=classify_group(g,page_bubbles,float(decision_threshold))
                    responses.extend([{'file':source_name,'page':page_no,**r} for r in grid_results])
                    detail_col = student_detail_column(g)
                    if detail_col:
                        # Keep undecodable positions visibly flagged instead of guessing a value.
                        decoded_parts = []
                        for rr in sorted(grid_results, key=lambda x: int(x.get('question_or_position', 0)) if str(x.get('question_or_position', '')).isdigit() else 0):
                            if rr.get('status') == 'ONE_MARK' and rr.get('value') not in ('', None):
                                decoded_parts.append(str(rr.get('value')))
                            elif detail_col == 'Student Name' and rr.get('status') == 'BLANK':
                                decoded_parts.append(' ')
                            else:
                                decoded_parts.append('?')
                        decoded_value = ''.join(decoded_parts)
                        if detail_col == 'Student Name':
                            decoded_value = re.sub(r'\s+', ' ', decoded_value).strip()
                            active_name_positions, active_name_counts = name_handwriting_slot_positions(aligned, template_gray, g)
                            bubble_name_candidate, bubble_name_quality, bubble_active_chars, bubble_known_count = active_slot_bubble_name(
                                grid_results, active_name_positions, cols=int(g.get('cols',17)))
                            name_bubble_comparison_value = bubble_name_candidate
                            if enable_name_ocr:
                                page_name_ocr_text, name_ocr_conf, name_ocr_note = bounded_handwriting_ocr(
                                    aligned, g, field_kind='letters', timeout_seconds=1.5,
                                    reference_gray=template_gray, use_trocr=bool(use_ai_name_ocr and trocr_available),
                                    bubble_grid_results=grid_results)
                                candidate_letters = re.sub(r'[^A-Z]', '', str(page_name_ocr_text or '').upper())
                                candidate_name = _clean_name_ocr_text(page_name_ocr_text)
                                selected_name, selected_from_ocr, name_selection_note = choose_name_value(
                                    bubble_name_candidate, bubble_name_quality, bubble_active_chars,
                                    candidate_name, name_ocr_note, len(active_name_positions),
                                    use_trocr=bool(use_ai_name_ocr and trocr_available))
                                if selected_from_ocr and len(candidate_letters)==len(active_name_positions):
                                    selected_name = format_name_candidate_by_slots(candidate_name, active_name_positions, cols=int(g.get('cols',17)))
                                elif not selected_name:
                                    selected_name = bubble_name_candidate or decoded_value
                                decoded_value = selected_name
                                page_student_details['Student Name'] = selected_name
                                page_name_value_set = True
                                name_ocr_proposal_applied = bool(selected_from_ocr)
                                page_name_comparison = (
                                    f'Bubble-derived name: {bubble_name_candidate or "[none]"} (known active-slot bubbles {bubble_known_count}/{len(active_name_positions)}). '
                                    f'Handwriting OCR candidate: {candidate_name or "[none]"}. Selected editable field: {selected_name or "[none]"}. '
                                    f'{name_selection_note} OCR note: {name_ocr_note}. NOT auto-verified; inspect and correct in Step 3.')
                                page_name_comparison += ' OCR is comparison-only; no OMR pixels are redrawn.'
                            else:
                                page_name_comparison = 'BUBBLE-ONLY; handwriting OCR disabled; inspect flagged slots and correct manually.'
                                page_name_ocr_text = ''
                                decoded_value = bubble_name_candidate or decoded_value
                        if detail_col == 'Class / Grade' and decoded_value and decoded_value != '?':
                            grade_map = {
                                '5th':'Class V', '6th':'Class VI', '7th':'Class VII',
                                '8th':'Class VIII', '9th':'Class IX', '10th':'Class X',
                                'Class V':'Class V', 'Class VI':'Class VI', 'Class VII':'Class VII',
                                'Class VIII':'Class VIII', 'Class IX':'Class IX', 'Class X':'Class X'
                            }
                            decoded_value = grade_map.get(decoded_value, decoded_value)
                        if not (detail_col == 'Student Name' and page_name_value_set):
                            page_student_details[detail_col] = decoded_value
                    label=str(g.get('label') or g.get('label_hint') or g.get('group_id','')).lower()
                    typ=str(g.get('suggested_type','')).lower()
                    is_digit_grid=('digit_grid' in typ)
                    is_relevant_field=any(k in label for k in ['mobile','phone','parent','guardian','exam code','centre code','center code','registration','roll no','roll number','student id','student code','code','dob','date of birth'])
                    is_name_grid=('letter_grid' in typ and 'name' in label)
                    if (is_digit_grid and is_relevant_field) or is_name_grid:
                        bubble_value=''.join(str(r.get('value','')) if r.get('status')=='ONE_MARK' else '?' for r in grid_results)
                        if is_name_grid:
                            bubble_value = name_bubble_comparison_value or bubble_value

                            field_ocr=page_name_ocr_text or '[No handwritten name characters confidently recognized]'
                            numeric_candidate=''
                            comparison=page_name_comparison
                        elif not run_text_crosscheck:
                            field_ocr='[Field OCR skipped for speed. Check handwritten text manually, or rerun a small validation batch with OCR enabled.]'
                            numeric_candidate=''
                            comparison='REVIEW — OCR SKIPPED; MANUAL TEXT/BUBBLE COMPARISON REQUIRED'
                        else:
                            field_ocr, field_ocr_conf, ocr_note = bounded_handwriting_ocr(
                                aligned, g, field_kind='digits' if is_digit_grid else 'text', timeout_seconds=1.5, reference_gray=template_gray)
                            numeric_candidate=extract_numeric_candidate(field_ocr,int(g.get('cols',0))) if is_digit_grid else ''
                            if is_digit_grid:
                                if numeric_candidate and len(bubble_value)==int(g.get('cols',0)) and '?' not in bubble_value:
                                    comparison='MATCH' if numeric_candidate==bubble_value else 'MISMATCH — REVIEW REQUIRED'
                                elif numeric_candidate:
                                    comparison='REVIEW — BUBBLE GRID INCOMPLETE/AMBIGUOUS'
                                else:
                                    comparison='REVIEW — TEXT OCR NOT CONFIDENT ENOUGH'
                            else:
                                comparison='REVIEW — NAME TEXT/BUBBLES REQUIRE HUMAN CONFIRMATION'
                        field_comparisons.append({'file':source_name,'page':page_no,'group_id':g['group_id'],
                            'field_label':g.get('label',g['group_id']),'detected_type':g.get('suggested_type'),
                            'OCR_crop_text':field_ocr,'OCR_confidence':(name_ocr_conf if is_name_grid and enable_name_ocr else (field_ocr_conf if run_text_crosscheck else -1)),
                            'OCR_note':(name_ocr_note if is_name_grid and enable_name_ocr else (ocr_note if run_text_crosscheck else 'OCR disabled for speed')),
                            'bubble_decoded_value':bubble_value,
                            'text_numeric_candidate':numeric_candidate,'comparison_status':comparison,
                            'note':'OCR candidate is comparison-only; confirm manually before correcting the editable result. Original OMR is never changed.'})
                        # OCR mismatches are recorded for manual review only. This version never redraws or exports altered OMR images.
                        if is_digit_grid and numeric_candidate and numeric_candidate != bubble_value and field_comparisons:
                            field_comparisons[-1]['comparison_status']='MISMATCH — REVIEW REQUIRED; NO OMR IMAGE CHANGED'
                # No proposed/rewritten OMR copies are generated; OCR differences stay in the review table.
                filled=sum(b['state']=='FILLED' for b in page_bubbles)
                ambiguous=sum(b['state']=='AMBIGUOUS' for b in page_bubbles)
                group_unmarked_template=profile.get('likely_marked_bubbles_in_template',0)
                issues=[]
                if align_conf<0.5: issues.append('ALIGNMENT_REVIEW')
                if ambiguous: issues.append('AMBIGUOUS_BUBBLES')
                if group_unmarked_template > max(5,int(len(circles)*0.03)): issues.append('TEMPLATE_MAY_CONTAIN_MARKS')
                # Keep only the requested student detail fields in the primary export.
                student_rows.append({col: page_student_details.get(col, '') for col in STUDENT_DETAIL_COLUMNS})
                detail_values=[str(page_student_details.get(col,'') or '') for col in STUDENT_DETAIL_COLUMNS]
                page_comparisons=[r for r in field_comparisons if r.get('file')==source_name and r.get('page')==page_no]
                text_conflict = (enable_name_ocr or run_text_crosscheck) and any(
                    ('MISMATCH' in str(r.get('comparison_status','')) or
                     ('REVIEW' in str(r.get('comparison_status','')) and 'OCR SKIPPED' not in str(r.get('comparison_status',''))))
                    for r in page_comparisons)
                review_status='NEEDS_REVIEW — NAME OCR PROPOSAL' if name_ocr_proposal_applied else ('NEEDS_REVIEW' if (issues or text_conflict or any('?' in v for v in detail_values) or any(not v.strip() for v in detail_values)) else 'PROCESSED — CHECK PREVIEW')
                review_rows.append({'Source File':source_name,'Page':page_no,'Review Status':review_status, **{col: page_student_details.get(col,'') for col in STUDENT_DETAIL_COLUMNS}})
                summaries.append({'file':source_name,'page':page_no,'profile':profile.get('profile_name','template'),
                                  'original_width_px':original_w,'original_height_px':original_h,
                                  'template_width_px':tw,'template_height_px':th,'alignment_status':align_status,
                                  'alignment_confidence':round(align_conf,3),'aspect_ratio_difference_pct':round(aspect_delta*100,2),
                                  'template_bubble_positions':len(circles),'filled':filled,'empty':len(page_bubbles)-filled-ambiguous,
                                  'ambiguous':ambiguous,'proposed_corrections':len(page_proposal_notes),'raw_ocr_characters':len(raw_ocr),'status':';'.join(issues) if issues else 'PROCESSED — REVIEW OUTPUT'})
            except Exception as e:
                err_msg=f'{type(e).__name__}: {str(e)}'
                errors.append({'file':source_name,'page':page_no,'error':err_msg,'traceback':traceback.format_exc()[-2500:]})
                # Keep any fields already decoded before a later optional operation failed.
                partial_details = {col: str(page_student_details.get(col, '') or '') for col in STUDENT_DETAIL_COLUMNS}
                review_rows.append({'Source File':source_name,'Page':page_no,'Review Status':f'PROCESSING_FAILED: {err_msg[:220]}', **partial_details})
        progress(1.0, desc='Writing Excel and review outputs...')
        (work/'page_manifest.json').write_text(json.dumps(page_records,ensure_ascii=False,indent=2),encoding='utf-8')
        sdf=pd.DataFrame(summaries)
        bdf=pd.DataFrame(bubble_details)
        rdf=pd.DataFrame(responses)
        odf=pd.DataFrame(ocr_records)
        cdf=pd.DataFrame(field_comparisons)
        edf=pd.DataFrame(errors)
        review_columns=['Source File','Page','Review Status'] + STUDENT_DETAIL_COLUMNS
        review_df=pd.DataFrame(review_rows, columns=review_columns)
        student_df=review_df[STUDENT_DETAIL_COLUMNS].copy() if len(review_df) else pd.DataFrame(columns=STUDENT_DETAIL_COLUMNS)
        # Values such as phone numbers and Exam Codes remain strings (including leading zeros).
        for _c in STUDENT_DETAIL_COLUMNS:
            student_df[_c]=student_df[_c].fillna('').astype(str)
        outdir=work/'outputs'; outdir.mkdir(exist_ok=True)
        xlsx=outdir/'PWNSAT_Student_Details_Landscape.xlsx'
        # A single clean worksheet only: no debug columns or bubble-level detail sheets.
        with pd.ExcelWriter(xlsx,engine='openpyxl') as writer:
            student_df.to_excel(writer,sheet_name='Student Details',index=False)
            ws=writer.sheets['Student Details']
            ws.sheet_view.showGridLines=False
            ws.freeze_panes='A2'
            ws.auto_filter.ref=ws.dimensions if len(student_df) else 'A1:F1'
            ws.sheet_properties.pageSetUpPr.fitToPage=True
            ws.page_setup.orientation='landscape'
            ws.page_setup.paperSize=ws.PAPERSIZE_A4
            ws.page_setup.fitToWidth=1
            ws.page_setup.fitToHeight=0
            ws.page_margins.left=0.25; ws.page_margins.right=0.25
            ws.page_margins.top=0.45; ws.page_margins.bottom=0.45
            ws.print_title_rows='1:1'
            ws.print_area=f'A1:{chr(64+len(STUDENT_DETAIL_COLUMNS))}{max(1,len(student_df)+1)}'
            from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
            header_fill=PatternFill('solid',fgColor='1F4E78')
            header_font=Font(name='Arial',size=10,bold=True,color='FFFFFF')
            body_font=Font(name='Arial',size=10,color='1F2937')
            thin=Side(style='thin',color='D9E2F3')
            for cell in ws[1]:
                cell.fill=header_fill; cell.font=header_font
                cell.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
                cell.border=Border(bottom=Side(style='medium',color='1F4E78'))
            ws.row_dimensions[1].height=32
            widths=[26,17,28,14,16,18]
            for idx,width in enumerate(widths,1):
                ws.column_dimensions[chr(64+idx)].width=width
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    cell.font=body_font
                    cell.alignment=Alignment(vertical='center',wrap_text=False)
                    cell.border=Border(bottom=thin)
                    # Keep digit strings such as mobile numbers, exam codes and DOB from being retyped as numbers.
                    if cell.column in (2,3,4,5,6):
                        cell.number_format='@'
                ws.row_dimensions[row[0].row].height=20
        student_csv=outdir/'PWNSAT_Student_Details.csv'
        student_df.to_csv(student_csv,index=False,encoding='utf-8-sig')
        review_report=outdir/'PWNSAT_Review_Report.csv'
        review_df.to_csv(review_report,index=False,encoding='utf-8-sig')
        comparisons_csv=outdir/'PWNSAT_Field_Text_Bubble_Comparison.csv'
        cdf.to_csv(comparisons_csv,index=False,encoding='utf-8-sig')
        errors_csv=outdir/'PWNSAT_Processing_Errors.csv'
        pd.DataFrame(errors).to_csv(errors_csv,index=False,encoding='utf-8-sig')
        zipp=outdir/'PWNSAT_Student_Details_Only.zip'
        with zipfile.ZipFile(zipp,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(xlsx,xlsx.name)
            z.write(student_csv,student_csv.name)
            z.write(review_report,review_report.name)
            z.write(comparisons_csv,comparisons_csv.name)
            z.write(errors_csv,errors_csv.name)
        failed_count=sum(str(r.get('Review Status','')).startswith('PROCESSING_FAILED') for r in review_rows)
        needs_review=sum('NEEDS_REVIEW' in str(r.get('Review Status','')) or str(r.get('Review Status','')).startswith('PROCESSING_FAILED') for r in review_rows)
        map_mode=str(profile.get('mapping_mode','GENERIC_TEMPLATE_MAPPING'))
        map_note=f' Mapping: {map_mode}.'
        if profile.get('_mapping_warning'):
            map_note += ' Mapping note: ' + str(profile['_mapping_warning'])
        empty_fields=sum(1 for row in student_rows for col in STUDENT_DETAIL_COLUMNS if not str(row.get(col,'') or '').strip())
        # Validate output artifacts before showing them, then stage them beneath an allowed Gradio download path.
        xlsx_download = stage_download_file(xlsx, 'xlsx')
        zip_download = stage_download_file(zipp, 'zip')
        msg=f'Finished batch: {len(allpages)-failed_count}/{len(allpages)} page(s) processed; {failed_count} failed; {needs_review} need review. Empty field cells: {empty_fields}/{max(1,len(student_rows)*len(STUDENT_DETAIL_COLUMNS))}. Main Excel: 6 student fields, A4 landscape. Bubble sensitivity={sensitivity:.2f}. Name OCR={"ON" if enable_name_ocr else "OFF"}; Handwriting model status: {trocr_status}. OCR is comparison-only; no OMR copies are redrawn. Original OMR files are unchanged; editable review table lets you correct exported values. DPI metadata will only be preserved when present in the input.' + map_note
        gallery_items=[(p,'Annotated scan preview: '+Path(p).name) for p in review_paths]
        original_gallery_items=load_original_omr_gallery(str(work))
        diag_cols=['Source File','Page','Outcome','Alignment','Alignment Confidence','Filled bubbles','Ambiguous bubbles','Error']
        diagnostics=[]
        for rr in review_rows:
            summary_row=next((x for x in summaries if x.get('file')==rr.get('Source File') and str(x.get('page'))==str(rr.get('Page'))),{})
            err=next((x.get('error','') for x in errors if x.get('file')==rr.get('Source File') and str(x.get('page'))==str(rr.get('Page')) and x.get('error')), '')
            diagnostics.append({'Source File':rr.get('Source File'),'Page':rr.get('Page'),'Outcome':rr.get('Review Status'),
                'Alignment':summary_row.get('alignment_status',''),'Alignment Confidence':summary_row.get('alignment_confidence',''),
                'Filled bubbles':summary_row.get('filled',''),'Ambiguous bubbles':summary_row.get('ambiguous',''),'Error':err})
        diagnostics_df=pd.DataFrame(diagnostics,columns=diag_cols)
        if not any(any(str(row.get(c,'') or '').strip() for c in STUDENT_DETAIL_COLUMNS) for row in review_rows):
            msg += ' WARNING: all six student fields are blank for every page. Do not use this Excel; inspect Diagnostics and re-analyze the blank template/profile.'
        return student_df, gallery_items, xlsx_download, zip_download, msg, review_df, str(work), diagnostics_df, cdf, original_gallery_items
    except Exception as e:
        empty_review=pd.DataFrame(columns=['Source File','Page','Review Status']+STUDENT_DETAIL_COLUMNS)
        diag_cols=['Source File','Page','Outcome','Alignment','Alignment Confidence','Filled bubbles','Ambiguous bubbles','Error']
        diag=pd.DataFrame([{'Source File':'','Page':'','Outcome':'BATCH FAILED','Alignment':'','Alignment Confidence':'','Filled bubbles':'','Ambiguous bubbles':'','Error':f'{type(e).__name__}: {e}'}],columns=diag_cols)
        return pd.DataFrame(columns=STUDENT_DETAIL_COLUMNS), [], None, None, f'PROCESSING FAILED: {type(e).__name__}: {e}\n{traceback.format_exc()[-1800:]}', empty_review, str(work), diag, pd.DataFrame(), []

def create_verified_corrected_image_copies(review_df, run_directory):
    """Create source-format corrected copies with local source-pixel edits; never perspective-warp the original scan."""
    run_dir=Path(str(run_directory)); manifest_path=run_dir/'page_manifest.json'; profile_path=run_dir/'profile_used.json'
    if not manifest_path.exists() or not profile_path.exists():
        raise ValueError('Image-correction manifest/profile missing; run a fresh scan before exporting images.')
    manifest=json.loads(manifest_path.read_text(encoding='utf-8')); profile=json.loads(profile_path.read_text(encoding='utf-8'))
    template_bgr=safe_read_image(profile['template_path']); template_gray=cv2.cvtColor(template_bgr,cv2.COLOR_BGR2GRAY)
    th,tw=template_gray.shape[:2]; groups={str(g.get('group_id')):g for g in profile.get('groups',[])}
    records={(str(m.get('Source File')),str(m.get('Page'))):m for m in manifest}
    output_dir=run_dir/'manual_final_outputs'/'corrected_omr_images'; output_dir.mkdir(parents=True,exist_ok=True)
    manual_state=_manual_json_read(run_dir/'manual_bubble_edits_nsat.json',{'pages':{}})
    created=[]; skipped=[]
    for _,rr in review_df.iterrows():
        state=str(rr.get('Review Status','')).strip().upper()
        is_verified = state in ('VERIFIED','CORRECTED & VERIFIED','CORRECTED AND VERIFIED','VERIFIED - CORRECTED') or state.startswith('VERIFIED:')
        key=(str(rr.get('Source File','')),str(rr.get('Page',''))); record=records.get(key)
        if not record:
            skipped.append(f'{key[0]} page {key[1]}: source page manifest not found'); continue
        page_idx=next((i for i,m in enumerate(manifest) if str(m.get('Source File'))==key[0] and str(m.get('Page'))==key[1]),None)
        page_edits=(manual_state.get('pages',{}).get(str(page_idx),{}).get('edits',{}) if page_idx is not None else {})
        has_manual_edits=bool(page_edits)
        # Export a separate copy if the user explicitly saved bubble edits, even before
        # marking the entire student row VERIFIED. For unverified rows, only user-saved
        # bubble edits are applied; auto-read fields are not redrawn onto the original.
        if not is_verified and not has_manual_edits:
            continue
        src_path=Path(record['page_path'])
        if not src_path.exists(): skipped.append(f'{key[0]} page {key[1]}: original source missing'); continue
        try:
            original=safe_read_image(src_path); ph,pw=original.shape[:2]
            H=np.asarray(record.get('source_to_template'),dtype=np.float64)
            if H.shape!=(3,3) or not np.isfinite(H).all(): raise ValueError('invalid source-to-template alignment transform')
            H_inv,clean_ref=_make_reference_in_source(template_gray,H,original.shape)
            corrected=original.copy(); row_values={col:str(rr.get(col,'') or '').strip() for col in STUDENT_DETAIL_COLUMNS}; corrections=[]
            def redraw(gid, symbols, kind):
                ok,msg=_apply_nsat_group_value_to_source(corrected,clean_ref,H_inv,profile,gid,symbols,kind,tw,th)
                corrections.append(msg if ok else 'SKIP '+msg)
            if is_verified:
                # Full record-based redraw is permitted only after the user marks the row VERIFIED.
                name=row_values.get('Student Name','').upper()
                if name and '?' not in name and len(name)<=17 and re.fullmatch(r'[A-Z ]+',name):
                    slots=list(re.sub(r' +',' ',name)); slots=slots[:17]+[' ']*max(0,17-len(slots)); redraw('P01',slots,'letters')
                else: corrections.append('P01 skipped: missing/ambiguous name or >17 letters')
                for gid,col,length in [('P02','Mobile No.',10),('P03','Exam Code',5),('P04','Parent/Guardian Mobile No.',10),('P05','DOB DDMMYY',6)]:
                    val=row_values.get(col,'')
                    if val and '?' not in val and val.isdigit() and len(val)==length: redraw(gid,list(val),'digits')
                    else: corrections.append(f'{gid} skipped: {col} incomplete/ambiguous or wrong length')
                grade=row_values.get('Class / Grade','').upper(); p06=groups.get('P06',{}); labels=[str(x).upper() for x in p06.get('choice_labels',[])]
                if grade and grade in labels and len(labels)==10:
                    slots=[' ']*10; slots[labels.index(grade)]=grade; redraw('P06',slots,'choices')
                else: corrections.append('P06 skipped: grade not recognized')
            else:
                corrections.append('UNVERIFIED page: only explicit saved manual bubble edits applied; auto-read fields were not redrawn')
            if page_edits:
                _apply_nsat_manual_edits_to_source(corrected,clean_ref,H_inv,profile,page_edits,tw,th)
                corrections.append(f'Manual bubble overrides applied: {len(page_edits)}')
            output_base=output_dir/f'CORRECTED_{slugify(Path(str(record.get("Source File",src_path.name))).stem)}_page{record.get("Page",1)}'
            out_path,meta=_save_corrected_copy_like_source(src_path,output_base,corrected)
            created.append({'path':out_path,'filename':Path(out_path).name,'Source File':key[0],'Page':key[1],**meta,'corrections':'; '.join(corrections)})
        except Exception as exc:
            skipped.append(f'{key[0]} page {key[1]}: {type(exc).__name__}: {exc}')
    return created,skipped

def _normalize_page_key(value):
    """Normalize Gradio's int/float/string page values to a stable key."""
    try:
        f=float(str(value).strip())
        if f.is_integer(): return str(int(f))
    except Exception:
        pass
    return str(value).strip()


def mapped_circle_bounds(cells, width, height, pad_factor=1.2):
    """Return a clipped rectangle around all bubble centres with overfill margin."""
    if not cells:
        raise ValueError('No mapped bubble cells were provided.')
    scale=float(min(height,width)); xs=[]; ys=[]
    for c in cells:
        x=float(c['x_norm'])*width; y=float(c['y_norm'])*height
        r=max(3.0,float(c['r_norm'])*scale)
        xs.extend([x-pad_factor*r,x+pad_factor*r]); ys.extend([y-pad_factor*r,y+pad_factor*r])
    x0=max(0,int(np.floor(min(xs)))); y0=max(0,int(np.floor(min(ys))))
    x1=min(width,int(np.ceil(max(xs)))+1); y1=min(height,int(np.ceil(max(ys)))+1)
    if x1<=x0 or y1<=y0: raise ValueError('Mapped bubble rectangle has invalid dimensions.')
    return x0,y0,x1,y1


def load_original_omr_gallery(run_directory):
    """Make read-only thumbnails of the original uploaded/extracted pages for Section 1."""
    try:
        run_dir=Path(str(run_directory))
        manifest_path=run_dir/'page_manifest.json'
        if not manifest_path.exists(): return []
        items=json.loads(manifest_path.read_text(encoding='utf-8'))
        thumb_dir=run_dir/'original_omr_read_only_thumbnails'
        thumb_dir.mkdir(parents=True,exist_ok=True)
        gallery=[]
        for i,rec in enumerate(items,1):
            src=Path(str(rec.get('page_path','')))
            if not src.exists(): continue
            try:
                img=safe_read_image(src)
                h,w=img.shape[:2]
                scale=min(1.0,480.0/max(h,w))
                thumb=cv2.resize(img,(max(1,int(w*scale)),max(1,int(h*scale))),interpolation=cv2.INTER_AREA) if scale<1 else img
                out=thumb_dir/f'ORIGINAL_PREVIEW_{i:04d}_{slugify(src.stem)}.jpg'
                ok,buf=cv2.imencode('.jpg',thumb,[cv2.IMWRITE_JPEG_QUALITY,88])
                if not ok: continue
                out.write_bytes(buf.tobytes())
                caption=f'ORIGINAL — {rec.get("Source File",src.name)} p{rec.get("Page",1)} | {w}×{h}px | read-only preview'
                gallery.append((str(out),caption))
            except Exception:
                continue
        return gallery
    except Exception:
        return []



def apply_manual_name_to_review(review_table, row_number, confirmed_name):
    """Mobile-friendly explicit name correction for one 1-based review row.

    The entered name is not marked VERIFIED; the user must compare it against the
    handwritten name on the original OMR. Corrected bubbling will then use this field.
    """
    try:
        df = review_table.copy() if isinstance(review_table, pd.DataFrame) else pd.DataFrame(review_table)
        if df.empty:
            raise ValueError('Pehle Tab 2 mein OMR batch scan karein; review table khali hai.')
        try:
            row_num = int(float(row_number))
        except Exception:
            raise ValueError('Row number 1, 2, 3... mein dein.')
        if row_num < 1 or row_num > len(df):
            raise ValueError(f'Row number 1 se {len(df)} ke beech hona chahiye.')
        name = re.sub(r'\s+', ' ', str(confirmed_name or '').upper()).strip()
        if not name or '?' in name or not re.fullmatch(r'[A-Z ]+', name):
            raise ValueError('Naam CAPITAL A-Z letters aur spaces mein likhein. ? ya digit allowed nahi hai.')
        if len(name) > 17 or not (2 <= len(re.sub(r'[^A-Z]', '', name)) <= 17):
            raise ValueError('Naam 17 name-grid slots ke andar hona chahiye. Spaces bhi slots lete hain.')
        if 'Student Name' not in df.columns:
            df['Student Name'] = ''
        if 'Review Status' not in df.columns:
            df['Review Status'] = 'NEEDS_REVIEW'
        index = df.index[row_num - 1]
        old = str(df.at[index, 'Student Name'] or '').strip()
        df.at[index, 'Student Name'] = name
        if not str(df.at[index, 'Review Status'] or '').upper().startswith('VERIFIED'):
            df.at[index, 'Review Status'] = 'NEEDS_REVIEW — MANUAL NAME SET; CHECK ORIGINAL BEFORE VERIFYING'
        fname = str(df.at[index, 'Source File'] if 'Source File' in df.columns else '')
        page = str(df.at[index, 'Page'] if 'Page' in df.columns else '1')
        message = f'Row {row_num} ({fname}, page {page}): Student Name set to {name}. Previous value: {old or "blank"}. No auto-verification performed. Inspect original handwriting, then create the corrected bubble copy.'
        return df, message
    except Exception as exc:
        return review_table, f'MANUAL NAME CORRECTION FAILED: {type(exc).__name__}: {exc}'

def apply_ocr_suggestions_to_review(review_table, comparison_table):
    """Copy eligible OCR candidates into the editable review table as PROPOSALS only.

    No row is marked VERIFIED. A person must check the original OMR and explicitly verify.
    Existing verified rows and already non-empty fields are left untouched.
    """
    try:
        review_df = review_table.copy() if isinstance(review_table, pd.DataFrame) else pd.DataFrame(review_table)
        comp_df = comparison_table.copy() if isinstance(comparison_table, pd.DataFrame) else pd.DataFrame(comparison_table)
        if review_df.empty:
            raise ValueError('Tab 2 scan pehle run karein; review table khali hai.')
        if comp_df.empty:
            raise ValueError('Field OCR vs Bubble Review table khali hai. Tab 2 mein OCR checkbox ke saath ek chhota batch dobara scan karein.')
        for col in ['Source File','Page','Review Status'] + STUDENT_DETAIL_COLUMNS:
            if col not in review_df.columns: review_df[col] = ''
        updates=[]
        length_by_col={'Mobile No.':10,'Parent/Guardian Mobile No.':10,'Exam Code':5,'DOB DDMMYY':6}
        for idx, row in review_df.iterrows():
            status=str(row.get('Review Status','')).upper()
            if status.startswith('VERIFIED'):
                continue
            fname=str(row.get('Source File',''))
            page=str(row.get('Page',''))
            matches=comp_df[(comp_df.get('file',pd.Series(dtype=str)).astype(str)==fname) & (comp_df.get('page',pd.Series(dtype=str)).astype(str)==page)]
            proposed_fields=[]
            for _, comp in matches.iterrows():
                label=str(comp.get('field_label','')).strip().lower()
                bubble=str(comp.get('bubble_decoded_value','') or '').strip()
                ocr=str(comp.get('OCR_crop_text','') or '').strip()
                candidate=str(comp.get('text_numeric_candidate','') or '').strip()
                if 'student name' in label:
                    # Use the OCR candidate only as an editable suggestion when bubble result is incomplete.
                    cleaned=re.sub(r'[^A-Z ]','',ocr.upper())
                    cleaned=re.sub(r'\s+',' ',cleaned).strip()
                    current_name=str(row.get('Student Name','') or '').strip()
                    candidate_letters=re.sub(r'[^A-Z]','',cleaned)
                    # The button is an explicit user action: allow the handwriting candidate to
                    # replace even a non-empty bubble-derived name, but never mark the row VERIFIED.
                    if cleaned and 3 <= len(candidate_letters) <= 17 and cleaned != current_name:
                        review_df.at[idx,'Student Name']=cleaned
                        proposed_fields.append('Student Name')
                else:
                    target=None
                    if 'parent' in label or 'guardian' in label:
                        target='Parent/Guardian Mobile No.'
                    elif 'mobile' in label or 'phone' in label:
                        target='Mobile No.'
                    elif 'exam code' in label or 'centre code' in label or 'center code' in label:
                        target='Exam Code'
                    elif 'dob' in label or 'date of birth' in label:
                        target='DOB DDMMYY'
                    else:
                        for column in length_by_col:
                            if column.lower() == label:
                                target=column; break
                    if target and candidate.isdigit() and len(candidate)==length_by_col[target]:
                        current=str(row.get(target,'') or '').strip()
                        # This button is an explicit user action: show a valid OCR alternative even if the bubble
                        # value is non-empty. Row remains NEEDS_REVIEW and is never auto-verified.
                        if current != candidate:
                            review_df.at[idx,target]=candidate
                            proposed_fields.append(target)
            if proposed_fields:
                review_df.at[idx,'Review Status']='NEEDS_REVIEW — OCR PROPOSAL APPLIED; VERIFY ORIGINAL BEFORE EXPORT'
                updates.append(f'{fname} p{page}: {", ".join(proposed_fields)}')
        if not updates:
            return review_df, 'No eligible OCR suggestions were applied. OCR may be blank/ambiguous, or fields already have values. Manual correction remains available; no row was auto-verified.'
        return review_df, 'Applied suggestions to editable fields (NOT verified): ' + '; '.join(updates[:10]) + ('; ...' if len(updates)>10 else '') + '. Check each original OMR, correct as needed, then mark only verified rows VERIFIED.'
    except Exception as exc:
        return review_table, f'Could not apply OCR suggestions: {type(exc).__name__}: {exc}'


def export_manual_corrections(review_table, run_directory, generate_corrected_images=False):
    """Rebuild final student outputs from the user's edited review table.

    The metadata columns are kept in the review report; the main Excel continues to
    contain only the six requested student fields. This edits result data, not source pixels.
    """
    try:
        if not run_directory or not Path(str(run_directory)).exists():
            raise ValueError('Pehle OMR batch scan complete karein; run folder nahi mila.')
        if isinstance(review_table, pd.DataFrame):
            review_df=review_table.copy()
        else:
            review_df=pd.DataFrame(review_table)
        if review_df.empty:
            raise ValueError('Manual review table khali hai.')
        required=['Source File','Page','Review Status'] + STUDENT_DETAIL_COLUMNS
        for col in required:
            if col not in review_df.columns:
                if col in STUDENT_DETAIL_COLUMNS:
                    review_df[col]=''
                else:
                    review_df[col]=''
        for col in required:
            review_df[col]=review_df[col].fillna('').astype(str)
        student_df=review_df[STUDENT_DETAIL_COLUMNS].copy()
        for col in STUDENT_DETAIL_COLUMNS:
            student_df[col]=student_df[col].fillna('').astype(str)
        nonempty=int(student_df.apply(lambda col: col.str.strip().ne('')).sum().sum())
        if nonempty == 0:
            failed=review_df.get('Review Status',pd.Series(dtype=str)).astype(str).tolist()
            raise ValueError('All student-detail cells are blank. Do not export this result. Review Tab 2 diagnostics, confirm that the six groups P01-P06 map correctly, and re-scan after saving a fresh template profile. Row statuses: ' + '; '.join(failed[:4]))
        # Keep leading zeros in every identifier/number field.
        outdir=Path(str(run_directory))/'manual_final_outputs'
        outdir.mkdir(parents=True,exist_ok=True)
        xlsx=outdir/'PWNSAT_Student_Details_Landscape_CORRECTED.xlsx'
        csv=outdir/'PWNSAT_Student_Details_CORRECTED.csv'
        report=outdir/'PWNSAT_Manual_Review_Report.csv'
        with pd.ExcelWriter(xlsx,engine='openpyxl') as writer:
            student_df.to_excel(writer,sheet_name='Student Details',index=False)
            ws=writer.sheets['Student Details']
            ws.sheet_view.showGridLines=False; ws.freeze_panes='A2'
            ws.auto_filter.ref=ws.dimensions if len(student_df) else 'A1:F1'
            ws.sheet_properties.pageSetUpPr.fitToPage=True
            ws.page_setup.orientation='landscape'; ws.page_setup.paperSize=ws.PAPERSIZE_A4
            ws.page_setup.fitToWidth=1; ws.page_setup.fitToHeight=0
            ws.page_margins.left=0.25; ws.page_margins.right=0.25
            ws.page_margins.top=0.45; ws.page_margins.bottom=0.45
            ws.print_title_rows='1:1'
            from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
            header_fill=PatternFill('solid',fgColor='1F4E78')
            header_font=Font(name='Arial',size=10,bold=True,color='FFFFFF')
            body_font=Font(name='Arial',size=10,color='1F2937')
            thin=Side(style='thin',color='D9E2F3')
            for cell in ws[1]:
                cell.fill=header_fill; cell.font=header_font
                cell.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
                cell.border=Border(bottom=Side(style='medium',color='1F4E78'))
            ws.row_dimensions[1].height=32
            widths=[26,17,28,14,16,18]
            for idx,width in enumerate(widths,1): ws.column_dimensions[chr(64+idx)].width=width
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    cell.font=body_font; cell.alignment=Alignment(vertical='center',wrap_text=False)
                    cell.border=Border(bottom=thin)
                    if cell.column in (2,3,4,5,6): cell.number_format='@'
                ws.row_dimensions[row[0].row].height=20
            ws.print_area=f'A1:F{max(1,len(student_df)+1)}'
        student_df.to_csv(csv,index=False,encoding='utf-8-sig')
        review_df[required].to_csv(report,index=False,encoding='utf-8-sig')
        corrected_images=[]; image_skips=[]
        if generate_corrected_images:
            corrected_images,image_skips=create_verified_corrected_image_copies(review_df,run_directory)
            image_manifest=outdir/'Corrected_Image_Metadata_and_Status.csv'
            pd.DataFrame(corrected_images).to_csv(image_manifest,index=False,encoding='utf-8-sig')
            if not image_skips:
                image_skips=[]
            with open(outdir/'Corrected_Image_Skips.txt','w',encoding='utf-8') as fh:
                fh.write('\n'.join(image_skips) if image_skips else 'No image copies skipped.')
        zipp=outdir/'PWNSAT_Corrected_Results_and_Review.zip'
        with zipfile.ZipFile(zipp,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(xlsx,xlsx.name); z.write(csv,csv.name); z.write(report,report.name)
            manual_edit_log=Path(str(run_directory))/'manual_bubble_edits_nsat.json'
            if manual_edit_log.exists():
                z.write(manual_edit_log,'Manual_Bubble_Corrections_NSAT.json')
            if generate_corrected_images:
                if (outdir/'Corrected_Image_Metadata_and_Status.csv').exists(): z.write(outdir/'Corrected_Image_Metadata_and_Status.csv','Corrected_Image_Metadata_and_Status.csv')
                if (outdir/'Corrected_Image_Skips.txt').exists(): z.write(outdir/'Corrected_Image_Skips.txt','Corrected_Image_Skips.txt')
                for item in corrected_images:
                    z.write(item['path'],f'corrected_omr_images/{Path(item["path"]).name}')
        pending=sum(('NEEDS_REVIEW' in x or 'PROCESSING_FAILED' in x) for x in review_df['Review Status'].astype(str))
        xlsx_download=stage_download_file(xlsx,'xlsx')
        zip_download=stage_download_file(zipp,'zip')
        corrected_images_zip=None; image_zip_note='No image-only ZIP created.'
        if generate_corrected_images and corrected_images:
            corrected_images_zip,image_zip_note=_build_images_only_zip([x['path'] for x in corrected_images],outdir/'PWNSAT_All_Corrected_OMR_Images.zip','nsat_corrected')
        elif generate_corrected_images:
            image_zip_note='No corrected image ZIP yet: mark a row VERIFIED or save at least one manual bubble edit for its OMR, then click Save again.'
        image_msg=f' Created {len(corrected_images)} corrected source-format image copy/copies (VERIFIED rows and/or pages with saved manual bubble edits); {len(image_skips)} image item(s) skipped.' if generate_corrected_images else ' Image-copy correction was not requested.'
        return student_df,xlsx_download,zip_download,corrected_images_zip,f'Manual corrections exported. Rows: {len(student_df)}. Review report retained. {pending} row(s) still marked NEEDS_REVIEW/PROCESSING_FAILED.{image_msg} {image_zip_note} Original scans unchanged outside edited bubble interiors; source pixel dimensions and any embedded DPI metadata are checked.'
    except Exception as e:
        return pd.DataFrame(columns=STUDENT_DETAIL_COLUMNS),None,None,None,f'MANUAL EXPORT FAILED: {e}\n{traceback.format_exc()[-1200:]}'


def export_manual_corrections_v18(review_table, run_directory, generate_corrected_images=False):
    """Return export artifacts both to the legacy outputs and to direct-download buttons in Tab 3."""
    student_df, xlsx_path, results_zip_path, images_zip_path, status = export_manual_corrections(
        review_table, run_directory, generate_corrected_images)
    return (student_df, xlsx_path, results_zip_path, images_zip_path, status,
            xlsx_path, results_zip_path)


# v18 download fix: direct DownloadButton outputs + same-tab exports + explicit manual-edit image copies.

# Small internal checks catch obvious helper-function regressions before launch.
# ---------- Additional PW OMR format: bubble-only 180-question sheet ----------
# Geometry was calibrated from the supplied blank U_NEET_N_180Q_V2_NG_26.pdf rendered
# by the same PyMuPDF 2.5x renderer used by this app (1489 x 2105 px).
# Coordinates are reference-template pixel centres. Student pages are aligned to this
# reference before scoring. Text handwriting (Name, Centre, Batch, Room, Date) is NEVER OCR'd.
BUBBLE180_REFERENCE_SIZE = (1489, 2105)  # width, height
BUBBLE180_REG_X = [149, 174, 200, 225, 250, 276, 301, 326]
BUBBLE180_REG_Y = [230, 256, 282, 307, 334, 359, 384, 411, 436, 463]  # digits 0-9 top-to-bottom
BUBBLE180_PHASE_X = [149, 174, 200, 225, 250, 276]
BUBBLE180_PHASE_Y = 566
BUBBLE180_SET_X = [326, 351, 377]
BUBBLE180_SET_Y = 566
BUBBLE180_CLASS_X = [201, 277, 378]  # 11th, 12th, Dropper
BUBBLE180_CLASS_Y = 695
BUBBLE180_Q_XS = [
    [603, 629, 654, 679],       # Questions 1-45
    [805, 830, 855, 881],       # Questions 46-90
    [1007, 1031, 1057, 1082],   # Questions 91-135
    [1207, 1234, 1259, 1284],   # Questions 136-180
]
BUBBLE180_Q_Y = [int(round(513 + i * (1649 - 513) / 44.0)) for i in range(45)]
BUBBLE180_CHOICE_LABELS = ['A', 'B', 'C', 'D']
BUBBLE180_CLASS_LABELS = ['11th', '12th', 'Dropper']


def get_bubble_only_reference():
    """Return built-in blank template image and grey reference, generating PNG only once."""
    if not BUBBLE_ONLY_TEMPLATE_PNG.exists():
        import fitz
        with fitz.open(str(BUBBLE_ONLY_TEMPLATE_PDF)) as doc:
            if len(doc) < 1:
                raise ValueError('Built-in 180Q reference PDF is empty.')
            pix = doc[0].get_pixmap(matrix=fitz.Matrix(2.5, 2.5), alpha=False)
            pix.save(str(BUBBLE_ONLY_TEMPLATE_PNG))
    bgr = safe_read_image(BUBBLE_ONLY_TEMPLATE_PNG)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    if gray.shape[1] != BUBBLE180_REFERENCE_SIZE[0] or gray.shape[0] != BUBBLE180_REFERENCE_SIZE[1]:
        raise ValueError(f'Built-in 180Q template dimensions changed unexpectedly: {gray.shape[1]}x{gray.shape[0]}')
    return bgr, gray


def _bubble180_geometry(template_gray):
    """Create explicit bubble geometry for the new form; all positions are scaled to template."""
    h, w = template_gray.shape[:2]
    sx = w / float(BUBBLE180_REFERENCE_SIZE[0])
    sy = h / float(BUBBLE180_REFERENCE_SIZE[1])
    sr = (sx + sy) / 2.0
    out = []
    def add(field, question, option, x, y, radius=9.0, digit=None):
        out.append({'field': field, 'question': question, 'option': option,
                    'x': float(x*sx), 'y': float(y*sy), 'r': float(radius*sr), 'digit': digit})
    for col, x in enumerate(BUBBLE180_REG_X, start=1):
        for digit, y in enumerate(BUBBLE180_REG_Y):
            add('Registration No.', col, str(digit), x, y, digit=digit)
    for idx, x in enumerate(BUBBLE180_PHASE_X, start=1):
        add('Phase', 1, str(idx), x, BUBBLE180_PHASE_Y)
    for idx, x in enumerate(BUBBLE180_SET_X, start=1):
        add('Set', 1, str(idx), x, BUBBLE180_SET_Y)
    for idx, x in enumerate(BUBBLE180_CLASS_X):
        add('Class', 1, BUBBLE180_CLASS_LABELS[idx], x, BUBBLE180_CLASS_Y)
    for block, xs in enumerate(BUBBLE180_Q_XS):
        for row, y in enumerate(BUBBLE180_Q_Y):
            qnum = block*45 + row + 1
            for col, x in enumerate(xs):
                add('Answer', qnum, BUBBLE180_CHOICE_LABELS[col], x, y)
    return out


def _bubble180_score(reference_gray, aligned_gray, x, y, radius=9.0):
    """Score extra ink at the bubble centre vs blank reference, excluding its printed ring.

    The main score is the mean intensity decrease inside the inner circle. A local outer
    annulus compensates scanner illumination, but is bounded away from the outline so a
    fully darkened bubble cannot cancel its own score. Returns a grayscale-difference score.
    """
    h, w = reference_gray.shape[:2]
    x, y, r = int(round(x)), int(round(y)), max(5.0, float(radius))
    pad = int(round(1.65*r))
    x0, x1 = max(0, x-pad), min(w, x+pad+1)
    y0, y1 = max(0, y-pad), min(h, y+pad+1)
    ref = reference_gray[y0:y1, x0:x1].astype(np.float32)
    cur = aligned_gray[y0:y1, x0:x1].astype(np.float32)
    yy, xx = np.ogrid[y0:y1, x0:x1]
    dist = np.sqrt((xx-x)**2 + (yy-y)**2)
    inner = dist <= 0.47*r
    # Outer annulus starts outside the printed circle, not at the bubble edge.
    annulus = (dist >= 1.18*r) & (dist <= 1.55*r)
    delta = ref - cur
    raw = float(np.mean(delta[inner])) if np.any(inner) else 0.0
    local = float(np.median(delta[annulus])) if np.any(annulus) else 0.0
    # 80% main-centre evidence + only a 20% illumination correction: does not over-cancel
    # dark fills that bleed up to the printed ring on low-quality scans.
    return raw - 0.20*local


def _decode_single_choice(scores, labels, threshold, review_gap=15.0):
    chosen = [labels[i] for i, s in enumerate(scores) if s >= threshold]
    near = [labels[i] for i, s in enumerate(scores) if threshold-review_gap <= s < threshold]
    if len(chosen) == 1:
        return chosen[0], 'ONE_MARK', max(scores)
    if len(chosen) > 1:
        return '|'.join(chosen), 'MULTIPLE_MARKS', max(scores)
    if near:
        return '?', 'AMBIGUOUS', max(scores)
    return '', 'BLANK', max(scores)


def _bubble180_registration_value(records, threshold):
    chars, statuses, scores_out = [], [], []
    for col in range(1, 9):
        group = [r for r in records if r['field'] == 'Registration No.' and r['question'] == col]
        scores = [next((b['score'] for b in group if int(b['digit']) == d), -999.0) for d in range(10)]
        value, status, score = _decode_single_choice(scores, [str(d) for d in range(10)], threshold)
        chars.append(value if status in ('ONE_MARK', 'MULTIPLE_MARKS', 'AMBIGUOUS') else '')
        statuses.append(status)
        scores_out.append(score)
    # If a registration column has multiple marks, preserve all possible digits explicitly.
    return ''.join(ch if ch and not ch.startswith('MULTIPLE') else ('?' if ch.startswith('MULTIPLE') else '') for ch in chars), statuses, scores_out


def bubble_only_180_batch(omr_files, sensitivity=0.75, progress=gr.Progress(track_tqdm=True)):
    """Scan the alternate 8-digit-reg / Phase / Set / Class / 180-MCQ form using bubbles only."""
    run_id = f'bubble180_{int(time.time()*1000)}_{os.getpid()}'
    work = RUN_DIR / run_id
    work.mkdir(parents=True, exist_ok=True)
    empty_columns = ['Registration No.', 'Phase', 'Set', 'Class'] + [f'Q{i}' for i in range(1, 181)]
    diag_columns = ['Source File', 'Page', 'Alignment', 'Alignment Confidence', 'Filled Bubbles', 'Ambiguous Bubbles', 'Multi-marked Questions', 'Status', 'Error']
    try:
        if not omr_files:
            raise ValueError('Pehle student OMR images/PDF upload karein.')
        _, template_gray = get_bubble_only_reference()
        geometry = _bubble180_geometry(template_gray)
        threshold = 88.0 - 38.0*float(np.clip(float(sensitivity), 0.0, 1.0))
        all_rows, diagnostics, gallery_items, bubble_rows = [], [], [], []
        page_records = []
        input_pages = []
        for file in omr_files:
            try:
                input_pages.extend(paths_to_pages(str(file), work, 'student180'))
            except Exception as exc:
                diagnostics.append({'Source File':Path(str(file)).name,'Page':'','Alignment':'','Alignment Confidence':'',
                    'Filled Bubbles':'','Ambiguous Bubbles':'','Multi-marked Questions':'','Status':'PROCESSING_FAILED','Error':f'{type(exc).__name__}: {exc}'})
        total = len(input_pages)
        if total == 0:
            raise ValueError('Kisi bhi input se readable OMR page nahi bana. JPG/PNG/PDF format check karein.')
        for page_idx, (page_path, source_name, page_no) in enumerate(input_pages, start=1):
            progress((page_idx-1)/max(1,total), desc=f'Bubble-only scan {page_idx}/{total}: {source_name} page {page_no}')
            try:
                original_bgr = safe_read_image(page_path)
                original_gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
                aligned, align_status, align_conf, aspect_delta, transform = align_to_template(template_gray, original_gray)
                current_page_record={'Source File':source_name,'Page':page_no,'page_path':str(page_path),
                    'original_width':int(original_bgr.shape[1]),'original_height':int(original_bgr.shape[0]),
                    'source_to_template':transform,'alignment_status':align_status,'alignment_confidence':float(align_conf),
                    'threshold':float(threshold),'result_row_index':None}
                scores = []
                overlay = aligned.copy()
                for g in geometry:
                    score = _bubble180_score(template_gray, aligned, g['x'], g['y'], g['r'])
                    scores.append(score)
                # Build field decoding from the explicit coordinates; no OCR is called here.
                cell_idx = 0
                for g, score in zip(geometry, scores):
                    state = 'FILLED' if score >= threshold else ('AMBIGUOUS' if score >= threshold-15.0 else 'EMPTY')
                    bubble_rows.append({'Source File':source_name,'Page':page_no,'Field':g['field'],
                        'Question/Position':g['question'],'Option/Digit':g['option'],'Score':round(float(score),1),'State':state})
                    colour = (20, 45, 220) if state == 'FILLED' else ((0, 180, 255) if state == 'AMBIGUOUS' else (50, 170, 50))
                    cv2.circle(overlay, (int(round(g['x'])), int(round(g['y']))), max(3,int(round(g['r']*0.75))), colour, 1, cv2.LINE_AA)
                # Registration number: one bubble digit per column is expected; never OCR handwriting.
                scores_by_field = {}
                for g, score in zip(geometry, scores):
                    scores_by_field[(g['field'],g['question'],str(g['option']))] = float(score)
                reg_chars, reg_statuses = [], []
                for col in range(1,9):
                    vals=[scores_by_field[('Registration No.',col,str(d))] for d in range(10)]
                    value, st, _ = _decode_single_choice(vals,[str(d) for d in range(10)],threshold)
                    reg_chars.append(value if st in ('ONE_MARK','MULTIPLE_MARKS','AMBIGUOUS') else '')
                    reg_statuses.append(st)
                reg_value=''.join(('?' if v and ('|' in v or v=='?') else v) for v in reg_chars)
                phase_scores=[scores_by_field[('Phase',1,str(i))] for i in range(1,7)]
                phase, phase_st, _ = _decode_single_choice(phase_scores,[str(i) for i in range(1,7)],threshold)
                set_scores=[scores_by_field[('Set',1,str(i))] for i in range(1,4)]
                set_val, set_st, _ = _decode_single_choice(set_scores,[str(i) for i in range(1,4)],threshold)
                class_scores=[scores_by_field[('Class',1,label)] for label in BUBBLE180_CLASS_LABELS]
                class_val, class_st, _ = _decode_single_choice(class_scores,BUBBLE180_CLASS_LABELS,threshold)
                answer_values=[]; ambiguous_answers=0; multi_questions=0
                for q in range(1,181):
                    block=(q-1)//45; row=(q-1)%45
                    vals=[scores_by_field[('Answer',q,lab)] for lab in BUBBLE180_CHOICE_LABELS]
                    value, st, mx = _decode_single_choice(vals,BUBBLE180_CHOICE_LABELS,threshold)
                    answer_values.append(value)
                    if st == 'AMBIGUOUS': ambiguous_answers += 1
                    if st == 'MULTIPLE_MARKS': multi_questions += 1
                row = {'Registration No.':reg_value,'Phase':phase,'Set':set_val,'Class':class_val}
                row.update({f'Q{i}':answer_values[i-1] for i in range(1,181)})
                current_page_record['result_row_index']=len(all_rows)
                all_rows.append(row)
                page_records.append(current_page_record)
                filled=sum(s>=threshold for s in scores)
                ambiguous=sum(threshold-15.0<=s<threshold for s in scores)
                page_status='NEEDS_REVIEW' if (align_conf < 0.50 or reg_statuses.count('ONE_MARK') < 6 or phase_st != 'ONE_MARK' or set_st != 'ONE_MARK' or class_st != 'ONE_MARK' or ambiguous or ambiguous_answers or multi_questions) else 'BUBBLE_SCAN_COMPLETE'
                diagnostics.append({'Source File':source_name,'Page':page_no,'Alignment':align_status,
                    'Alignment Confidence':round(float(align_conf),3),'Filled Bubbles':filled,
                    'Ambiguous Bubbles':ambiguous,'Multi-marked Questions':multi_questions,
                    'Status':page_status,'Error':''})
                # Add question-level detail to the zipped diagnostic file, without handwriting fields.
                for q, val in enumerate(answer_values, start=1):
                    bubble_rows.append({'Source File':source_name,'Page':page_no,'Field':'Answer decoded',
                        'Question/Position':q,'Option/Digit':val,'Score':'','State':'MULTIPLE_MARKS' if '|' in val else ('AMBIGUOUS' if val=='?' else ('BLANK' if not val else 'DECODED'))})
                annotated = work / f'preview_{page_idx:04d}_{slugify(Path(source_name).stem)}_p{page_no}.jpg'
                save_image(annotated, overlay)
                gallery_items.append((str(annotated), f'{source_name} | page {page_no} | {page_status}'))
            except Exception as exc:
                diagnostics.append({'Source File':source_name,'Page':page_no,'Alignment':'','Alignment Confidence':'',
                    'Filled Bubbles':'','Ambiguous Bubbles':'','Multi-marked Questions':'','Status':'PROCESSING_FAILED',
                    'Error':f'{type(exc).__name__}: {exc}'})
        results_df = pd.DataFrame(all_rows, columns=empty_columns)
        diag_df = pd.DataFrame(diagnostics, columns=diag_columns)
        (work/'bubble180_page_manifest.json').write_text(json.dumps(page_records,ensure_ascii=False,indent=2),encoding='utf-8')
        bubble_df = pd.DataFrame(bubble_rows, columns=['Source File','Page','Field','Question/Position','Option/Digit','Score','State'])
        if results_df.empty:
            raise ValueError('Kisi OMR se data extract nahi hua. Diagnostics table mein exact error dekhein.')
        outdir=work/'outputs'; outdir.mkdir(exist_ok=True)
        xlsx=outdir/'PW_OMR_180Q_Bubble_Only.xlsx'
        csvp=outdir/'PW_OMR_180Q_Bubble_Only.csv'
        diag_csv=outdir/'PW_OMR_180Q_Diagnostics.csv'
        detail_csv=outdir/'PW_OMR_180Q_Bubble_Details.csv'
        # Main Excel has only bubbled fields: no handwritten name, centre, batch, room or date.
        with pd.ExcelWriter(xlsx, engine='openpyxl') as writer:
            results_df.to_excel(writer, sheet_name='Bubbled Results', index=False)
            ws=writer.sheets['Bubbled Results']
            ws.freeze_panes='E2'; ws.sheet_view.showGridLines=False
            ws.auto_filter.ref=ws.dimensions
            ws.page_setup.orientation='landscape'; ws.page_setup.paperSize=ws.PAPERSIZE_A3
            ws.page_setup.fitToWidth=1; ws.page_setup.fitToHeight=0
            ws.sheet_properties.pageSetUpPr.fitToPage=True
            ws.column_dimensions['A'].width=18; ws.column_dimensions['B'].width=10; ws.column_dimensions['C'].width=10; ws.column_dimensions['D'].width=14
            for c in range(5,185): ws.column_dimensions[ws.cell(1,c).column_letter].width=5
            from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
            for cell in ws[1]:
                cell.fill=PatternFill('solid',fgColor='1F4E78'); cell.font=Font(name='Arial',size=9,bold=True,color='FFFFFF')
                cell.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
            ws.row_dimensions[1].height=30
            for row_i in range(2,ws.max_row+1):
                for col_i in range(1,ws.max_column+1):
                    cell=ws.cell(row_i,col_i); cell.font=Font(name='Arial',size=9)
                    if col_i<=4: cell.number_format='@'
                    cell.alignment=Alignment(horizontal='center',vertical='center')
        results_df.to_csv(csvp,index=False,encoding='utf-8-sig')
        diag_df.to_csv(diag_csv,index=False,encoding='utf-8-sig')
        bubble_df.to_csv(detail_csv,index=False,encoding='utf-8-sig')
        zipp=outdir/'PW_OMR_180Q_Bubble_Only_Results.zip'
        with zipfile.ZipFile(zipp,'w',zipfile.ZIP_DEFLATED) as z:
            for p in [xlsx,csvp,diag_csv,detail_csv]: z.write(p,p.name)
        xlsx_dl=stage_download_file(xlsx,'xlsx')
        csv_dl=stage_download_file(csvp,'file')
        zip_dl=stage_download_file(zipp,'zip')
        failed=int((diag_df['Status']=='PROCESSING_FAILED').sum()) if not diag_df.empty else 0
        review=int((diag_df['Status']=='NEEDS_REVIEW').sum()) if not diag_df.empty else 0
        msg=(f'Bubble-only scan complete: {len(results_df)} page(s) exported; {failed} failed; {review} need review. '
             f'Sensitivity={float(sensitivity):.2f}; score threshold={threshold:.1f}. '
             'Export fields are Registration No., Phase, Set, Class and Q1-Q180 only. No handwriting OCR or name/centre/batch/room/date extraction is used. '
             'Multiple marked answers are preserved as A|C; ambiguous readings are shown as ?. Original scans remain unchanged.')
        return msg, results_df, diag_df, gallery_items, xlsx_dl, csv_dl, zip_dl, str(work)
    except Exception as exc:
        msg=f'BUBBLE-ONLY PROCESSING FAILED: {type(exc).__name__}: {exc}\n{traceback.format_exc()[-2000:]}'
        return msg, pd.DataFrame(columns=empty_columns), pd.DataFrame([{'Source File':'','Page':'','Alignment':'','Alignment Confidence':'','Filled Bubbles':'','Ambiguous Bubbles':'','Multi-marked Questions':'','Status':'BATCH_FAILED','Error':f'{type(exc).__name__}: {exc}'}],columns=diag_columns), [], None, None, None, ''


def run_internal_self_tests():
    blank = np.full((41,41), 255, dtype=np.uint8)
    cv2.circle(blank, (20,20), 10, 0, 1)
    filled = blank.copy()
    cv2.circle(filled, (20,20), 7, 0, -1)
    blank_score, _ = bubble_metrics(blank,20,20,10)
    filled_score, _ = bubble_metrics(filled,20,20,10)
    assert filled_score > blank_score, 'Bubble score self-test failed.'
    # A visibly light-grey filled circle must score above the blank circle instead of being ignored.
    faint = blank.copy()
    cv2.circle(faint, (20,20), 6, 170, -1)
    faint_score, _ = bubble_metrics(faint,20,20,10)
    relative_faint_score=max(0.0, faint_score-blank_score+0.08)
    test_sensitivity=0.70
    test_threshold=0.55-0.36*test_sensitivity
    test_margin=max(0.045,0.080-0.025*test_sensitivity)
    faint_state='FILLED' if relative_faint_score >= test_threshold+test_margin else ('EMPTY' if relative_faint_score <= test_threshold-test_margin else 'AMBIGUOUS')
    assert relative_faint_score > 0.15 and faint_state != 'EMPTY', 'Faint-bubble self-test failed.'
    assert guess_group_type(12,4,0.95).startswith('mcq_grid'), 'MCQ inference self-test failed.'
    # Digit-grid rows should always map 0-9 top-to-bottom in the fixed template profile.
    assert guess_group_type(10,10,0.95).startswith('digit_grid'), 'Digit-grid inference self-test failed.'
    assert guess_group_type(26,16,0.90).startswith('letter_grid'), 'Letter-grid inference self-test failed.'
    one_mark=[{'question_or_position':1,'status':'ONE_MARK','value':'A'}]
    # Regression: OCR returns empty/low-ink for a real bubbled A. The A must not disappear.
    fused,status=fuse_name_bubbles_with_handwriting(one_mark,[{'char':'','confidence':-1,'ink_fraction':0.01,'support':0}])
    assert fused=='A', f'Name fusion erased a valid bubble letter: {fused!r}'
    # A weak OCR disagreement must keep the bubble result and flag review, not overwrite it.
    fused,status=fuse_name_bubbles_with_handwriting(one_mark,[{'char':'B','confidence':75,'ink_fraction':0.15,'support':2}])
    assert fused=='A' and 'REVIEW SLOTS: 1' in status, 'Weak OCR disagreement must preserve bubble value and request review.'
    # Bubble-first regression: OCR disagreement must never replace the detected bubbles.
    selected,from_ocr,reason=choose_name_value('WRONGNAME',0.99,'WRONGNAME','SUHANA SHAIKH','Tesseract line OCR; confidence uncalibrated',12,False)
    assert selected=='WRONGNAME' and not from_ocr and 'BUBBLE-FIRST' in reason, 'OCR must not overwrite bubble-derived name.'
    assert _normalize_page_key('1.0')=='1' and _normalize_page_key(2)=='2', 'Page-key normalization regression failed.'
    # Alternate 180Q bubble-only form: expected 80 registration bubbles, 12 metadata choices, 720 answer bubbles.
    _ref_bgr, _ref_gray = get_bubble_only_reference()
    _geometry = _bubble180_geometry(_ref_gray)
    assert len(_geometry) == 812, f'180Q bubble-only geometry count mismatch: {len(_geometry)}'
    assert sum(1 for _g in _geometry if _g['field'] == 'Answer') == 720, '180Q answer bubble grid must contain 720 cells.'
    assert sum(1 for _g in _geometry if _g['field'] == 'Registration No.') == 80, 'Registration bubble grid must contain 80 cells.'
    return {'bubble_score_test':'PASS','faint_bubble_test':'PASS','group_type_test':'PASS','bubble_first_no_ocr_overwrite':'PASS','page_key_normalization':'PASS','180Q_bubble_geometry_812_cells':'PASS'}

SELF_TEST_RESULT = run_internal_self_tests()

# ================== v12 MANUAL BUBBLE CORRECTION WORKFLOW ==================
# Provides an in-app, per-page visual editor for both the NSAT grid template and
# the 180-question bubble-only template. Original source scans are never overwritten.

def _manual_json_read(path, default=None):
    try:
        p=Path(path)
        if p.exists():
            return json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        pass
    return {} if default is None else default


def _manual_json_write(path, payload):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_suffix(p.suffix+'.tmp')
    tmp.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
    tmp.replace(p)
    return str(p)


def _save_png_copy_with_source_dpi(source_path, png_path, bgr_image):
    """Backward-compatible wrapper. Corrected previews now follow the source image format and metadata."""
    return _save_corrected_copy_like_source(source_path,Path(str(png_path)).with_suffix(''),bgr_image)[0]

# ---------- v15: source-pixel-preserving bubble correction + image-only ZIP ----------
CORRECTED_BUBBLE_FILL_RATIO = 0.925
CORRECTED_BUBBLE_CLEAR_RATIO = 0.865

def _project_template_circle_to_source(H_inv, x, y, r, ratio=1.0, samples=64):
    """Map a circle in registered-template coordinates into source-pixel polygon coordinates."""
    angles=np.linspace(0.0,2.0*np.pi,int(samples),endpoint=False,dtype=np.float32)
    rr=max(1.0,float(r)*float(ratio))
    pts=np.stack([float(x)+rr*np.cos(angles),float(y)+rr*np.sin(angles)],axis=1).astype(np.float32)
    mapped=cv2.perspectiveTransform(pts.reshape(1,-1,2),np.asarray(H_inv,dtype=np.float32))[0]
    if not np.isfinite(mapped).all():
        raise ValueError('Bubble mapping produced invalid source coordinates.')
    return np.round(mapped).astype(np.int32)


def _blend_reference_polygon(image_bgr, reference_bgr, polygon, black_fill=False):
    """Blend only a polygon ROI; all pixels outside it remain byte-for-byte unchanged in memory."""
    if polygon is None or len(polygon)<3: return image_bgr
    h,w=image_bgr.shape[:2]
    x0=max(0,int(np.min(polygon[:,0]))-2); x1=min(w,int(np.max(polygon[:,0]))+3)
    y0=max(0,int(np.min(polygon[:,1]))-2); y1=min(h,int(np.max(polygon[:,1]))+3)
    if x1<=x0 or y1<=y0: return image_bgr
    local=polygon.copy(); local[:,0]-=x0; local[:,1]-=y0
    mask=np.zeros((y1-y0,x1-x0),dtype=np.uint8)
    cv2.fillPoly(mask,[local.reshape(-1,1,2)],255,lineType=cv2.LINE_AA)
    alpha=(mask.astype(np.float32)/255.0)[:,:,None]
    roi=image_bgr[y0:y1,x0:x1].astype(np.float32)
    if black_fill:
        mixed=roi*(1.0-alpha)
    else:
        ref=reference_bgr[y0:y1,x0:x1].astype(np.float32)
        mixed=roi*(1.0-alpha)+ref*alpha
    image_bgr[y0:y1,x0:x1]=np.clip(mixed,0,255).astype(np.uint8)
    return image_bgr


def _make_reference_in_source(template_gray, H, source_shape):
    """Warp only the blank reference template into source geometry; never warp the source scan."""
    h,w=source_shape[:2]
    H=np.asarray(H,dtype=np.float64)
    if H.shape!=(3,3) or not np.isfinite(H).all(): raise ValueError('Invalid source-to-template homography.')
    H_inv=np.linalg.inv(H)
    clean=cv2.warpPerspective(template_gray,H_inv,(w,h),flags=cv2.INTER_CUBIC,borderValue=255)
    clean_bgr=cv2.cvtColor(clean,cv2.COLOR_GRAY2BGR)
    return H_inv,clean_bgr


def _cell_template_geometry(cell, template_width, template_height):
    """Convert either normalized NSAT circle data or absolute 180Q geometry to template pixels."""
    if 'x_norm' in cell:
        return (float(cell['x_norm'])*template_width,
                float(cell['y_norm'])*template_height,
                max(2.0,float(cell['r_norm'])*min(template_height,template_width)))
    return float(cell['x']),float(cell['y']),max(2.0,float(cell.get('r',9.0)))


def _apply_source_slot_circles(out_bgr, clean_reference_bgr, H_inv, cells, selected_cells,
                               template_width, template_height,
                               clear_ratio=CORRECTED_BUBBLE_CLEAR_RATIO,
                               fill_ratio=CORRECTED_BUBBLE_FILL_RATIO):
    """Restore competing bubbles from the aligned blank template, then fully darken the chosen bubble(s)."""
    # First clean every bubble interior in this slot / field. Printed ring and outside pixels are preserved.
    for cell in cells:
        x,y,r=_cell_template_geometry(cell,template_width,template_height)
        poly=_project_template_circle_to_source(H_inv,x,y,r,clear_ratio)
        _blend_reference_polygon(out_bgr,clean_reference_bgr,poly,black_fill=False)
    # Then cover nearly all inner circle area with a solid black mark, with a narrow outline margin.
    for cell in selected_cells:
        x,y,r=_cell_template_geometry(cell,template_width,template_height)
        poly=_project_template_circle_to_source(H_inv,x,y,r,fill_ratio)
        _blend_reference_polygon(out_bgr,clean_reference_bgr,poly,black_fill=True)
    return out_bgr


def _save_corrected_copy_like_source(source_path, output_base, bgr_image):
    """Save corrected pixels in the source image format, retaining dimensions and available DPI/EXIF/ICC metadata.

    JPEG source files are encoded at quality 100 with 4:4:4 chroma. This avoids the previous full-page
    perspective-warp blur; only bubble ROIs are changed before encoding. If source DPI is absent, none is invented.
    """
    from PIL import Image as PILImage
    source_path=Path(str(source_path)); output_base=Path(str(output_base))
    output_base.parent.mkdir(parents=True,exist_ok=True)
    with PILImage.open(source_path) as src_im:
        fmt=(src_im.format or source_path.suffix.lstrip('.')).upper()
        source_size=src_im.size
        dpi=src_im.info.get('dpi')
        exif=src_im.info.get('exif')
        icc=src_im.info.get('icc_profile')
        pnginfo=src_im.info.get('pnginfo')
        fmt_ext={'JPEG':'.jpg','JPG':'.jpg','PNG':'.png','TIFF':'.tif','TIF':'.tif','WEBP':'.webp','BMP':'.bmp'}
        out_fmt=fmt if fmt in fmt_ext else 'PNG'
        out_ext=fmt_ext.get(out_fmt,'.png')
    target=output_base.with_suffix(out_ext)
    rgb=cv2.cvtColor(bgr_image,cv2.COLOR_BGR2RGB)
    out_im=PILImage.fromarray(rgb)
    kwargs={}
    if dpi:
        kwargs['dpi']=tuple(dpi) if isinstance(dpi,(tuple,list)) else dpi
    if exif and out_fmt in ('JPEG','WEBP','TIFF'):
        kwargs['exif']=exif
    if icc:
        kwargs['icc_profile']=icc
    if out_fmt in ('JPEG','JPG'):
        kwargs.update({'format':'JPEG','quality':100,'subsampling':0,'optimize':False,'progressive':False})
    elif out_fmt=='PNG':
        kwargs.update({'format':'PNG','compress_level':1})
    elif out_fmt in ('TIFF','TIF'):
        kwargs.update({'format':'TIFF','compression':'tiff_lzw'})
    elif out_fmt=='WEBP':
        kwargs.update({'format':'WEBP','quality':100,'lossless':True,'method':4})
    elif out_fmt=='BMP':
        kwargs.update({'format':'BMP'})
    else:
        kwargs.update({'format':'PNG'}); target=output_base.with_suffix('.png')
    out_im.save(target,**kwargs)
    with PILImage.open(target) as chk:
        if chk.size != source_size or chk.size != (bgr_image.shape[1],bgr_image.shape[0]):
            target.unlink(missing_ok=True)
            raise ValueError(f'Corrected image dimension mismatch: source={source_size}, output={chk.size}')
        out_dpi=chk.info.get('dpi')
        if dpi and out_dpi:
            if any(abs(float(a)-float(b))>1.0 for a,b in zip(dpi,out_dpi)):
                target.unlink(missing_ok=True)
                raise ValueError(f'DPI metadata verification failed: source={dpi}, output={out_dpi}')
    return str(target), {'source_format':fmt,'output_format':out_fmt,'dimensions_px':f'{source_size[0]}x{source_size[1]}',
                         'source_dpi':str(dpi) if dpi else 'not embedded in source','output_dpi':str(out_dpi) if out_dpi else 'not embedded'}


def _build_images_only_zip(image_paths, out_path, run_tag='corrected_omr_images'):
    """Bundle every successful corrected image into one validated image-only ZIP for direct download."""
    image_paths=[Path(str(p)) for p in (image_paths or []) if p and Path(str(p)).is_file() and Path(str(p)).stat().st_size>0]
    if not image_paths: return None, 'No corrected OMR images exist yet. Correct at least one OMR and save/export again.'
    out_path=Path(out_path); out_path.parent.mkdir(parents=True,exist_ok=True)
    used=set()
    with zipfile.ZipFile(out_path,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
        for path in image_paths:
            name=path.name
            if name in used:
                stem=path.stem; suffix=path.suffix; n=2
                while f'{stem}_{n}{suffix}' in used: n+=1
                name=f'{stem}_{n}{suffix}'
            used.add(name)
            zf.write(path,arcname=f'corrected_omr/{name}')
        zf.writestr('README.txt',f'Corrected OMR image copies: {len(used)}. Original uploads were not overwritten. Pixel dimensions are preserved; DPI metadata is included only if present in each source image.\n')
    with zipfile.ZipFile(out_path,'r') as zf:
        bad=zf.testzip()
        if bad: raise ValueError(f'Corrected-image ZIP verification failed at {bad}')
        image_count=sum(1 for n in zf.namelist() if n.startswith('corrected_omr/'))
        if image_count != len(used): raise ValueError('Corrected-image ZIP count validation failed.')
    return stage_download_file(out_path,'zip'), f'Validated image-only ZIP created with {len(used)} corrected OMR image(s).'


def _apply_nsat_group_value_to_source(out_bgr, clean_ref_bgr, H_inv, profile, group_id, slot_symbols, kind, tw, th):
    groups={str(g.get('group_id')):g for g in profile.get('groups',[])}
    group=groups.get(str(group_id))
    if not group: return False,f'{group_id}: group not in profile'
    col_count=int(group.get('cols',0))
    if len(slot_symbols)!=col_count: return False,f'{group_id}: expected {col_count} symbols, got {len(slot_symbols)}'
    cells=[c for c in profile.get('circles',[]) if str(c.get('group_id'))==str(group_id)]
    if not cells: return False,f'{group_id}: no mapped bubbles'
    wanted=[]
    for col,sym in enumerate(slot_symbols,1):
        sym=str(sym).upper()
        if not sym or sym==' ': continue
        if kind=='letters':
            if sym not in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ': return False,f'{group_id}: invalid name character {sym}'
            row=ord(sym)-ord('A')+1
        elif kind=='digits':
            if sym not in '0123456789': return False,f'{group_id}: invalid digit {sym}'
            row=int(sym)+1
        elif kind=='choices':
            labels=[str(x).upper() for x in group.get('choice_labels',[])]
            if sym not in labels: return False,f'{group_id}: invalid choice {sym}'
            row=1
        else: return False,f'{group_id}: unsupported field kind'
        match=[c for c in cells if int(c.get('grid_col') or -1)==col and int(c.get('grid_row') or -1)==row]
        if len(match)!=1: return False,f'{group_id}: requested row {row}, column {col} not mapped'
        wanted.append(match[0])
    _apply_source_slot_circles(out_bgr,clean_ref_bgr,H_inv,cells,wanted,tw,th)
    return True,f'{group_id}: redraw OK (source-pixel edits)'


def _apply_nsat_manual_edits_to_source(out_bgr, clean_ref_bgr, H_inv, profile, edits, tw, th):
    groups={str(g.get('group_id')):g for g in profile.get('groups',[])}
    all_cells=profile.get('circles',[])
    for edit in (edits or {}).values():
        gid=str(edit.get('group_id','')); group=groups.get(gid)
        if not group: continue
        try: pos=int(edit.get('position',1))
        except Exception: continue
        axis=str(edit.get('axis','col')); selected=[int(x) for x in edit.get('selected_indices',[]) if str(x).lstrip('-').isdigit()]
        cells=[c for c in all_cells if str(c.get('group_id'))==gid]
        if axis=='col': slot_cells=[c for c in cells if int(c.get('grid_col') or -1)==pos]
        elif axis=='row': slot_cells=[c for c in cells if int(c.get('grid_row') or -1)==pos]
        else: slot_cells=list(cells)
        if not slot_cells: continue
        selected_cells=[]
        for c in slot_cells:
            index=int(c.get('grid_row') or -1) if axis=='col' else int(c.get('grid_col') or -1)
            if index in selected: selected_cells.append(c)
        _apply_source_slot_circles(out_bgr,clean_ref_bgr,H_inv,slot_cells,selected_cells,tw,th)
    return out_bgr


def _get_nsat_manual_context(run_directory):
    if not run_directory or not Path(str(run_directory)).exists():
        raise ValueError('Pehle Tab 2 mein NSAT OMR batch scan karein.')
    run=Path(str(run_directory))
    manifest_path=run/'page_manifest.json'; profile_path=run/'profile_used.json'
    if not manifest_path.exists() or not profile_path.exists():
        raise ValueError('NSAT manual correction ke liye page manifest/profile nahi mila. Batch ko dobara scan karein.')
    manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
    profile=json.loads(profile_path.read_text(encoding='utf-8'))
    if not manifest: raise ValueError('Scan batch mein koi page record nahi hai.')
    template_bgr=safe_read_image(profile['template_path'])
    template_gray=cv2.cvtColor(template_bgr,cv2.COLOR_BGR2GRAY)
    return run, manifest, profile, template_gray


def _nsat_page_label(rec, i):
    return f"{i+1}. {rec.get('Source File','OMR')} — page {rec.get('Page',1)}"


def _nsat_page_choices(run_directory):
    run, manifest, profile, _ = _get_nsat_manual_context(run_directory)
    return [(_nsat_page_label(rec,i),str(i)) for i,rec in enumerate(manifest)]


def _nsat_group_choices(profile):
    choices=[]
    for g in profile.get('groups',[]):
        gid=str(g.get('group_id',''))
        typ=str(g.get('suggested_type','generic_grid'))
        label=str(g.get('label') or g.get('label_hint') or gid)
        choices.append((f"{gid} — {label} ({typ}; {g.get('rows','?')}×{g.get('cols','?')})",gid))
    return choices


def _nsat_group_axis_and_labels(group):
    typ=str(group.get('suggested_type','generic_grid')).lower()
    rows=max(1,int(group.get('rows') or 1)); cols=max(1,int(group.get('cols') or 1))
    if 'letter_grid' in typ:
        return 'col', [(str(i),f"Slot {i}") for i in range(1,cols+1)], list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')[:rows]
    if 'digit_grid' in typ:
        return 'col', [(str(i),f"Digit position {i}") for i in range(1,cols+1)], [str(i) for i in range(min(10,rows))]
    if 'mcq_grid' in typ:
        try: start=int(float(str(group.get('question_start','')).strip()))
        except Exception: start=None
        pos=[(str(i),f"Question {start+i-1}" if start is not None else f"Grid row {i}") for i in range(1,rows+1)]
        option_labels=list('ABCD') if cols==4 else (list('ABCDE')[:cols] if cols<=5 else [str(i) for i in range(1,cols+1)])
        return 'row',pos,option_labels
    if 'single_row_choices' in typ:
        labels=list(group.get('choice_labels',[])) or [f'Choice {i}' for i in range(1,cols+1)]
        return 'single', [('1','Choice row')], [str(x) for x in labels]
    # For unclassified grids, use column slots and row-number choices so the operator can still edit safely.
    return 'col', [(str(i),f"Column / slot {i}") for i in range(1,cols+1)], [str(i) for i in range(1,rows+1)]


def load_nsat_manual_editor(run_directory):
    try:
        run, manifest, profile, template_gray = _get_nsat_manual_context(run_directory)
        pages=[(_nsat_page_label(r,i),str(i)) for i,r in enumerate(manifest)]
        groups=_nsat_group_choices(profile)
        page_val=pages[0][1] if pages else None
        group_val=groups[0][1] if groups else None
        group=next((g for g in profile.get('groups',[]) if str(g.get('group_id'))==str(group_val)),{})
        axis,pos,opts=_nsat_group_axis_and_labels(group) if group else ('col',[],[])
        original, corrected, msg=render_nsat_manual_page(run_directory,page_val)
        return gr.update(choices=pages,value=page_val), gr.update(choices=groups,value=group_val), gr.update(choices=[(lab,val) for val,lab in pos],value=(pos[0][0] if pos else None)), gr.update(choices=[(str(x),str(x)) for x in opts],value=(str(opts[0]) if opts else None)), original, corrected, 'Loaded '+str(len(pages))+' NSAT page(s). Choose a page, grid, position and bubble. '+msg
    except Exception as exc:
        return gr.update(choices=[],value=None), gr.update(choices=[],value=None), gr.update(choices=[],value=None), gr.update(choices=[],value=None), None, None, f'Could not load NSAT manual correction: {type(exc).__name__}: {exc}'


def nsat_manual_group_controls(run_directory, group_id):
    try:
        run, manifest, profile, _ = _get_nsat_manual_context(run_directory)
        group=next((g for g in profile.get('groups',[]) if str(g.get('group_id'))==str(group_id)),None)
        if not group: return gr.update(choices=[],value=None),gr.update(choices=[],value=None),'Select a valid grid group.'
        axis,pos,opts=_nsat_group_axis_and_labels(group)
        return gr.update(choices=[(label,val) for val,label in pos],value=(pos[0][0] if pos else None)), gr.update(choices=[(str(x),str(x)) for x in opts],value=(str(opts[0]) if opts else None)), f'Grid {group_id}: editing by {axis}. Set-selected clears competing marks in the same slot; Toggle can add/remove a bubble.'
    except Exception as exc:
        return gr.update(choices=[],value=None),gr.update(choices=[],value=None),f'Grid control error: {type(exc).__name__}: {exc}'


def _get_nsat_edits(run_directory, page_idx):
    state=_manual_json_read(Path(str(run_directory))/'manual_bubble_edits_nsat.json',{'pages':{}})
    return state.get('pages',{}).get(str(page_idx),{}).get('edits',{})


def _apply_nsat_manual_edits_to_aligned(aligned_gray, change_mask, template_gray, profile, edits):
    """Apply saved grid-slot edits on an aligned copy; return corrected grey image and mask."""
    corrected=aligned_gray.copy(); h,w=corrected.shape[:2]
    groups={str(g.get('group_id')):g for g in profile.get('groups',[])}
    all_cells=profile.get('circles',[])
    if change_mask is None: change_mask=np.zeros((h,w),dtype=np.uint8)
    for edit in (edits or {}).values():
        gid=str(edit.get('group_id','')); group=groups.get(gid)
        if not group: continue
        try: pos=int(edit.get('position',1))
        except Exception: continue
        axis=str(edit.get('axis','col')); selected=[int(x) for x in edit.get('selected_indices',[]) if str(x).lstrip('-').isdigit()]
        cells=[c for c in all_cells if str(c.get('group_id'))==gid]
        if axis=='col': slot_cells=[c for c in cells if int(c.get('grid_col') or -1)==pos]
        elif axis=='row': slot_cells=[c for c in cells if int(c.get('grid_row') or -1)==pos]
        else: slot_cells=list(cells)
        if not slot_cells: continue
        # Restore the bubble interiors to reference pixels, preserving printed outlines and surrounding text.
        for c in slot_cells:
            x=int(round(float(c['x_norm'])*w)); y=int(round(float(c['y_norm'])*h)); r=max(3,int(round(float(c['r_norm'])*min(h,w))))
            y0=max(0,y-int(1.22*r)); y1=min(h,y+int(1.22*r)+1); x0=max(0,x-int(1.22*r)); x1=min(w,x+int(1.22*r)+1)
            yy,xx=np.ogrid[y0:y1,x0:x1]; dist=np.sqrt((xx-x)**2+(yy-y)**2)
            inner=dist <= max(2.0,0.88*r)
            roi=corrected[y0:y1,x0:x1]; ref=template_gray[y0:y1,x0:x1]
            roi[inner]=ref[inner]
            cm=change_mask[y0:y1,x0:x1]; cm[dist<=1.25*r]=255
        # Axis col -> selected_indices are grid rows; axis row/single -> selected_indices are grid columns.
        for c in slot_cells:
            index=int(c.get('grid_row') or -1) if axis=='col' else int(c.get('grid_col') or -1)
            if index not in selected: continue
            x=int(round(float(c['x_norm'])*w)); y=int(round(float(c['y_norm'])*h)); r=max(3,int(round(float(c['r_norm'])*min(h,w))))
            draw_filled_bubble(corrected,x,y,r,fill_ratio=0.80)
    return corrected, change_mask


def render_nsat_manual_page(run_directory, page_value):
    """Render a sharp corrected preview by editing only bubble ROIs in original source pixels."""
    try:
        run,manifest,profile,template_gray=_get_nsat_manual_context(run_directory)
        page_idx=int(page_value or 0)
        if page_idx<0 or page_idx>=len(manifest): raise ValueError('Selected OMR page does not exist.')
        rec=manifest[page_idx]; src=Path(rec['page_path'])
        if not src.exists(): raise FileNotFoundError('Original page copy missing: '+src.name)
        original=safe_read_image(src); h,w=original.shape[:2]; th,tw=template_gray.shape[:2]
        H=np.asarray(rec.get('source_to_template'),dtype=np.float64)
        H_inv,clean_ref=_make_reference_in_source(template_gray,H,original.shape)
        edits=_get_nsat_edits(run,page_idx)
        corrected=original.copy()
        if edits: _apply_nsat_manual_edits_to_source(corrected,clean_ref,H_inv,profile,edits,tw,th)
        viewdir=run/'manual_bubble_correction'/'nsat_previews'; viewdir.mkdir(parents=True,exist_ok=True)
        outbase=viewdir/f"PREVIEW_{page_idx+1:04d}_{slugify(Path(str(rec.get('Source File','OMR'))).stem)}_p{rec.get('Page',1)}"
        outpath,meta=_save_corrected_copy_like_source(src,outbase,corrected)
        msg=f"Selected: {rec.get('Source File')} | page {rec.get('Page')}. Original is unchanged. Saved bubble-slot edits: {len(edits)}. Corrected preview is rendered in source pixel coordinates ({w}×{h}px); {meta.get('source_dpi')} DPI metadata."
        return str(src),str(outpath),msg
    except Exception as exc:
        return None,None,f'NSAT preview error: {type(exc).__name__}: {exc}'

def _update_nsat_review_value(review_table, run_directory, page_idx, group_id, position, selected_indices, axis):
    if review_table is None: return review_table, 'Review data table is unavailable; image correction was still saved.'
    df=review_table.copy() if isinstance(review_table,pd.DataFrame) else pd.DataFrame(review_table)
    if df.empty: return df,'No student result rows found; saved correction remains in image/audit log.'
    run,manifest,profile,template_gray=_get_nsat_manual_context(run_directory)
    rec=manifest[int(page_idx)]; source=str(rec.get('Source File','')); page=str(rec.get('Page','1'))
    idxs=df.index[(df.get('Source File',pd.Series(index=df.index,dtype=str)).astype(str)==source) & (df.get('Page',pd.Series(index=df.index,dtype=str)).astype(str)==page)]
    if len(idxs)==0: return df,'Saved bubble correction for '+source+' page '+page+'; matching student row not found in table.'
    rowidx=idxs[0]
    group=next((g for g in profile.get('groups',[]) if str(g.get('group_id'))==str(group_id)),None)
    if not group: return df,'Saved image edit, but group metadata was not found to update the table.'
    field=student_detail_column(group)
    if not field: return df,'Saved image edit for '+str(group_id)+' (question grid). This grid is logged in the correction audit; main Student Details sheet contains only six student fields.'
    typ=str(group.get('suggested_type','')).lower(); selected=[int(x) for x in selected_indices]
    symbol='?'
    if len(selected)==1:
        k=selected[0]
        if 'letter_grid' in typ and 1<=k<=26: symbol=chr(64+k)
        elif 'digit_grid' in typ and 1<=k<=10: symbol=str(k-1)
        elif 'single_row_choices' in typ:
            labels=list(group.get('choice_labels',[])); symbol=str(labels[k-1]) if 1<=k<=len(labels) else '?'
        elif 'mcq_grid' in typ:
            labels=list('ABCD') if int(group.get('cols') or 4)==4 else list('ABCDE')
            symbol=labels[k-1] if 1<=k<=len(labels) else '?'
        else: symbol=str(k)
    try: pos=int(position)
    except Exception: pos=1
    if field=='Student Name' and 'letter_grid' in typ:
        # Name table is compressed to hand-written active slots; map selected bubble column back to that slot index.
        H=np.asarray(rec.get('source_to_template'),dtype=np.float64)
        src_gray=cv2.cvtColor(safe_read_image(rec['page_path']),cv2.COLOR_BGR2GRAY)
        aligned=cv2.warpPerspective(src_gray,H,(template_gray.shape[1],template_gray.shape[0]),flags=cv2.INTER_CUBIC,borderValue=255)
        active,_=name_handwriting_slot_positions(aligned,template_gray,group)
        col0=pos-1
        current=str(df.at[rowidx,field] or '')
        if col0 in active:
            char_pos=active.index(col0)
            slots=list(current)
            if len(slots)<len(active): slots += ['?']*(len(active)-len(slots))
            slots[char_pos]=symbol if symbol!='?' else '?'
            df.at[rowidx,field]=''.join(slots).strip()
        else:
            return df, f'Image edit saved, but slot {pos} was not detected as an active handwritten name box; edit the Student Name table cell manually if needed.'
    elif 'digit_grid' in typ and field in df.columns:
        length=int(group.get('cols') or pos); cur=list(str(df.at[rowidx,field] or '').ljust(length,'?')[:length])
        if 1<=pos<=length: cur[pos-1]=symbol
        df.at[rowidx,field]=''.join(cur)
    elif 'single_row_choices' in typ and field in df.columns:
        if symbol!='?':
            grade_map={'5th':'Class V','6th':'Class VI','7th':'Class VII','8th':'Class VIII','9th':'Class IX','10th':'Class X'}
            df.at[rowidx,field]=grade_map.get(symbol,symbol)
        else: df.at[rowidx,field]='?'
    df.at[rowidx,'Review Status']='NEEDS_REVIEW — MANUAL BUBBLE EDIT APPLIED; VERIFY ORIGINAL BEFORE VERIFIED'
    return df, f'Updated {field} for {source} page {page} to {df.at[rowidx,field]}. Row is NOT auto-verified.'


def apply_nsat_manual_bubble_edit(run_directory, page_idx, group_id, position, bubble_label, action, review_table):
    try:
        run,manifest,profile,template_gray=_get_nsat_manual_context(run_directory)
        page_idx=int(page_idx); rec=manifest[page_idx]
        group=next((g for g in profile.get('groups',[]) if str(g.get('group_id'))==str(group_id)),None)
        if not group: raise ValueError('Choose a detected bubble grid/group first.')
        axis,positions,options=_nsat_group_axis_and_labels(group)
        pos=int(position); selected_label=str(bubble_label or '').strip()
        option_index=next((i+1 for i,x in enumerate(options) if str(x)==selected_label),None)
        if action!='Clear all bubbles in this slot' and option_index is None: raise ValueError('Selected bubble option does not belong to this grid.')
        state_path=run/'manual_bubble_edits_nsat.json'; state=_manual_json_read(state_path,{'pages':{}})
        page=state.setdefault('pages',{}).setdefault(str(page_idx),{'Source File':rec.get('Source File'),'Page':rec.get('Page'),'edits':{}})
        edit_key=f'{group_id}::{pos}'; existing=page['edits'].get(edit_key,{})
        chosen=list(existing.get('selected_indices',[]))
        if action=='Clear all bubbles in this slot': chosen=[]
        elif action=='Set selected bubble only': chosen=[int(option_index)]
        elif action=='Toggle selected bubble':
            chosen=[int(x) for x in chosen]
            if int(option_index) in chosen: chosen.remove(int(option_index))
            else: chosen.append(int(option_index))
        else: raise ValueError('Unknown manual action.')
        page['edits'][edit_key]={'group_id':str(group_id),'position':pos,'axis':axis,'selected_indices':chosen,
                                'selected_labels':[options[i-1] for i in chosen if 1<=i<=len(options)],'action':action,
                                'updated_at':datetime.datetime.now().isoformat(timespec='seconds')}
        _manual_json_write(state_path,state)
        review_table,msg_table=_update_nsat_review_value(review_table,run_directory,page_idx,group_id,pos,chosen,axis)
        original,corrected,msg_preview=render_nsat_manual_page(run_directory,str(page_idx))
        msg=f"{action}: {rec.get('Source File')} page {rec.get('Page')}, {group_id}, position {pos}, selected {page['edits'][edit_key]['selected_labels'] or '[cleared]'}. {msg_table} {msg_preview}"
        return original,corrected,msg,review_table
    except Exception as exc:
        return None,None,f'NSAT bubble correction failed: {type(exc).__name__}: {exc}',review_table


# ---------- 180-question bubble-only manual correction ----------
def _get_180_manual_context(run_directory):
    if not run_directory or not Path(str(run_directory)).exists(): raise ValueError('First scan 180Q OMR in Tab 4.')
    run=Path(str(run_directory)); path=run/'bubble180_page_manifest.json'
    if not path.exists(): raise ValueError('180Q page manifest is missing. Rescan a small batch in Tab 4.')
    manifest=json.loads(path.read_text(encoding='utf-8'))
    if not manifest: raise ValueError('No successfully processed 180Q pages are available for correction.')
    _,template_gray=get_bubble_only_reference()
    return run,manifest,template_gray


def load_180_manual_editor(run_directory):
    try:
        run,manifest,template=_get_180_manual_context(run_directory)
        choices=[(f"{i+1}. {r.get('Source File')} — page {r.get('Page')}",str(i)) for i,r in enumerate(manifest)]
        value=choices[0][1]
        original,corrected,msg=render_180_manual_page(run_directory,value)
        return gr.update(choices=choices,value=value),original,corrected,f"Loaded {len(choices)} OMR page(s). {msg}"
    except Exception as exc:
        return gr.update(choices=[],value=None),None,None,f'Could not load 180Q pages: {type(exc).__name__}: {exc}'


def _bubble180_manual_field_controls(field_name):
    field=str(field_name or 'Answer (Q1-Q180)')
    if field=='Registration No. digit column':
        p=[(str(i),f'Digit column {i}') for i in range(1,9)]; o=[str(i) for i in range(10)]
    elif field=='Phase': p=[('1','Phase selection')]; o=[str(i) for i in range(1,7)]
    elif field=='Set': p=[('1','Set selection')]; o=[str(i) for i in range(1,4)]
    elif field=='Class': p=[('1','Class selection')]; o=list(BUBBLE180_CLASS_LABELS)
    else: p=[(str(i),f'Question {i}') for i in range(1,181)]; o=list(BUBBLE180_CHOICE_LABELS)
    return gr.update(choices=[(lab,val) for val,lab in p],value=p[0][0]), gr.update(choices=[(x,x) for x in o],value=o[0]), f'{field}: select a position and choose a bubble; Replace clears the other marks in that slot.'


def _get_180_page_edits(run, page_idx):
    state=_manual_json_read(run/'manual_bubble_edits_180q.json',{'pages':{}})
    return state.get('pages',{}).get(str(page_idx),{}).get('edits',{})


def render_180_manual_page(run_directory, page_value, final_path=None):
    """Render 180Q correction on a source-format copy without warping/resampling the source scan."""
    run,manifest,template_gray=_get_180_manual_context(run_directory)
    idx=int(page_value or 0)
    if idx<0 or idx>=len(manifest): raise ValueError('Selected 180Q OMR page not found.')
    rec=manifest[idx]; src=Path(rec['page_path'])
    if not src.exists(): raise FileNotFoundError('Original OMR page copy missing: '+src.name)
    original=safe_read_image(src); h,w=original.shape[:2]; th,tw=template_gray.shape[:2]
    H=np.asarray(rec.get('source_to_template'),dtype=np.float64)
    H_inv,clean_ref=_make_reference_in_source(template_gray,H,original.shape)
    corrected=original.copy(); geometry=_bubble180_geometry(template_gray); edits=_get_180_page_edits(run,idx)
    field_mapping={'Registration No. digit column':'Registration No.','Phase':'Phase','Set':'Set','Class':'Class','Answer (Q1-Q180)':'Answer'}
    for edit in edits.values():
        field=field_mapping.get(str(edit.get('field_name')))
        if not field: continue
        pos=int(edit.get('position',1)); selected=[str(x) for x in edit.get('selected_values',[])]
        question=pos if field in ('Answer','Registration No.') else 1
        slot=[g for g in geometry if g['field']==field and int(g['question'])==question]
        if not slot: continue
        selected_cells=[g for g in slot if str(g.get('option')) in selected]
        _apply_source_slot_circles(corrected,clean_ref,H_inv,slot,selected_cells,tw,th)
    if final_path:
        target_base=Path(final_path).with_suffix('')
    else:
        target_base=run/'manual_bubble_correction'/'180q_previews'/f'PREVIEW_{idx+1:04d}_{slugify(Path(str(rec.get("Source File","OMR"))).stem)}_p{rec.get("Page",1)}'
    target,meta=_save_corrected_copy_like_source(src,target_base,corrected)
    msg=f"Selected {rec.get('Source File')} page {rec.get('Page')}. Original unchanged; {len(edits)} edited slot(s). Corrected copy source format={meta['output_format']}, dimensions={w}×{h}px, source DPI={meta['source_dpi']}."
    return str(src),str(target),msg

def show_180_manual_page(run_directory,page_value):
    try: return (*render_180_manual_page(run_directory,page_value),)
    except Exception as exc: return None,None,f'180Q preview error: {type(exc).__name__}: {exc}'


def apply_180_manual_bubble_edit(run_directory,page_value,field_name,position,bubble_choice,action,results_table):
    try:
        run,manifest,template=_get_180_manual_context(run_directory); idx=int(page_value or 0); rec=manifest[idx]
        field_name=str(field_name or 'Answer (Q1-Q180)'); pos=int(position or 1); choice=str(bubble_choice or '')
        allowed_field={'Registration No. digit column':([str(i) for i in range(10)],range(1,9)),
                       'Phase':([str(i) for i in range(1,7)],[1]),'Set':([str(i) for i in range(1,4)],[1]),
                       'Class':(list(BUBBLE180_CLASS_LABELS),[1]),'Answer (Q1-Q180)':(list(BUBBLE180_CHOICE_LABELS),range(1,181))}
        if field_name not in allowed_field: raise ValueError('Choose a field type.')
        options,positions=allowed_field[field_name]
        if pos not in list(positions): raise ValueError('Position is outside the selected field range.')
        if action!='Clear this slot' and choice not in options: raise ValueError('Bubble choice is not valid for this field.')
        state_path=run/'manual_bubble_edits_180q.json'; state=_manual_json_read(state_path,{'pages':{}})
        page=state.setdefault('pages',{}).setdefault(str(idx),{'Source File':rec.get('Source File'),'Page':rec.get('Page'),'edits':{}})
        key=f'{field_name}::{pos}'; old=page['edits'].get(key,{})
        selected=list(old.get('selected_values',[]))
        if action=='Clear this slot': selected=[]
        elif action=='Replace with selected bubble': selected=[choice]
        elif action=='Toggle selected bubble':
            if choice in selected: selected.remove(choice)
            else: selected.append(choice)
        else: raise ValueError('Unknown correction action.')
        order=options; selected=sorted(selected,key=lambda x:order.index(x) if x in order else 999)
        page['edits'][key]={'field_name':field_name,'position':pos,'selected_values':selected,'action':action,
                            'updated_at':datetime.datetime.now().isoformat(timespec='seconds')}
        _manual_json_write(state_path,state)
        df=results_table.copy() if isinstance(results_table,pd.DataFrame) else pd.DataFrame(results_table)
        row_index=int(rec.get('result_row_index',idx))
        if not df.empty and 0<=row_index<len(df):
            if field_name=='Registration No. digit column':
                cur=list(str(df.iloc[row_index].get('Registration No.','') or '').ljust(8,'?')[:8])
                cur[pos-1]=selected[0] if len(selected)==1 else '?'
                df.iat[row_index,df.columns.get_loc('Registration No.')]=''.join(cur)
            elif field_name=='Phase': df.iat[row_index,df.columns.get_loc('Phase')]=selected[0] if len(selected)==1 else ('|'.join(selected) if selected else '')
            elif field_name=='Set': df.iat[row_index,df.columns.get_loc('Set')]=selected[0] if len(selected)==1 else ('|'.join(selected) if selected else '')
            elif field_name=='Class': df.iat[row_index,df.columns.get_loc('Class')]=selected[0] if len(selected)==1 else ('|'.join(selected) if selected else '')
            else: df.iat[row_index,df.columns.get_loc(f'Q{pos}')]='|'.join(selected)
        original,corrected,preview_msg=render_180_manual_page(run_directory,str(idx))
        msg=f"{action}: {rec.get('Source File')} page {rec.get('Page')}, {field_name}, position {pos}, selected {selected or '[cleared]'}. Results table updated; original scan unchanged. {preview_msg}"
        return original,corrected,msg,df
    except Exception as exc:
        return None,None,f'180Q correction failed: {type(exc).__name__}: {exc}',results_table


def export_180_manual_results(results_table,run_directory):
    try:
        run,manifest,template=_get_180_manual_context(run_directory)
        df=results_table.copy() if isinstance(results_table,pd.DataFrame) else pd.DataFrame(results_table)
        expected=['Registration No.','Phase','Set','Class']+[f'Q{i}' for i in range(1,181)]
        if df.empty: raise ValueError('Results table is empty; scan a batch before exporting.')
        for col in expected:
            if col not in df.columns: df[col]=''
        df=df[expected].fillna('').astype(str)
        out=run/'manual_final_outputs_180q'; out.mkdir(parents=True,exist_ok=True)
        xlsx=out/'PW_OMR_180Q_Bubble_Only_CORRECTED.xlsx'; csvp=out/'PW_OMR_180Q_Bubble_Only_CORRECTED.csv'
        with pd.ExcelWriter(xlsx,engine='openpyxl') as writer:
            df.to_excel(writer,sheet_name='Bubbled Results',index=False)
            ws=writer.sheets['Bubbled Results']; ws.freeze_panes='E2'; ws.sheet_view.showGridLines=False; ws.auto_filter.ref=ws.dimensions
            ws.page_setup.orientation='landscape'; ws.page_setup.paperSize=ws.PAPERSIZE_A3; ws.page_setup.fitToWidth=1; ws.page_setup.fitToHeight=0; ws.sheet_properties.pageSetUpPr.fitToPage=True
            ws.column_dimensions['A'].width=18; ws.column_dimensions['B'].width=10; ws.column_dimensions['C'].width=10; ws.column_dimensions['D'].width=14
            for c in range(5,185): ws.column_dimensions[ws.cell(1,c).column_letter].width=5
            from openpyxl.styles import Font,PatternFill,Alignment
            for cell in ws[1]: cell.fill=PatternFill('solid',fgColor='1F4E78'); cell.font=Font(bold=True,color='FFFFFF'); cell.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
            ws.row_dimensions[1].height=30
            for rr in ws.iter_rows(min_row=2):
                for cell in rr:
                    cell.alignment=Alignment(horizontal='center',vertical='center')
                    if cell.column<=4: cell.number_format='@'
        df.to_csv(csvp,index=False,encoding='utf-8-sig')
        state=_manual_json_read(run/'manual_bubble_edits_180q.json',{'pages':{}})
        edits_csv=out/'Manual_Bubble_Corrections_180Q.csv'; audit=[]; corrected_paths=[]
        for token,pinfo in state.get('pages',{}).items():
            for ed in pinfo.get('edits',{}).values(): audit.append({'Source File':pinfo.get('Source File'),'Page':pinfo.get('Page'),**ed})
            target=out/'corrected_omr_images'/f"CORRECTED_{int(token)+1:04d}_{slugify(Path(str(pinfo.get('Source File','OMR'))).stem)}_p{pinfo.get('Page',1)}"
            try:
                _,p,msg=render_180_manual_page(run_directory,token,final_path=target); corrected_paths.append(p)
            except Exception as exc:
                audit.append({'Source File':pinfo.get('Source File'),'Page':pinfo.get('Page'),'error':f'{type(exc).__name__}: {exc}'})
        pd.DataFrame(audit).to_csv(edits_csv,index=False,encoding='utf-8-sig')
        zipp=out/'PW_OMR_180Q_Corrected_Results_and_Images.zip'
        diag=run/'outputs'/'PW_OMR_180Q_Diagnostics.csv'; details=run/'outputs'/'PW_OMR_180Q_Bubble_Details.csv'
        with zipfile.ZipFile(zipp,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(xlsx,xlsx.name); z.write(csvp,csvp.name); z.write(edits_csv,edits_csv.name)
            for p in [diag,details]:
                if p.exists(): z.write(p,p.name)
            for p in corrected_paths:
                if Path(p).exists(): z.write(p,f'corrected_omr_images/{Path(p).name}')
        xdl=stage_download_file(xlsx,'xlsx'); cdl=stage_download_file(csvp,'file'); zdl=stage_download_file(zipp,'zip')
        image_zip_dl=None; image_zip_note='No corrected image ZIP created.'
        if corrected_paths:
            image_zip_dl,image_zip_note=_build_images_only_zip(corrected_paths,out/'PW_OMR_180Q_All_Corrected_Images.zip','neet180_corrected')
        return xdl,cdl,zdl,image_zip_dl,f'Manual export completed: {len(df)} page result row(s), {len(audit)} correction audit row(s), {len(corrected_paths)} separate corrected page image(s). Original scans unchanged. Excel/ZIP validated. {image_zip_note}'
    except Exception as exc:
        return None,None,None,None,f'MANUAL 180Q EXPORT FAILED: {type(exc).__name__}: {exc}\n{traceback.format_exc()[-1200:]}'


# ================== v14 EASY TAP-TO-CORRECT WORKFLOW ==================
# Mobile-first workflow: search the OMR, tap its field/question on the original image,
# type the correct value, and replace that slot's bubbles in a separate corrected copy.
# Original scan files are read-only. All edits are stored per page and survive rerenders.

def _quick_xy_from_event(evt, image_path):
    idx=getattr(evt,'index',None)
    if isinstance(idx,dict):
        x=idx.get('x',idx.get(0)); y=idx.get('y',idx.get(1))
    elif isinstance(idx,(list,tuple)) and len(idx)>=2:
        x,y=idx[0],idx[1]
    else:
        raise ValueError('Image tap coordinates were not received. Tap directly on the displayed OMR image.')
    x=float(x); y=float(y)
    img=safe_read_image(image_path); h,w=img.shape[:2]
    # Some frontends return normalized coordinates; convert those when clearly applicable.
    if 0 <= x <= 1.0 and 0 <= y <= 1.0:
        x*=w; y*=h
    # Clamp tiny edge overshoots from mobile click handling.
    x=min(max(0.0,x),float(w-1)); y=min(max(0.0,y),float(h-1))
    return x,y


def _source_click_to_template_xy(rec, x, y, tw, th):
    H=np.asarray(rec.get('source_to_template'),dtype=np.float64)
    if H.shape!=(3,3) or not np.isfinite(H).all():
        raise ValueError('Saved alignment matrix is missing or invalid. Re-scan this OMR batch first.')
    pt=np.array([[[float(x),float(y)]]],dtype=np.float32)
    q=cv2.perspectiveTransform(pt,H)[0,0]
    tx=float(q[0]); ty=float(q[1])
    if not np.isfinite([tx,ty]).all(): raise ValueError('Tapped position could not be aligned to template.')
    return min(max(0.0,tx),float(tw-1)),min(max(0.0,ty),float(th-1))


def _nsat_group_hit(profile, tx, ty):
    """Map a tap on a printed field/grid to its bubble group; accept handwritten text row above bubbles."""
    matches=[]
    for g in profile.get('groups',[]):
        try:
            x0,y0,x1,y1=[float(v) for v in g.get('bbox',[0,0,0,0])]
        except Exception:
            continue
        typ=str(g.get('suggested_type','')).lower()
        # Some one-row choice profiles have a zero-height bounding box; expand around centres.
        if abs(y1-y0)<8:
            ys=[float(v) for v in g.get('y_centers',[])]; xs=[float(v) for v in g.get('x_centers',[])]
            cy=float(np.median(ys)) if ys else y0
            if xs: x0=min(xs)-22; x1=max(xs)+22
            y0=cy-24; y1=cy+24
        padx=25 if 'mcq_grid' in typ else 18
        top_pad=68 if ('letter_grid' in typ or 'digit_grid' in typ) else 20
        bottom_pad=18 if ('letter_grid' in typ or 'digit_grid' in typ) else 20
        ex0=x0-padx; ex1=x1+padx; ey0=y0-top_pad; ey1=y1+bottom_pad
        if ex0<=tx<=ex1 and ey0<=ty<=ey1:
            # Prefer a tap inside the actual grid. If in handwritten field area, distance is small.
            dx=max(x0-tx,0,tx-x1); dy=max(y0-ty,0,ty-y1)
            dist=dx+dy
            area=max(1.0,(x1-x0+1)*(y1-y0+1))
            matches.append((dist,area,g))
    if not matches:
        return None,None
    matches.sort(key=lambda x:(x[0],x[1]))
    group=matches[0][2]
    typ=str(group.get('suggested_type','')).lower()
    target={'group_id':str(group.get('group_id')),'mode':'full_field',
            'field_label':str(group.get('label') or group.get('label_hint') or group.get('group_id')),
            'position':None,'question_label':None}
    if 'mcq_grid' in typ:
        ys=[float(v) for v in group.get('y_centers',[]) if np.isfinite(float(v))]
        if ys:
            loc=min(range(len(ys)),key=lambda i:abs(ys[i]-ty))
            target['mode']='single_slot'; target['position']=loc+1
            try: qstart=int(float(group.get('question_start') or 1))
            except Exception: qstart=1
            target['question_label']=f"Question {qstart+loc}"
            target['field_label']=target['question_label']
        else:
            return None,None
    return target,group


def tap_to_pick_nsat_field(run_directory, page_value, evt: gr.SelectData):
    try:
        run,manifest,profile,template_gray=_get_nsat_manual_context(run_directory)
        idx=int(page_value or 0); rec=manifest[idx]
        src=str(rec['page_path']); x,y=_quick_xy_from_event(evt,src)
        th,tw=template_gray.shape[:2]
        tx,ty=_source_click_to_template_xy(rec,x,y,tw,th)
        target,group=_nsat_group_hit(profile,tx,ty)
        if target is None:
            return {},'Tap landed outside a mapped field. Tap the name box, digit box, grade row, or the question row on the original OMR.',gr.update(value='',placeholder='Tap a field on the OMR first')
        typ=str(group.get('suggested_type','')).lower()
        if target['mode']=='single_slot': placeholder='Type one answer: A, B, C, or D (or CLEAR)'
        elif 'letter_grid' in typ: placeholder='Type the correct name; each letter or space occupies one slot'
        elif 'digit_grid' in typ: placeholder=f"Type digits; space skips one slot (or CLEAR)"
        elif 'single_row_choices' in typ: placeholder='Type grade, e.g. 10th NEET, 12th JEE, Class X (or CLEAR)'
        else: placeholder='Type the corrected value'
        hint=f"Selected: {target['field_label']} — {target['group_id']}. Now type the correct value below and press Apply. Other bubbles in each edited slot will be removed automatically."
        return target,hint,gr.update(value='',placeholder=placeholder)
    except Exception as exc:
        return {},f'Tap selection error: {type(exc).__name__}: {exc}',gr.update(value='',placeholder='Tap a field on the OMR first')


def _nsat_manual_store_field_edits(run, manifest, page_idx, group, edits_by_position):
    rec=manifest[page_idx]
    state_path=run/'manual_bubble_edits_nsat.json'
    state=_manual_json_read(state_path,{'pages':{}})
    page=state.setdefault('pages',{}).setdefault(str(page_idx),{'Source File':rec.get('Source File'),'Page':rec.get('Page'),'edits':{}})
    for position, selected_indices, axis in edits_by_position:
        key=f"{group.get('group_id')}::{int(position)}"
        options= _nsat_group_axis_and_labels(group)[2]
        if axis=='col': labels=[options[i-1] for i in selected_indices if 1<=i<=len(options)]
        elif axis=='row': labels=[options[i-1] for i in selected_indices if 1<=i<=len(options)]
        else: labels=[options[i-1] for i in selected_indices if 1<=i<=len(options)]
        page['edits'][key]={'group_id':str(group.get('group_id')),'position':int(position),'axis':axis,
                            'selected_indices':[int(i) for i in selected_indices],
                            'selected_labels':labels,'action':'TAP_TO_TYPE_REPLACE_SLOT',
                            'updated_at':datetime.datetime.now().isoformat(timespec='seconds')}
    _manual_json_write(state_path,state)
    return state


def _nsat_find_review_row(df, rec):
    if not isinstance(df,pd.DataFrame) or df.empty: return None
    if 'Source File' not in df.columns: return None
    m=(df['Source File'].astype(str)==str(rec.get('Source File','')))
    if 'Page' in df.columns: m &= (df['Page'].astype(str)==str(rec.get('Page','1')))
    idx=list(df.index[m])
    return idx[0] if idx else None


def _slot_preview_html(value, target, system='nsat'):
    """Render fixed-width, faint OMR-like boxes; spaces occupy one blank slot."""
    import html as _html
    raw=str(value or '').replace('\r','').replace('\n','').upper()
    if system=='nsat':
        gid=str((target or {}).get('group_id',''))
        if gid=='P01': count=17; label='NAME — each character/space uses one bubble column'
        elif gid=='P02': count=10; label='MOBILE — 10 slots'
        elif gid=='P03': count=5; label='EXAM CODE — 5 slots'
        elif gid=='P04': count=10; label='PARENT/GUARDIAN MOBILE — 10 slots'
        elif gid=='P05': count=6; label='DOB DDMMYY — 6 slots'
        elif gid=='P06': count=1; label='GRADE — one selected option'
        elif (target or {}).get('mode')=='single_slot': count=1; label=str((target or {}).get('question_label') or 'ANSWER')
        else: count=0; label='Tap a mapped field on the original OMR first'
    else:
        field=str((target or {}).get('field_name',''))
        if field=='Registration No. digit column': count=8; label='REGISTRATION NUMBER — 8 slots'
        elif field in ('Phase','Set','Class'): count=1; label=field.upper()+' — one selected option'
        elif field=='Answer (Q1-Q180)': count=1; label='ANSWER — one selected option'
        else: count=0; label='Tap a mapped field on the original OMR first'
    if not count:
        return '<div style="padding:8px;color:#666;border:1px solid #ddd;border-radius:6px">'+_html.escape(label)+'</div>'
    shown=list(raw[:count])
    shown += ['']*max(0,count-len(shown))
    # Keep space visible as a blank cell; use subtle borders and a small slot index.
    cells=[]
    for i,ch in enumerate(shown,1):
        display=_html.escape(ch) if ch!=' ' else '&nbsp;'
        cells.append(f'<div style="flex:0 0 30px;min-width:30px;height:42px;border:1px solid #b8c2cc;border-radius:3px;background:#fbfcfd;display:flex;flex-direction:column;align-items:center;justify-content:center;box-sizing:border-box"><span style="font:600 16px/20px monospace;color:#202833">{display}</span><span style="font:9px/11px sans-serif;color:#9aa3ad">{i}</span></div>')
    msg='Space = skip one slot; Backspace = correct. Blank boxes clear those bubble columns when Apply is pressed.'
    return '<div style="margin:6px 0 10px"><div style="font:600 12px sans-serif;color:#4b5563;margin-bottom:5px">'+_html.escape(label)+'</div><div style="display:flex;gap:3px;overflow-x:auto;padding:3px 0 7px">'+''.join(cells)+'</div><div style="font:11px sans-serif;color:#6b7280">'+_html.escape(msg)+'</div></div>'


def render_nsat_slot_preview(value, target):
    return _slot_preview_html(value, target, 'nsat')


def render_180_slot_preview(value, target):
    return _slot_preview_html(value, target, '180')


def apply_typed_nsat_value(run_directory,page_idx,target,value,review_table):
    original=corrected=None
    try:
        run,manifest,profile,template_gray=_get_nsat_manual_context(run_directory)
        idx=int(page_idx or 0); rec=manifest[idx]
        if not isinstance(target,dict) or not target.get('group_id'):
            raise ValueError('Pehle original OMR par kisi field/question ko tap karein.')
        group=next((g for g in profile.get('groups',[]) if str(g.get('group_id'))==str(target.get('group_id'))),None)
        if not group: raise ValueError('Tapped field profile me nahi mila; template dobara register karein.')
        raw=str(value or '').replace('\r','').replace('\n',''); typ=str(group.get('suggested_type','')).lower()
        if not raw.strip(): raise ValueError('Pehle corrected value type karein. Field clear karne ke liye CLEAR likhein.')
        clear=raw.strip().upper()=='CLEAR'
        axis,positions,options=_nsat_group_axis_and_labels(group)
        edits=[]; table_value=''
        if 'mcq_grid' in typ:
            if target.get('mode')!='single_slot' or not target.get('position'):
                raise ValueError('Question grid ke specific row par tap karein.')
            pos=int(target['position']); answers=[] if clear else [raw.upper()]
            if answers and answers[0] not in list('ABCD')[:int(group.get('cols') or 4)]:
                raise ValueError('MCQ answer A, B, C, D me se ho ya CLEAR likhein.')
            idxs=[] if clear else [list('ABCD').index(answers[0])+1]
            edits.append((pos,idxs,'row'))
            field_for_msg=f"{target.get('question_label') or 'Question'}"
        elif 'letter_grid' in typ:
            # Fixed slots: a literal space consumes one position and clears that whole bubble column.
            name=raw.upper()
            cols=int(group.get('cols') or 17)
            if not clear and not re.fullmatch(r'[A-Z ]+',name): raise ValueError('Name me sirf A-Z letters aur spaces use karein.')
            if len(name)>cols: raise ValueError(f"Name/slot text is too long for this grid ({cols} slots max, including spaces). Remove extra characters or spaces.")
            for pos in range(1,cols+1):
                ch=name[pos-1] if not clear and pos<=len(name) else ''
                selected=[] if not ch or ch==' ' else [ord(ch)-64]
                edits.append((pos,selected,'col'))
            table_value='' if clear else name.rstrip()
            field_for_msg='Student Name'
        elif 'digit_grid' in typ:
            # Spaces are intentional empty slots; never strip/remove them because that shifts digits.
            digits=raw
            cols=int(group.get('cols') or 0)
            if not clear and not re.fullmatch(r'[0-9 ]+',digits): raise ValueError('Digit field me sirf 0-9 aur space use karein; space ek bubble slot khaali chhodta hai.')
            if not clear and len(digits)>cols: raise ValueError(f"{group.get('label') or 'This field'} has {cols} slots; you entered {len(digits)} characters including spaces.")
            for pos in range(1,cols+1):
                ch=digits[pos-1] if not clear and pos<=len(digits) else ''
                edits.append((pos,[] if not ch or ch==' ' else [int(ch)+1],'col'))
            table_value='' if clear else digits.rstrip()
            field_for_msg=str(group.get('label') or 'Digit field')
        elif 'single_row_choices' in typ:
            labels=list(group.get('choice_labels',[])) or options
            aliases={'5TH':'CLASS V','6TH':'CLASS VI','7TH':'CLASS VII','8TH':'CLASS VIII','9TH':'CLASS IX','10TH':'CLASS X',
                     '11TH':'11TH JEE','12TH':'12TH JEE','11TH NEET':'11TH NEET','12TH NEET':'12TH NEET'}
            val=' '.join(raw.upper().split())
            if val in aliases: val=aliases[val]
            lookup={str(x).upper():i+1 for i,x in enumerate(labels)}
            if val not in lookup and not clear:
                raise ValueError('Grade value not recognized. Use one of: '+', '.join(labels)+'.')
            edits.append((1,[] if clear else [lookup[val]],'single'))
            table_value='' if clear else labels[lookup[val]-1]
            field_for_msg=str(group.get('label') or 'Grade')
        else:
            raise ValueError('This grid type is not supported by typed correction yet.')
        _nsat_manual_store_field_edits(run,manifest,idx,group,edits)
        df=review_table.copy() if isinstance(review_table,pd.DataFrame) else pd.DataFrame(review_table)
        rowidx=_nsat_find_review_row(df,rec)
        if rowidx is not None:
            target_field=student_detail_column(group)
            if target_field and target_field in df.columns and target['mode']!='single_slot':
                df.at[rowidx,target_field]=table_value
            if 'Review Status' in df.columns: df.at[rowidx,'Review Status']='NEEDS_REVIEW — TYPED BUBBLE CORRECTION APPLIED; VERIFY'
        original,corrected,msg_preview=render_nsat_manual_page(run_directory,str(idx))
        msg=f"Saved correction for {rec.get('Source File')} page {rec.get('Page')}: {field_for_msg} = {table_value or ('[cleared]' if clear else raw)}. Fixed-slot mode: each typed character uses one position; each space skips/clears exactly one slot; all competing bubbles are removed. Original scan unchanged. {msg_preview}"
        return original,corrected,msg,df
    except Exception as exc:
        try:
            if run_directory and page_idx is not None: original,corrected,_=render_nsat_manual_page(run_directory,str(page_idx))
        except Exception: pass
        return original,corrected,f'Typed NSAT correction failed: {type(exc).__name__}: {exc}',review_table


def find_nsat_page_by_serial(run_directory, query, review_table):
    try:
        run,manifest,profile,_=_get_nsat_manual_context(run_directory)
        choices=[(_nsat_page_label(r,i),str(i)) for i,r in enumerate(manifest)]
        q=str(query or '').strip().casefold()
        df=review_table.copy() if isinstance(review_table,pd.DataFrame) else pd.DataFrame(review_table)
        if not q:
            if not choices: return gr.update(choices=[],value=None),None,None,'No scanned NSAT pages available.'
            idx=0; original,corrected,msg=render_nsat_manual_page(run_directory,str(idx))
            return gr.update(choices=choices,value=str(idx)),original,corrected,'Enter OMR serial/file name, page number, mobile, Exam Code or another visible result field. '+msg
        hits=[]
        for i,rec in enumerate(manifest):
            bits=[str(i+1),str(rec.get('Source File','')),str(rec.get('Page',''))]
            rowidx=_nsat_find_review_row(df,rec)
            if rowidx is not None: bits += [str(df.at[rowidx,c]) for c in df.columns if c!='Review Status']
            if q in ' '.join(bits).casefold(): hits.append(i)
        if not hits:
            original,corrected,msg=render_nsat_manual_page(run_directory,'0')
            return gr.update(choices=choices,value=None),original,corrected,f'No OMR matched "{query}". Try the file serial/name or visible student data. {msg}'
        idx=hits[0]; original,corrected,msg=render_nsat_manual_page(run_directory,str(idx))
        return gr.update(choices=choices,value=str(idx)),original,corrected,f'Found {len(hits)} matching OMR page(s). Selected: {_nsat_page_label(manifest[idx],idx)}. {msg}'
    except Exception as exc:
        return gr.update(choices=[],value=None),None,None,f'Search failed: {type(exc).__name__}: {exc}'


def _pick_180_target_at_template_xy(tx,ty):
    # Coords match the built-in 1489x2105 template. Text field bands immediately above grids are included.
    if 110<=tx<=365 and 185<=ty<=480:
        return {'field_name':'Registration No. digit column','mode':'full_field','position':None,'field_label':'Registration No.'}, 'Registration No.'
    if 120<=tx<=300 and 525<=ty<=610:
        return {'field_name':'Phase','mode':'full_field','position':None,'field_label':'Phase'}, 'Phase'
    if 305<=tx<=415 and 525<=ty<=610:
        return {'field_name':'Set','mode':'full_field','position':None,'field_label':'Set'}, 'Set'
    if 150<=tx<=430 and 645<=ty<=745:
        return {'field_name':'Class','mode':'full_field','position':None,'field_label':'Class'}, 'Class'
    blocks=[(560,700),(760,905),(960,1110),(1160,1310)]
    for block,(x0,x1) in enumerate(blocks):
        if x0<=tx<=x1 and 485<=ty<=1680:
            pos0=min(range(len(BUBBLE180_Q_Y)),key=lambda i:abs(BUBBLE180_Q_Y[i]-ty))
            q=block*45+pos0+1
            return {'field_name':'Answer (Q1-Q180)','mode':'single_slot','position':q,'field_label':f'Question {q}'},f'Question {q}'
    return None,None


def tap_to_pick_180_field(run_directory,page_value,evt: gr.SelectData):
    try:
        run,manifest,template_gray=_get_180_manual_context(run_directory)
        idx=int(page_value or 0); rec=manifest[idx]; src=str(rec['page_path'])
        x,y=_quick_xy_from_event(evt,src); th,tw=template_gray.shape[:2]
        tx,ty=_source_click_to_template_xy(rec,x,y,tw,th)
        target,label=_pick_180_target_at_template_xy(tx,ty)
        if target is None: return {},'Tap the Registration No., Phase, Set, Class or one Q1-Q180 row on the original OMR.',gr.update(value='',placeholder='Tap a field/question on the OMR first')
        if target['field_name']=='Answer (Q1-Q180)': placeholder='Type A, B, C, D or CLEAR for this question'
        elif target['field_name']=='Registration No. digit column': placeholder='Type up to 8 digits; space skips one slot'
        elif target['field_name']=='Phase': placeholder='Type phase 1–6 or CLEAR'
        elif target['field_name']=='Set': placeholder='Type set 1–3 or CLEAR'
        else: placeholder='Type 11th, 12th or Dropper (or CLEAR)'
        return target,f"Selected {label}. Type the correct value below; existing competing marks in that slot/field will be removed.",gr.update(value='',placeholder=placeholder)
    except Exception as exc:
        return {},f'Tap selection error: {type(exc).__name__}: {exc}',gr.update(value='',placeholder='Tap a field/question on the OMR first')


def _update_180_df_value(df, rec, field, pos, value):
    if not isinstance(df,pd.DataFrame): df=pd.DataFrame(df)
    rowidx=int(rec.get('result_row_index',0))
    if df.empty or rowidx<0 or rowidx>=len(df): return df
    if field in ('Registration No. digit column','Registration No.'): df.at[df.index[rowidx],'Registration No.']=value
    elif field in ('Phase','Set','Class'): df.at[df.index[rowidx],field]=value
    elif field=='Answer (Q1-Q180)': df.at[df.index[rowidx],f'Q{int(pos)}']=value
    return df


def apply_typed_180_value(run_directory,page_value,target,value,results_table):
    original=corrected=None
    try:
        run,manifest,template=_get_180_manual_context(run_directory)
        idx=int(page_value or 0); rec=manifest[idx]
        if not isinstance(target,dict) or not target.get('field_name'): raise ValueError('Pehle original NEET/180Q OMR par field/question tap karein.')
        raw=str(value or '').replace('\r','').replace('\n',''); clear=raw.strip().upper()=='CLEAR'
        field=str(target['field_name']); edits=[]; final_value=''
        state_path=run/'manual_bubble_edits_180q.json'; state=_manual_json_read(state_path,{'pages':{}})
        page=state.setdefault('pages',{}).setdefault(str(idx),{'Source File':rec.get('Source File'),'Page':rec.get('Page'),'edits':{}})
        if field=='Registration No. digit column':
            digits=raw
            if not clear and not re.fullmatch(r'[0-9 ]+',digits): raise ValueError('Registration No. me sirf 0-9 aur space use karein; space ek slot khaali chhodta hai.')
            if not clear and len(digits)>8: raise ValueError(f'Registration No. has 8 slots; you entered {len(digits)} characters including spaces.')
            for pos in range(1,9):
                ch=digits[pos-1] if not clear and pos<=len(digits) else ''
                selected=[] if not ch or ch==' ' else [ch]
                key=f'{field}::{pos}'; page['edits'][key]={'field_name':field,'position':pos,'selected_values':selected,'action':'TAP_TO_TYPE_REPLACE_FIELD','updated_at':datetime.datetime.now().isoformat(timespec='seconds')}
            final_value='' if clear else digits.rstrip()
            table_field='Registration No.'; table_pos=1
        elif field=='Phase':
            if not clear and (not raw.isdigit() or raw not in [str(i) for i in range(1,7)]): raise ValueError('Phase must be a single digit 1–6 or CLEAR.')
            selected=[] if clear else [raw]
            page['edits'][f'{field}::1']={'field_name':field,'position':1,'selected_values':selected,'action':'TAP_TO_TYPE_REPLACE_FIELD','updated_at':datetime.datetime.now().isoformat(timespec='seconds')}
            final_value='' if clear else raw; table_field=field; table_pos=1
        elif field=='Set':
            if not clear and raw not in ('1','2','3'): raise ValueError('Set must be 1, 2, or 3, or CLEAR.')
            selected=[] if clear else [raw]
            page['edits'][f'{field}::1']={'field_name':field,'position':1,'selected_values':selected,'action':'TAP_TO_TYPE_REPLACE_FIELD','updated_at':datetime.datetime.now().isoformat(timespec='seconds')}
            final_value='' if clear else raw; table_field=field; table_pos=1
        elif field=='Class':
            aliases={'11TH':'11th','11':'11th','12TH':'12th','12':'12th','DROPPER':'Dropper'}
            val=aliases.get(raw.upper(),raw)
            if not clear and val not in BUBBLE180_CLASS_LABELS: raise ValueError('Class must be 11th, 12th, or Dropper (or CLEAR).')
            selected=[] if clear else [val]
            page['edits'][f'{field}::1']={'field_name':field,'position':1,'selected_values':selected,'action':'TAP_TO_TYPE_REPLACE_FIELD','updated_at':datetime.datetime.now().isoformat(timespec='seconds')}
            final_value='' if clear else val; table_field=field; table_pos=1
        elif field=='Answer (Q1-Q180)':
            if target.get('mode')!='single_slot': raise ValueError('Tap directly on one question row, then type A/B/C/D.')
            pos=int(target.get('position') or 0)
            if not 1<=pos<=180: raise ValueError('Question position outside Q1-Q180.')
            val=raw.upper()
            if not clear and val not in BUBBLE180_CHOICE_LABELS: raise ValueError('Answer must be A, B, C, or D (or CLEAR).')
            selected=[] if clear else [val]
            page['edits'][f'{field}::{pos}']={'field_name':field,'position':pos,'selected_values':selected,'action':'TAP_TO_TYPE_REPLACE_SLOT','updated_at':datetime.datetime.now().isoformat(timespec='seconds')}
            final_value='' if clear else val; table_field=field; table_pos=pos
        else: raise ValueError('Unsupported tapped field.')
        _manual_json_write(state_path,state)
        df=results_table.copy() if isinstance(results_table,pd.DataFrame) else pd.DataFrame(results_table)
        df=_update_180_df_value(df,rec,table_field,table_pos,final_value)
        original,corrected,msg_preview=render_180_manual_page(run_directory,str(idx))
        return original,corrected,f"Saved typed correction for {rec.get('Source File')} page {rec.get('Page')}: {table_field} = {final_value or '[cleared]'}. Other bubbles in edited slots are cleared, and only your typed value is filled. Original scan unchanged. {msg_preview}",df
    except Exception as exc:
        try:
            if run_directory and page_value is not None: original,corrected,_=render_180_manual_page(run_directory,str(page_value))
        except Exception: pass
        return original,corrected,f'Typed 180Q correction failed: {type(exc).__name__}: {exc}',results_table


def find_180_page_by_serial(run_directory, query, results_table):
    try:
        run,manifest,template=_get_180_manual_context(run_directory)
        choices=[(f"{i+1}. {r.get('Source File')} — page {r.get('Page')}",str(i)) for i,r in enumerate(manifest)]
        q=str(query or '').strip().casefold(); df=results_table.copy() if isinstance(results_table,pd.DataFrame) else pd.DataFrame(results_table)
        if not q:
            if not choices: return gr.update(choices=[],value=None),None,None,'No scanned 180Q pages available.'
            original,corrected,msg=render_180_manual_page(run_directory,choices[0][1])
            return gr.update(choices=choices,value=choices[0][1]),original,corrected,'Enter source filename/serial or Registration No. '+msg
        hits=[]
        for i,rec in enumerate(manifest):
            bits=[str(i+1),str(rec.get('Source File','')),str(rec.get('Page',''))]
            ridx=int(rec.get('result_row_index',i))
            if not df.empty and 0<=ridx<len(df): bits += [str(v) for v in df.iloc[ridx].tolist()]
            if q in ' '.join(bits).casefold(): hits.append(i)
        if not hits:
            original,corrected,msg=render_180_manual_page(run_directory,'0')
            return gr.update(choices=choices,value=None),original,corrected,f'No OMR found for "{query}". Check filename serial or registration number. {msg}'
        i=hits[0]; original,corrected,msg=render_180_manual_page(run_directory,str(i))
        return gr.update(choices=choices,value=str(i)),original,corrected,f'Found {len(hits)} match(es): {choices[i][0]}. {msg}'
    except Exception as exc:
        return gr.update(choices=[],value=None),None,None,f'Search failed: {type(exc).__name__}: {exc}'


# ---------- UI ----------
with gr.Blocks(title='PWNSAT OMR Scanner v16 — Fixed Slot Typing + Bubble Correction') as demo:
    profile_state=gr.State({})
    gr.Markdown("""# PWNSAT OMR Scanner v16 — Fixed slot typing + bubble correction
**Step 1:** register the blank template. **Step 2:** detect bubbles and scan the batch. **Step 3:** tap a field/question on the displayed original OMR, type the correct value, and apply one-click bubble replacement. Then review/edit data and export Excel/ZIP.

**Bubble-first mode is the default.** Handwritten NAME OCR is optional and stays as a comparison/suggestion only; it never automatically replaces a bubble-derived name. Turn on the OCR checkbox only when you want to compare handwriting against the detected bubbles. If a bubble is unclear, it remains marked for manual review rather than being guessed. Original OMR files are never overwritten.

The PWNSAT mapper supports the name-letter grid, mobile numbers, exam code, parent mobile, DOB, grade, and MCQs. Tab 3 includes visual manual bubble correction for every detected group. Tab 4 adds a separate bubble-only reader for the supplied PW 180Q sheet: Registration No. bubbles, Phase, Set, Class, and Q1-Q180 only; it does not run name or handwriting OCR. Output Excel contains the six student-detail fields. The OCR option can use TrOCR when available and a bounded Tesseract fallback otherwise. Every uncertain or conflicting result remains editable and unverified until checked by a person. No system can guarantee 100% OCR/bubble accuracy for every scan; review flagged rows before official use.""")
    with gr.Tab('1. Template Scan / Save'):
        with gr.Row():
            template_file=gr.File(label='Blank/reference OMR template (JPG/PNG/TIFF/PDF)',file_types=['.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp','.pdf'],type='filepath')
            profile_name=gr.Textbox(label='Profile name (e.g., Exam A OMR)',value='My OMR Template')
        analyze_btn=gr.Button('Analyze Template Automatically',variant='primary')
        template_status=gr.Textbox(label='Template analysis status',lines=3)
        with gr.Row():
            template_preview=gr.Image(label='Detected template preview (orange circles; green boxes = detected groups)',type='filepath')
            profile_bundle_out=gr.File(label='Download / keep reusable Template Profile ZIP')
        map_table=gr.Dataframe(label='Detected grid groups — editable once per template',headers=['Group ID','Detected label hint','Your field/group label','Suggested type','Rows','Columns','Detected bubbles','Grid fill ratio','Question start (optional)'],interactive=True,wrap=True)
        save_map_btn=gr.Button('Save Group Labels / Types to Profile')
        mapping_status=gr.Textbox(label='Mapping status',lines=2)
        analyze_btn.click(analyze_template, inputs=[template_file,profile_name], outputs=[profile_state,template_preview,map_table,template_status,profile_bundle_out])
        save_map_btn.click(save_group_mapping, inputs=[profile_state,map_table], outputs=[profile_state,map_table,mapping_status,profile_bundle_out])
    with gr.Tab('2. Automatic Scan'):
        gr.Markdown('Use the profile created in Tab 1 or upload a saved Template Profile ZIP. Upload multiple OMR images/PDFs of the same design. The scan shows progress and makes a preview for every page.')
        profile_bundle_in=gr.File(label='Optional: saved Template Profile ZIP',file_types=['.zip'],type='filepath')
        omr_files=gr.File(label='Upload student OMR files (multiple JPG/PNG/TIFF/PDF)',file_count='multiple',file_types=['.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp','.pdf'],type='filepath')
        fill_sensitivity=gr.Slider(minimum=0.0,maximum=1.0,value=0.70,step=0.05,label='Faint-bubble sensitivity (higher may also increase false positives)')
        enable_name_ocr=gr.Checkbox(label='Optional: read handwritten NAME text with OCR and compare with bubbles (slower; bubble result remains primary)',value=False)
        use_ai_name_ocr=gr.Checkbox(label='Use AI handwriting recognizer (TrOCR) for NAME — first run downloads model; if unavailable, Tesseract fallback is used',value=True)
        run_text_crosscheck=gr.Checkbox(label='Optional: OCR handwritten numeric fields for comparison (slower)',value=False)
        scan_btn=gr.Button('Scan OMR Batch',variant='primary')
        scan_status=gr.Textbox(label='Processing status / progress',lines=5)
        student_table=gr.Dataframe(label='Student Details — six result fields (A4 Landscape export)',headers=STUDENT_DETAIL_COLUMNS,wrap=True,interactive=False)
        diagnostics_table=gr.Dataframe(label='Diagnostics — per-file status and exact processing errors',headers=['Source File','Page','Outcome','Alignment','Alignment Confidence','Filled bubbles','Ambiguous bubbles','Error'],wrap=True,interactive=False)
        comparison_table=gr.Dataframe(label='Field OCR vs Bubble Review — OCR is a proposal, not automatic truth',wrap=True,interactive=False)
        review_gallery=gr.Gallery(label='All annotated OMR previews (red filled / amber uncertain / green empty); verify name text in Step 3',columns=3,height=500)
        excel_out=gr.DownloadButton(label='Download Student Details Excel (A4 Landscape)', value=None, variant='primary', size='lg')
        bundle_out=gr.DownloadButton(label='Download Results + Review Report ZIP', value=None, variant='secondary', size='lg')
        run_directory_state=gr.State('')
    with gr.Tab('3. Preview / Manual Correction'):
        gr.Markdown("""## Section 1 — Original OMR (read-only)
The thumbnails below are made from the original uploaded/extracted pages. They are not annotated or edited. Original source files stay unchanged throughout the workflow.""")
        original_review_gallery=gr.Gallery(label='Original OMR pages — source copy, unchanged',columns=2,height=420,object_fit='contain')
        gr.Markdown("""## Bubble-first result review
The editable table below contains the bubble-derived result. OCR, when enabled in Step 2, is a comparison/suggestion only and does not automatically replace the bubble reading. Correct uncertain values manually against the original OMR, then mark rows VERIFIED only after checking them.""")
        manual_review_table=gr.Dataframe(label='All OMR results — editable review table',headers=['Source File','Page','Review Status']+STUDENT_DETAIL_COLUMNS,interactive=True,wrap=True,type='pandas')
        with gr.Accordion('NSAT — EASY TAP-TO-TYPE CORRECTION (recommended for mobile)',open=True):
            gr.Markdown("**Easy method:** enter a serial/file name (or a visible student value) and press Find OMR. Tap the required field directly on the large ORIGINAL OMR image below. The detected field appears under the image. Type into the fixed slots shown under the input and press Apply. Each letter/digit/space uses exactly one slot; a space leaves that slot blank and does not shift later characters. For question grids, tap the question row and type A/B/C/D. Use CLEAR only when you intentionally want to empty that field/question. Original OMR is never overwritten.")
            nsat_manual_load_btn=gr.Button('Load scanned NSAT pages for correction')
            with gr.Row():
                nsat_find_query=gr.Textbox(label='OMR serial / filename / mobile / Exam Code',placeholder='e.g. 0002 or 9876543210')
                nsat_find_btn=gr.Button('Find OMR')
            nsat_manual_page=gr.Dropdown(label='Selected OMR page',choices=[],value=None)
            nsat_quick_target=gr.State({})
            nsat_quick_target_hint=gr.Textbox(label='Tapped field / question',interactive=False,lines=2)
            nsat_quick_value=gr.Textbox(label='Type corrected value (space skips one box)',placeholder='Tap the original OMR image first',lines=1)
            nsat_slot_preview=gr.HTML(value=_slot_preview_html('',{},'nsat'))
            nsat_quick_apply_btn=gr.Button('Apply typed value — replace bubbles automatically',variant='primary')
            nsat_manual_status=gr.Textbox(label='Manual bubble correction status',lines=3)
            nsat_original_view=gr.Image(label='ORIGINAL OMR — tap the field or question you want to correct (unchanged)',type='filepath',height=900,interactive=False)
            nsat_corrected_view=gr.Image(label='CORRECTED OMR COPY — full dark bubbles, original remains unchanged',type='filepath',height=900)
            with gr.Accordion('Advanced slot controls (optional)',open=False):
                nsat_manual_group=gr.Dropdown(label='Field / question grid',choices=[],value=None)
                with gr.Row():
                    nsat_manual_position=gr.Dropdown(label='Slot / question',choices=[],value=None)
                    nsat_manual_bubble=gr.Dropdown(label='Bubble to mark',choices=[],value=None)
                nsat_manual_action=gr.Radio(label='Correction action',choices=['Set selected bubble only','Clear all bubbles in this slot','Toggle selected bubble'],value='Set selected bubble only')
                nsat_apply_bubble_btn=gr.Button('Apply advanced bubble correction',variant='secondary')
        apply_suggestions_btn=gr.Button('Apply OCR proposals to editable fields (review required)')
        suggestion_status=gr.Textbox(label='OCR suggestion status',lines=3)
        apply_suggestions_btn.click(apply_ocr_suggestions_to_review,inputs=[manual_review_table,comparison_table],outputs=[manual_review_table,suggestion_status])
        gr.Markdown("""## Final student-data export
After checking the student details, mark a row VERIFIED to redraw all confirmed student-detail bubbles. If you saved a manual bubble correction through the OMR editor, that page can also be exported before the whole row is VERIFIED; in that case only the explicitly saved manual bubble edits are applied. Original scans are never overwritten. After saving, use the direct Excel, Results ZIP, and corrected-images ZIP download buttons below in this same tab.""")
        manual_export_images_checkbox=gr.Checkbox(label='Create corrected OMR copies for VERIFIED rows OR pages with saved manual bubble edits (source format/DPI)',value=True)
        manual_export_btn=gr.Button('Save Manual Corrections & Build Final Excel/ZIP',variant='primary')
        with gr.Row():
            manual_export_excel_dl=gr.DownloadButton(label='Download corrected Student Excel', value=None, variant='primary', size='lg')
            manual_export_bundle_dl=gr.DownloadButton(label='Download Results + Review ZIP', value=None, variant='secondary', size='lg')
            corrected_images_zip_out=gr.DownloadButton(label='Download ALL corrected NSAT OMR images (ZIP)', value=None, variant='primary', size='lg')
        manual_export_btn.click(export_manual_corrections_v18,inputs=[manual_review_table,run_directory_state,manual_export_images_checkbox],outputs=[student_table,excel_out,bundle_out,corrected_images_zip_out,scan_status,manual_export_excel_dl,manual_export_bundle_dl])
        nsat_manual_load_btn.click(load_nsat_manual_editor,inputs=[run_directory_state],outputs=[nsat_manual_page,nsat_manual_group,nsat_manual_position,nsat_manual_bubble,nsat_original_view,nsat_corrected_view,nsat_manual_status])
        nsat_manual_page.change(render_nsat_manual_page,inputs=[run_directory_state,nsat_manual_page],outputs=[nsat_original_view,nsat_corrected_view,nsat_manual_status])
        nsat_manual_group.change(nsat_manual_group_controls,inputs=[run_directory_state,nsat_manual_group],outputs=[nsat_manual_position,nsat_manual_bubble,nsat_manual_status])
        nsat_apply_bubble_btn.click(apply_nsat_manual_bubble_edit,inputs=[run_directory_state,nsat_manual_page,nsat_manual_group,nsat_manual_position,nsat_manual_bubble,nsat_manual_action,manual_review_table],outputs=[nsat_original_view,nsat_corrected_view,nsat_manual_status,manual_review_table])

        # v14: mobile-first serial search, direct image tap, type value, and automatic slot replacement.
        nsat_find_btn.click(find_nsat_page_by_serial,inputs=[run_directory_state,nsat_find_query,manual_review_table],outputs=[nsat_manual_page,nsat_original_view,nsat_corrected_view,nsat_manual_status])
        nsat_original_view.select(tap_to_pick_nsat_field,inputs=[run_directory_state,nsat_manual_page],outputs=[nsat_quick_target,nsat_quick_target_hint,nsat_quick_value])
        nsat_quick_value.input(render_nsat_slot_preview,inputs=[nsat_quick_value,nsat_quick_target],outputs=[nsat_slot_preview])
        nsat_quick_target.change(render_nsat_slot_preview,inputs=[nsat_quick_value,nsat_quick_target],outputs=[nsat_slot_preview])
        nsat_quick_apply_btn.click(apply_typed_nsat_value,inputs=[run_directory_state,nsat_manual_page,nsat_quick_target,nsat_quick_value,manual_review_table],outputs=[nsat_original_view,nsat_corrected_view,nsat_manual_status,manual_review_table])

    with gr.Tab('4. PW OMR 180Q — Bubble-only'):
        gr.Markdown("""## Alternate Physics Wallah OMR — bubbling-only reader
This tab is for the supplied OMR SHEET with 8-digit Registration No., Phase (1-6), Set (1-3), Class (11th/12th/Dropper), and 180 questions with options 1-4. The blank reference template is built in. **No name, centre, batch, room, date, or handwriting OCR is used in this tab.**

Upload student scans of this exact design. Registration No. is decoded only from its 8-column digit bubble grid; answers are read from Q1-Q180. Multiple marks are preserved (e.g. A|C); unclear readings show `?` and the diagnostics indicate review. Check a small sample against the original before using large batches.""")
        bubble180_files = gr.File(label='Upload student OMR scans (JPG/PNG/TIFF/PDF; multiple files allowed)',file_count='multiple',file_types=['.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp','.pdf'],type='filepath')
        bubble180_sensitivity = gr.Slider(minimum=0.0,maximum=1.0,value=0.75,step=0.05,label='Faint-bubble sensitivity (higher = detects lighter marks; may add false positives)')
        bubble180_scan_btn = gr.Button('Scan 180Q OMR — Bubble Data Only',variant='primary')
        bubble180_status = gr.Textbox(label='Bubble-only processing status',lines=4)
        bubble180_results = gr.Dataframe(label='Bubble-only Results — Registration No., Phase, Set, Class, Q1-Q180',interactive=False,wrap=False)
        bubble180_diagnostics = gr.Dataframe(label='Per-page diagnostics — check NEEDS_REVIEW before official use',interactive=False,wrap=True)
        bubble180_gallery = gr.Gallery(label='Annotated bubble previews (red=filled, amber=uncertain, green=empty)',columns=2,height=420,object_fit='contain')
        gr.Markdown('**Mobile downloads:** Excel, CSV and ZIP appear as file cards after a successful scan/export. Tap the download arrow on the file card; a generated file must have a non-empty size and pass validation before it appears.')
        with gr.Row():
            bubble180_excel = gr.DownloadButton(label='Download Bubble Results Excel', value=None, variant='primary', size='lg')
            bubble180_csv = gr.DownloadButton(label='Download Bubble Results CSV', value=None, variant='secondary', size='lg')
            bubble180_zip = gr.DownloadButton(label='Download Results + Diagnostics ZIP', value=None, variant='primary', size='lg')
            bubble180_images_zip = gr.DownloadButton(label='Download ALL corrected NEET OMR copies (images-only ZIP)', value=None, variant='secondary', size='lg')
        bubble180_run_state = gr.State('')
        bubble180_scan_btn.click(bubble_only_180_batch,inputs=[bubble180_files,bubble180_sensitivity],outputs=[bubble180_status,bubble180_results,bubble180_diagnostics,bubble180_gallery,bubble180_excel,bubble180_csv,bubble180_zip,bubble180_run_state])
        with gr.Accordion('180Q — EASY TAP-TO-TYPE CORRECTION (recommended for mobile)',open=True):
            gr.Markdown("**Easy method:** search by source file/serial or detected Registration No. Tap the original OMR at the Registration No., Phase, Set, Class, or the row of a question. For Registration No. use the 8 light slot boxes below the input; each digit/space occupies one slot. For an MCQ tap the specific question row and type A/B/C/D. The typed choice replaces all competing marks in that slot. Type CLEAR only when intentionally emptying a field/question. No handwriting OCR is used here.")
            bubble180_load_manual_btn=gr.Button('Load scanned 180Q OMR pages for correction')
            with gr.Row():
                bubble180_find_query=gr.Textbox(label='OMR serial / filename / Registration No.',placeholder='e.g. 0002 or 23392227')
                bubble180_find_btn=gr.Button('Find OMR')
            bubble180_manual_page=gr.Dropdown(label='Selected 180Q OMR page',choices=[],value=None)
            bubble180_quick_target=gr.State({})
            bubble180_quick_target_hint=gr.Textbox(label='Tapped field / question',interactive=False,lines=2)
            bubble180_quick_value=gr.Textbox(label='Type corrected value (space skips one box)',placeholder='Tap the original OMR image first',lines=1)
            bubble180_slot_preview=gr.HTML(value=_slot_preview_html('',{},'180'))
            bubble180_quick_apply_btn=gr.Button('Apply typed value — replace bubbles automatically',variant='primary')
            bubble180_manual_status=gr.Textbox(label='Manual correction status',lines=3)
            bubble180_original_view=gr.Image(label='ORIGINAL NEET/180Q OMR — tap a field or question row (unchanged)',type='filepath',height=900,interactive=False)
            bubble180_corrected_view=gr.Image(label='CORRECTED OMR COPY — full dark bubbles, original remains unchanged',type='filepath',height=900)
            with gr.Accordion('Advanced slot controls (optional)',open=False):
                bubble180_manual_field=gr.Dropdown(label='Field to correct',choices=['Answer (Q1-Q180)','Registration No. digit column','Phase','Set','Class'],value='Answer (Q1-Q180)')
                with gr.Row():
                    bubble180_manual_position=gr.Dropdown(label='Question / digit position',choices=[],value=None)
                    bubble180_manual_choice=gr.Dropdown(label='Bubble option to mark',choices=[],value=None)
                bubble180_manual_action=gr.Radio(label='Correction action',choices=['Replace with selected bubble','Clear this slot','Toggle selected bubble'],value='Replace with selected bubble')
                bubble180_apply_manual_btn=gr.Button('Apply advanced correction',variant='secondary')
            bubble180_manual_export_btn=gr.Button('Save manual corrections and rebuild final Excel + ZIP',variant='primary')
        bubble180_load_manual_btn.click(load_180_manual_editor,inputs=[bubble180_run_state],outputs=[bubble180_manual_page,bubble180_original_view,bubble180_corrected_view,bubble180_manual_status])
        bubble180_manual_page.change(show_180_manual_page,inputs=[bubble180_run_state,bubble180_manual_page],outputs=[bubble180_original_view,bubble180_corrected_view,bubble180_manual_status])
        bubble180_manual_field.change(_bubble180_manual_field_controls,inputs=[bubble180_manual_field],outputs=[bubble180_manual_position,bubble180_manual_choice,bubble180_manual_status])
        bubble180_apply_manual_btn.click(apply_180_manual_bubble_edit,inputs=[bubble180_run_state,bubble180_manual_page,bubble180_manual_field,bubble180_manual_position,bubble180_manual_choice,bubble180_manual_action,bubble180_results],outputs=[bubble180_original_view,bubble180_corrected_view,bubble180_manual_status,bubble180_results])
        bubble180_manual_export_btn.click(export_180_manual_results,inputs=[bubble180_results,bubble180_run_state],outputs=[bubble180_excel,bubble180_csv,bubble180_zip,bubble180_images_zip,bubble180_status])

        # v14: easy 180Q search + direct image tap + typed field/answer replacement.
        bubble180_find_btn.click(find_180_page_by_serial,inputs=[bubble180_run_state,bubble180_find_query,bubble180_results],outputs=[bubble180_manual_page,bubble180_original_view,bubble180_corrected_view,bubble180_manual_status])
        bubble180_original_view.select(tap_to_pick_180_field,inputs=[bubble180_run_state,bubble180_manual_page],outputs=[bubble180_quick_target,bubble180_quick_target_hint,bubble180_quick_value])
        bubble180_quick_value.input(render_180_slot_preview,inputs=[bubble180_quick_value,bubble180_quick_target],outputs=[bubble180_slot_preview])
        bubble180_quick_target.change(render_180_slot_preview,inputs=[bubble180_quick_value,bubble180_quick_target],outputs=[bubble180_slot_preview])
        bubble180_quick_apply_btn.click(apply_typed_180_value,inputs=[bubble180_run_state,bubble180_manual_page,bubble180_quick_target,bubble180_quick_value,bubble180_results],outputs=[bubble180_original_view,bubble180_corrected_view,bubble180_manual_status,bubble180_results])


    scan_btn.click(process_batch, inputs=[profile_state,profile_bundle_in,omr_files,fill_sensitivity,run_text_crosscheck,enable_name_ocr,use_ai_name_ocr], outputs=[student_table,review_gallery,excel_out,bundle_out,scan_status,manual_review_table,run_directory_state,diagnostics_table,comparison_table,original_review_gallery])

if os.environ.get('OMR_NO_LAUNCH') == '1':
    print('OMR v18 UI initialized. Server launch skipped because OMR_NO_LAUNCH=1.')
else:
    demo.queue(max_size=50).launch(share=True, debug=False, show_error=True, allowed_paths=[str(ROOT.resolve()), str((ROOT / 'gradio_downloads').resolve())], max_file_size='50mb')
